# MRMS × HydroFabric — Step 1: Read the HydroFabric and inventory everything

**Goal of the whole project**
1. Read the CONUS NextGen HydroFabric and extract every layer/feature.
2. Filter MRMS grid locations to those falling inside HydroFabric catchments.
3. Pick which dates/events to pull MRMS for.
4. Download 2-minute MRMS as CSV, each row tagged with: location, VPU, catchment (divide_id), and the nexus that catchment drains to.
5. (Later) filter locations by radar coverage.
6. (Later) filter VPUs to only those touched by the filtered flood events, then download MRMS for just those.

**This notebook = Step 1 only:** open `conus_nextgen.gpkg`, confirm I can read it,
inventory every layer and field, load the backbone layers, and build the master
catchment table (divide_id → VPU → nexus) that the rest of the project depends on.

## 1.0 Imports

The PROJ env fix below is the same defensive trick from the CUAHSI hydrofabric
notebook — it prevents a coordinate-database clash on JupyterHub. It must run
*before* geopandas/pyproj are imported.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import pyproj

pyproj.network.set_network_enabled(False)

In [ ]:
import os
import sys
from pathlib import Path

# --- PROJ database fix (set BEFORE importing geopandas/pyproj) ---
proj_data = Path(sys.prefix) / "share" / "proj"
if proj_data.exists():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)

import fiona
import pandas as pd
import geopandas as gpd

# pyogrio gives fast, memory-cheap layer info (no need to read the data).
# It ships with modern geopandas; we fall back to fiona if it's missing.
try:
    import pyogrio

    HAVE_PYOGRIO = True
except ImportError:
    HAVE_PYOGRIO = False

import warnings

warnings.filterwarnings("ignore")

print("geopandas", gpd.__version__, "| pyogrio available:", HAVE_PYOGRIO)

In [ ]:
import pyproj

pyproj.network.set_network_enabled(
    False,
)  # use offline transforms; avoids corrupt PROJ grid TIFFs

## 1.1 Locate the GeoPackage and confirm it exists

Set the path once here. The CONUS file is large, so we check it's present and
report its size before touching it.

In [ ]:
# Adjust if your file lives elsewhere. We check a couple of common spots.
CANDIDATES = [
    Path("conus_nextgen.gpkg"),
    Path("storm_usgs_data/conus_nextgen.gpkg"),
]

gpkg_path = next((p for p in CANDIDATES if p.exists()), None)

if gpkg_path is None:
    raise FileNotFoundError(
        "conus_nextgen.gpkg not found. Edit CANDIDATES with the correct path.\n"
        f"Looked in: {[str(p) for p in CANDIDATES]}",
    )

size_gb = gpkg_path.stat().st_size / 1e9
print(f"Found: {gpkg_path.resolve()}")
print(f"Size : {size_gb:.2f} GB")

## 1.2 List every layer, then inventory each one

"Extract every feature" = see every layer, and for each layer its fields,
geometry type, CRS, and row count. We use `pyogrio.read_info`, which reads only
metadata (fast, no data loaded into memory).

In [ ]:
layers = fiona.listlayers(str(gpkg_path))
print(f"{len(layers)} layers found:\n")
for name in layers:
    print("  -", name)

In [ ]:
def layer_info(path, layer):
    """Return (n_features, geometry_type, crs, [field names]) cheaply."""
    if HAVE_PYOGRIO:
        info = pyogrio.read_info(str(path), layer=layer)
        crs = info.get("crs")
        return (
            int(info.get("features", -1)),
            info.get("geometry_type"),
            str(crs) if crs else None,
            list(info.get("fields", [])),
        )
    # Fallback via fiona
    with fiona.open(str(path), layer=layer) as src:
        return (
            len(src),
            src.schema.get("geometry"),
            str(src.crs),
            list(src.schema["properties"].keys()),
        )


rows = []
fields_by_layer = {}
for name in layers:
    n, geom, crs, fields = layer_info(gpkg_path, name)
    fields_by_layer[name] = fields
    rows.append(
        {
            "layer": name,
            "n_features": n,
            "geometry": geom,
            "crs": crs,
            "n_fields": len(fields),
        },
    )

inventory = pd.DataFrame(rows)
inventory

In [ ]:
# The fields (columns) available in each layer — this is the "every feature" map.
for name in layers:
    print(f"\n=== {name} ({len(fields_by_layer[name])} fields) ===")
    print(", ".join(fields_by_layer[name]))

## 1.3 Load the backbone layers

We need:
- `network`  — attribute table linking divide_id ↔ flowpath id ↔ toid ↔ vpuid (no geometry, cheap)
- `divides`  — catchment polygons (this is what MRMS points get joined to)
- `flowpaths`— stream reaches (kept for later / radar-coverage and routing context)
- `nexus`    — junction points where catchments discharge

**Memory note:** `divides`/`flowpaths` carry geometry for all of CONUS, so we read
only the columns we need. Loading takes a bit. Comment out `flowpaths` if RAM is tight.

In [ ]:
# Attribute-only table: fast, full read.
network = gpd.read_file(
    gpkg_path,
    layer="network",
    columns=["id", "toid", "divide_id", "vpuid"],
    read_geometry=False,
)

# Polygons / lines — restrict columns to keep memory down.
divides = gpd.read_file(
    gpkg_path,
    layer="divides",
    columns=["divide_id", "id", "toid", "areasqkm"],
)

flowpaths = gpd.read_file(
    gpkg_path,
    layer="flowpaths",
    columns=["id", "toid"],
)

nexus = gpd.read_file(
    gpkg_path,
    layer="nexus",
    columns=["id", "toid"],
)

print(f"network  : {len(network):,} rows (no geometry)")
print(f"divides  : {len(divides):,} polygons  | CRS {divides.crs}")
print(f"flowpaths: {len(flowpaths):,} lines    | CRS {flowpaths.crs}")
print(f"nexus    : {len(nexus):,} points   | CRS {nexus.crs}")

In [ ]:
# Quick peek so we can see real values/ID formats (cat-..., nex-..., wb-...).
display(network.head(3))
display(divides.head(3))
display(nexus.head(3))

## 1.4 Build the master catchment table (divide_id → VPU → nexus)

This is the key product of Step 1. Each catchment polygon gets:
- `divide_id`  — the catchment number
- `vpuid`      — which VPU it belongs to (drives event-based VPU filtering later)
- `nexus_id`   — the nexus it drains into (a catchment's `toid` is its `nex-...`)
- `areasqkm`, `geometry`

Later steps spatially join MRMS grid points **into** this table, so every MRMS
location inherits its catchment, VPU, and nexus in one shot. A `radar_covered`
flag will be added here in a later step (placeholder noted, not built yet).

In [ ]:
# network can have several rows per divide_id; keep one vpuid per divide.
net_lookup = network.dropna(subset=["divide_id"]).drop_duplicates(subset=["divide_id"])[
    ["divide_id", "vpuid"]
]

catchments_master = divides.rename(
    columns={"id": "flowpath_id", "toid": "nexus_id"},
).merge(net_lookup, on="divide_id", how="left")

# Sanity checks
n = len(catchments_master)
print(f"catchments_master: {n:,} catchments")
print(f"  with a VPU     : {catchments_master['vpuid'].notna().sum():,}")
print(
    f"  with a nexus   : {catchments_master['nexus_id'].astype(str).str.startswith('nex-').sum():,}",
)
print(f"  unique VPUs    : {catchments_master['vpuid'].nunique()}")

catchments_master.head()

catchments_master["nexus_type"] = (
    catchments_master["nexus_id"].astype(str).str.extract(r"^([a-z]+)-")[0]
)

catchments_master["is_terminal"] = catchments_master["nexus_type"].isin(
    ["tnx", "cnx", "inx"],
)

## 1.5 What VPUs exist (sets up event-based filtering later)

Later you'll intersect your flood events with these VPUs and keep only the ones
that contain events. For now, just confirm what's available and how big each is.

In [ ]:
vpu_summary = (
    catchments_master.groupby("vpuid")
    .agg(n_catchments=("divide_id", "size"), area_km2=("areasqkm", "sum"))
    .sort_index()
)
print(f"{len(vpu_summary)} VPUs")
vpu_summary

## Step 1 — done ✅

I can read `conus_nextgen.gpkg`, I've inventoried every layer and field, and I
built `catchments_master` (divide_id → VPU → nexus + geometry), which is the
table everything downstream attaches to.

**Coming next**
- **Step 2:** build the MRMS 1 km grid points and spatially join them into
  `catchments_master`, so each MRMS cell gets a catchment / VPU / nexus.
- **Step 3:** load flood events, find which VPUs they fall in, filter to those VPUs.
- **Step 4:** download 2-min MRMS for the selected dates → CSV
  (location, VPU, divide_id, nexus_id), with a radar-coverage flag.

<!-- ## 2.1 Build the crosswalk, one VPU at a time (cached)

Each VPU's result is written to its own parquet so the loop is resumable — if it's
interrupted, re-running skips VPUs already done. The combined CONUS crosswalk is
written at the end. Delete `mrms_crosswalk_cache/` to force a full rebuild. -->

## Step 1b — Optional terminal outlet nexus

This step adds `terminal_nexus_id`, which is the final basin outlet reached by walking `toid` downstream until water exits the hydrofabric domain at `wb-0`.

The immediate `nexus_id` is the main column needed for Steps 2–4. The terminal outlet is optional and mainly useful for later routing analysis.

A few catchments may sit in residual network cycles. These are flagged with `terminal_is_clean = False`.

In [ ]:
nexus_next = dict(
    zip(
        nexus["id"].astype(str),
        nexus["toid"].astype(str),
    ),
)

net_wb = network[network["id"].astype(str).str.startswith("wb-")]

wb_next = dict(
    zip(
        net_wb["id"].astype(str),
        net_wb["toid"].astype(str),
    ),
)

TERMINAL_WB = "wb-0"


def _down_nexus(n):
    wb = nexus_next.get(n)

    if wb is None or wb == TERMINAL_WB:
        return None

    return wb_next.get(wb)


terminal_cache = {}


def terminal_of(start):
    path = []
    cur = start
    seen = set()

    while cur is not None and cur not in terminal_cache:
        if cur in seen:
            # cycle guard
            for p in path:
                terminal_cache[p] = cur
            return cur

        seen.add(cur)
        path.append(cur)

        nxt = _down_nexus(cur)

        if nxt is None:
            terminal_cache[cur] = cur
            break

        cur = nxt

    result = terminal_cache.get(cur, cur)

    for p in path:
        terminal_cache[p] = result

    return result


for n in set(catchments_master["nexus_id"].astype(str)):
    terminal_of(n)


catchments_master["terminal_nexus_id"] = (
    catchments_master["nexus_id"].astype(str).map(terminal_cache)
)

catchments_master["terminal_is_clean"] = ~(
    catchments_master["terminal_nexus_id"].astype(str).str.startswith("nex-")
)

print(
    "terminal outlets:",
    catchments_master["terminal_nexus_id"].nunique(),
    "| cycle-flagged:",
    (~catchments_master["terminal_is_clean"]).sum(),
)

# Step 2 — Build the MRMS → HydroFabric crosswalk

For each VPU, this step generates MRMS grid-cell center points inside that VPU bounding box, reprojects the points to EPSG:5070, and spatially joins them to the hydrofabric catchments.

Each MRMS grid cell is assigned to one catchment using point-in-polygon.

The result is cached by VPU, so the process is resumable.

Important assumptions:

- MRMS grid resolution is 0.01 degrees.
- MRMS grid size is 3500 × 7000.
- MRMS longitude edges run from -130 to -60.
- MRMS latitude edges run from 20 to 55.
- Cell centers are offset by half a cell.
- `cell_id = row * 7000 + col`.

In [ ]:
import shutil
import time

# ------------------------------------------------------------
# MRMS CONUS grid definition
# ------------------------------------------------------------

MRMS_LON_MIN_EDGE = -130.0
MRMS_LAT_MAX_EDGE = 55.0

MRMS_RES = 0.01
MRMS_NLON = 7000
MRMS_NLAT = 3500

HALF = MRMS_RES / 2.0


def mrms_centers_for_bbox(west, east, south, north, pad=1):
    """
    Return MRMS cell centers covering a bounding box.

    Output:
    row, col, lon, lat
    """
    i0 = int(np.floor((west - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) - pad
    i1 = int(np.ceil((east - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) + pad

    j0 = int(np.floor((MRMS_LAT_MAX_EDGE - HALF - north) / MRMS_RES)) - pad
    j1 = int(np.ceil((MRMS_LAT_MAX_EDGE - HALF - south) / MRMS_RES)) + pad

    i0 = max(0, i0)
    i1 = min(MRMS_NLON - 1, i1)

    j0 = max(0, j0)
    j1 = min(MRMS_NLAT - 1, j1)

    if i1 < i0 or j1 < j0:
        return None

    cols = np.arange(i0, i1 + 1)
    rows = np.arange(j0, j1 + 1)

    lon = MRMS_LON_MIN_EDGE + HALF + cols * MRMS_RES
    lat = MRMS_LAT_MAX_EDGE - HALF - rows * MRMS_RES

    LON, LAT = np.meshgrid(lon, lat)
    COL, ROW = np.meshgrid(cols, rows)

    return ROW.ravel(), COL.ravel(), LON.ravel(), LAT.ravel()


# ------------------------------------------------------------
# Cache setup
# ------------------------------------------------------------

CACHE = Path("mrms_crosswalk_cache")
CACHE.mkdir(exist_ok=True)

COMBINED = CACHE / "mrms_hf_crosswalk_conus.parquet"

CARRY = [
    "divide_id",
    "vpuid",
    "nexus_id",
    "nexus_type",
    "is_terminal",
    "geometry",
]

# To force a full rebuild, uncomment these two lines:
# shutil.rmtree(CACHE, ignore_errors=True)
# CACHE.mkdir(exist_ok=True)


# ------------------------------------------------------------
# Build VPU-batched crosswalk
# ------------------------------------------------------------

vpus = sorted(catchments_master["vpuid"].dropna().unique())

parts = []
t_all = time.time()

for vpu in vpus:
    out = CACHE / f"crosswalk_vpu_{vpu}.parquet"

    if out.exists():
        df_cached = pd.read_parquet(out)
        parts.append(df_cached)
        print(f"{vpu}: cached ({len(df_cached):,})")
        continue

    sub = catchments_master[catchments_master["vpuid"] == vpu]

    west, south, east, north = sub.to_crs(4326).total_bounds

    res = mrms_centers_for_bbox(west, east, south, north)

    if res is None:
        print(f"{vpu}: no grid cells in bbox")
        continue

    row, col, lon, lat = res

    pts = gpd.GeoDataFrame(
        {
            "cell_id": row.astype(np.int64) * MRMS_NLON + col.astype(np.int64),
            "row": row.astype(np.int32),
            "col": col.astype(np.int32),
            "lon": lon,
            "lat": lat,
        },
        geometry=gpd.points_from_xy(lon, lat),
        crs=4326,
    ).to_crs(5070)

    joined = gpd.sjoin(
        pts,
        sub[CARRY],
        how="inner",
        predicate="within",
    )

    df = pd.DataFrame(
        joined.drop(columns=["geometry", "index_right"]),
    ).drop_duplicates("cell_id")

    df.to_parquet(out, index=False)
    parts.append(df)

    print(
        f"{vpu}: {len(pts):,} grid pts -> {len(df):,} in catchments "
        f"({100 * len(df) / len(pts):.0f}% land)",
    )


crosswalk = pd.concat(parts, ignore_index=True).drop_duplicates("cell_id")

crosswalk.to_parquet(COMBINED, index=False)

print(
    f"\nDONE in {time.time() - t_all:.0f}s | "
    f"CONUS crosswalk: {len(crosswalk):,} MRMS cells -> {COMBINED}",
)

## 2.1 Sanity checks

These checks confirm that the crosswalk is usable.

Expected:

- `unique cell_id` should equal total rows.
- There should be no nulls in `divide_id`, `vpuid`, or `nexus_id`.
- Some very small catchments may receive zero MRMS cells. That is expected because the MRMS grid is about 1 km.

In [ ]:
cw = pd.read_parquet(COMBINED)

print(f"total MRMS cells in crosswalk  : {len(cw):,}")
print(f"unique cell_id should = total  : {cw['cell_id'].nunique():,}")

print(
    f"unique catchments hit          : "
    f"{cw['divide_id'].nunique():,} of "
    f"{catchments_master['divide_id'].nunique():,}",
)

print(
    f"nulls -> divide_id {cw['divide_id'].isna().sum()}, "
    f"vpuid {cw['vpuid'].isna().sum()}, "
    f"nexus_id {cw['nexus_id'].isna().sum()}",
)

print(
    f"\nmean MRMS cells per catchment  : {len(cw) / cw['divide_id'].nunique():.1f}",
)

print(
    f"catchments with 0 cells        : "
    f"{catchments_master['divide_id'].nunique() - cw['divide_id'].nunique():,}",
)

print("\nMRMS cells per VPU:")
print(
    cw.groupby("vpuid").size().sort_values(ascending=False).to_string(),
)

## 2.2 Correctness visual — cells colored by assigned catchment

This visual checks whether MRMS grid cells are being assigned to the correct catchments.

Correct behavior:

- The target catchment boundary should contain cells assigned to that catchment.
- Cell colors should switch at catchment borders.
- The red boundary should line up with the assigned MRMS cells.

In [ ]:
import matplotlib.pyplot as plt

VPU_DEMO = "03W"

cwv = pd.read_parquet(CACHE / f"crosswalk_vpu_{VPU_DEMO}.parquet")

cwg = gpd.GeoDataFrame(
    cwv,
    geometry=gpd.points_from_xy(cwv.lon, cwv.lat),
    crs=4326,
).to_crs(5070)

cats = catchments_master[catchments_master["vpuid"] == VPU_DEMO]

counts = cwv["divide_id"].value_counts()

target_id = counts[(counts > 8) & (counts < 40)].index[0]

target = cats[cats["divide_id"] == target_id]

minx, miny, maxx, maxy = target.total_bounds

px = (maxx - minx) * 3 + 800
py = (maxy - miny) * 3 + 800

xlim = (minx - px, maxx + px)
ylim = (miny - py, maxy + py)

local_cats = cats.cx[xlim[0] : xlim[1], ylim[0] : ylim[1]]
local_pts = cwg.cx[xlim[0] : xlim[1], ylim[0] : ylim[1]]

fig, ax = plt.subplots(figsize=(9, 9))

local_cats.boundary.plot(
    ax=ax,
    color="0.75",
    linewidth=0.6,
)

local_pts.plot(
    ax=ax,
    column="divide_id",
    cmap="tab20",
    markersize=10,
    legend=False,
)

target.boundary.plot(
    ax=ax,
    color="red",
    linewidth=2.2,
)

ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_axis_off()

ax.set_title(
    f"MRMS cells assigned to catchments — VPU {VPU_DEMO}\n"
    f"red = {target_id} ({counts[target_id]} cells)",
)

plt.tight_layout()
plt.show()

## 2.3 Correctness visual — catchment, cells, and draining nexus

This visual checks the catchment-to-nexus relationship.

The magenta star is the target catchment's draining nexus. It should sit near the outlet of the red target catchment.

Black dots are neighboring nexus points.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
import pyproj

pyproj.network.set_network_enabled(False)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

nexus_geom = gpd.read_file(
    gpkg_path,
    layer="nexus",
    columns=["id", "geometry"],
).to_crs(5070)

local_cats = local_cats.merge(
    catchments_master[["divide_id", "nexus_id"]],
    on="divide_id",
    how="left",
)

local_nexus = nexus_geom[
    nexus_geom["id"].astype(str).isin(local_cats["nexus_id"].astype(str))
]

target_nex_id = catchments_master.loc[
    catchments_master["divide_id"] == target_id,
    "nexus_id",
].iloc[0]

target_nex = nexus_geom[nexus_geom["id"].astype(str) == str(target_nex_id)]

fig, ax = plt.subplots(figsize=(11, 11))
fig.patch.set_facecolor("white")

local_cats.boundary.plot(
    ax=ax,
    color="0.78",
    linewidth=0.7,
    zorder=1,
)

local_pts.plot(
    ax=ax,
    column="divide_id",
    cmap="tab20",
    markersize=14,
    alpha=0.75,
    legend=False,
    zorder=2,
)

target.boundary.plot(
    ax=ax,
    color="#d62728",
    linewidth=2.6,
    zorder=5,
)

local_nexus.plot(
    ax=ax,
    color="black",
    markersize=34,
    edgecolor="white",
    linewidth=0.6,
    zorder=6,
)

target_nex.plot(
    ax=ax,
    color="magenta",
    marker="*",
    markersize=620,
    edgecolor="black",
    linewidth=1.2,
    zorder=7,
)

ax.set_aspect("equal")
ax.set_axis_off()

ax.set_title(
    f"MRMS grid cells → catchments → draining nexus · VPU {VPU_DEMO}",
    fontsize=15,
    fontweight="bold",
    pad=14,
)

ax.text(
    0.5,
    1.005,
    f"Catchment {target_id} drains into nexus {target_nex_id}",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=11,
    color="0.35",
)

ax.legend(
    handles=[
        Patch(
            facecolor="0.6",
            edgecolor="0.4",
            alpha=0.7,
            label="MRMS cells colored by catchment",
        ),
        Line2D(
            [0],
            [0],
            color="0.78",
            lw=1.2,
            label="Catchment boundaries",
        ),
        Line2D(
            [0],
            [0],
            color="#d62728",
            lw=2.6,
            label=f"Target catchment ({target_id})",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="black",
            markeredgecolor="white",
            markersize=9,
            label="Neighboring nexus points",
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="none",
            markerfacecolor="magenta",
            markeredgecolor="black",
            markersize=20,
            label="Draining nexus of target catchment",
        ),
    ],
    loc="lower right",
    frameon=True,
    framealpha=0.95,
    fontsize=10,
    borderpad=0.9,
    edgecolor="0.8",
)

plt.tight_layout()
plt.show()

In [ ]:
import textwrap
import matplotlib.pyplot as plt
from shapely.geometry import box
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ============================================================
# Four-panel visual check:
# 1. CONUS context
# 2. Selected VPU
# 3. Local zoom around selected catchment
# 4. Selected catchment with attributes
# ============================================================

# -----------------------------
# Choose VPU and target catchment
# -----------------------------
VPU_DEMO = "03W"

cwv = pd.read_parquet(CACHE / f"crosswalk_vpu_{VPU_DEMO}.parquet")

cwg = gpd.GeoDataFrame(
    cwv,
    geometry=gpd.points_from_xy(cwv.lon, cwv.lat),
    crs=4326,
).to_crs(5070)

cats = catchments_master[catchments_master["vpuid"] == VPU_DEMO]

# Pick a mid-sized catchment based on number of MRMS cells
counts = cwv["divide_id"].value_counts()

candidate = counts[(counts > 8) & (counts < 40)]

if len(candidate) > 0:
    target_id = candidate.index[0]
else:
    target_id = counts.index[len(counts) // 2]

target = cats[cats["divide_id"] == target_id]

# -----------------------------
# Local zoom extent around target catchment
# -----------------------------
minx, miny, maxx, maxy = target.total_bounds

px = (maxx - minx) * 3 + 800
py = (maxy - miny) * 3 + 800

xlim_local = (minx - px, maxx + px)
ylim_local = (miny - py, maxy + py)

local_cats = cats.cx[xlim_local[0] : xlim_local[1], ylim_local[0] : ylim_local[1]]
local_pts = cwg.cx[xlim_local[0] : xlim_local[1], ylim_local[0] : ylim_local[1]]

# -----------------------------
# Build fast VPU bounding boxes for CONUS context
# -----------------------------
vpu_box_records = []

for vpu, grp in catchments_master.groupby("vpuid"):
    bxmin, bymin, bxmax, bymax = grp.total_bounds
    vpu_box_records.append(
        {
            "vpuid": vpu,
            "geometry": box(bxmin, bymin, bxmax, bymax),
        },
    )

vpu_boxes = gpd.GeoDataFrame(vpu_box_records, crs=catchments_master.crs)

vpu_demo_box = vpu_boxes[vpu_boxes["vpuid"] == VPU_DEMO]

# -----------------------------
# Prepare target attributes text
# -----------------------------
target_row = target.drop(columns="geometry").iloc[0].to_dict()
target_row["mrms_cell_count"] = int(counts[target_id])

attr_lines = []

for key, value in target_row.items():
    if pd.isna(value):
        value = "NA"
    attr_lines.append(f"{key}: {value}")

attr_text = "\n".join(textwrap.wrap("\n".join(attr_lines), width=55))

# ============================================================
# Plot
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.patch.set_facecolor("white")

ax1, ax2, ax3, ax4 = axes.ravel()

# ------------------------------------------------------------
# Panel 1: CONUS context
# ------------------------------------------------------------
vpu_boxes.boundary.plot(
    ax=ax1,
    color="0.75",
    linewidth=0.8,
)

vpu_demo_box.boundary.plot(
    ax=ax1,
    color="blue",
    linewidth=2.2,
)

target.centroid.plot(
    ax=ax1,
    color="red",
    markersize=35,
    zorder=5,
)

ax1.set_title(
    f"1. CONUS context\nVPU {VPU_DEMO} and selected catchment location",
    fontweight="bold",
)
ax1.set_axis_off()
ax1.set_aspect("equal")

# ------------------------------------------------------------
# Panel 2: selected VPU only
# ------------------------------------------------------------
cats.boundary.plot(
    ax=ax2,
    color="0.80",
    linewidth=0.25,
)

target.plot(
    ax=ax2,
    facecolor="none",
    edgecolor="red",
    linewidth=2.5,
)

ax2.set_title(
    f"2. Selected VPU only\nVPU {VPU_DEMO}, target catchment {target_id}",
    fontweight="bold",
)
ax2.set_axis_off()
ax2.set_aspect("equal")

# ------------------------------------------------------------
# Panel 3: local zoom around target catchment
# ------------------------------------------------------------
local_cats.boundary.plot(
    ax=ax3,
    color="0.75",
    linewidth=0.6,
)

local_pts.plot(
    ax=ax3,
    column="divide_id",
    cmap="tab20",
    markersize=10,
    legend=False,
)

target.boundary.plot(
    ax=ax3,
    color="red",
    linewidth=2.5,
)

ax3.set_xlim(xlim_local)
ax3.set_ylim(ylim_local)

ax3.set_title(
    "3. Local zoom\nMRMS cells colored by assigned catchment",
    fontweight="bold",
)
ax3.set_axis_off()
ax3.set_aspect("equal")

# ------------------------------------------------------------
# Panel 4: selected catchment with attributes
# ------------------------------------------------------------
target.plot(
    ax=ax4,
    facecolor="lightgray",
    edgecolor="red",
    linewidth=2.5,
)

ax4.set_title(
    f"4. Target catchment attributes\n{target_id}",
    fontweight="bold",
)
ax4.set_axis_off()
ax4.set_aspect("equal")

ax4.text(
    0.02,
    0.98,
    attr_text,
    transform=ax4.transAxes,
    ha="left",
    va="top",
    fontsize=8,
    family="monospace",
    bbox={
        "facecolor": "white",
        "edgecolor": "0.5",
        "alpha": 0.90,
        "boxstyle": "round,pad=0.5",
    },
)

# -----------------------------
# Overall title
# -----------------------------
fig.suptitle(
    f"MRMS → HydroFabric visual check | VPU {VPU_DEMO} | Catchment {target_id}",
    fontsize=16,
    fontweight="bold",
    y=0.98,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
import textwrap
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import box

# ============================================================
# Four-panel visual check:
# 1. CONUS with VPU bounding boxes and target catchment location
# 2. Selected VPU with target catchment highlighted
# 3. Local zoom around target catchment with MRMS cells
# 4. Target catchment only, MRMS points inside it, and attributes
# ============================================================

# -----------------------------
# Choose VPU and target catchment
# -----------------------------
VPU_DEMO = "03W"

cwv = pd.read_parquet(CACHE / f"crosswalk_vpu_{VPU_DEMO}.parquet")

cwg = gpd.GeoDataFrame(
    cwv,
    geometry=gpd.points_from_xy(cwv.lon, cwv.lat),
    crs=4326,
).to_crs(5070)

cats = catchments_master[catchments_master["vpuid"] == VPU_DEMO].copy()

# Pick a mid-sized catchment based on number of MRMS cells
counts = cwv["divide_id"].value_counts()
candidate = counts[(counts > 8) & (counts < 40)]

if len(candidate) > 0:
    target_id = candidate.index[0]
else:
    target_id = counts.index[len(counts) // 2]

target = cats[cats["divide_id"] == target_id].copy()

# MRMS points assigned to the target catchment
target_pts = cwg[cwg["divide_id"] == target_id].copy()

# -----------------------------
# Fast VPU bounding boxes for CONUS context
# This is faster than dissolving true VPU polygons.
# These are rectangular envelopes, so they can overlap.
# -----------------------------
vpu_box_cache = CACHE / "vpu_bounding_boxes.parquet"

if vpu_box_cache.exists():
    vpu_boxes = gpd.read_parquet(vpu_box_cache)
else:
    vpu_box_records = []

    for vpu, grp in catchments_master.groupby("vpuid"):
        bxmin, bymin, bxmax, bymax = grp.total_bounds
        vpu_box_records.append(
            {
                "vpuid": vpu,
                "geometry": box(bxmin, bymin, bxmax, bymax),
            },
        )

    vpu_boxes = gpd.GeoDataFrame(
        vpu_box_records,
        geometry="geometry",
        crs=catchments_master.crs,
    )

    vpu_boxes.to_parquet(vpu_box_cache, index=False)

vpu_demo_box = vpu_boxes[vpu_boxes["vpuid"] == VPU_DEMO]

# -----------------------------
# Local zoom extent around target catchment
# -----------------------------
minx, miny, maxx, maxy = target.total_bounds

px = (maxx - minx) * 3 + 800
py = (maxy - miny) * 3 + 800

xlim_local = (minx - px, maxx + px)
ylim_local = (miny - py, maxy + py)

local_cats = cats.cx[xlim_local[0] : xlim_local[1], ylim_local[0] : ylim_local[1]]
local_pts = cwg.cx[xlim_local[0] : xlim_local[1], ylim_local[0] : ylim_local[1]]

# -----------------------------
# Target-only zoom extent
# -----------------------------
txmin, tymin, txmax, tymax = target.total_bounds

target_pad_x = (txmax - txmin) * 0.35 + 300
target_pad_y = (tymax - tymin) * 0.35 + 300

xlim_target = (txmin - target_pad_x, txmax + target_pad_x)
ylim_target = (tymin - target_pad_y, tymax + target_pad_y)

# -----------------------------
# Prepare hydrofabric attributes text
# -----------------------------
target_attrs = target.drop(columns="geometry").iloc[0].to_dict()
target_attrs["mrms_cell_count"] = int(len(target_pts))

attr_lines = []

for key, value in target_attrs.items():
    if pd.isna(value):
        value = "NA"
    attr_lines.append(f"{key}: {value}")

attr_text = "\n".join(attr_lines)

# ============================================================
# Plot
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.patch.set_facecolor("white")

ax1, ax2, ax3, ax4 = axes.ravel()

# ------------------------------------------------------------
# Panel 1: CONUS context with VPU bounding boxes
# ------------------------------------------------------------
vpu_boxes.boundary.plot(
    ax=ax1,
    color="0.82",
    linewidth=0.7,
)

vpu_demo_box.boundary.plot(
    ax=ax1,
    color="blue",
    linewidth=2.2,
)

target.centroid.plot(
    ax=ax1,
    color="red",
    markersize=35,
    zorder=5,
)

ax1.set_title(
    f"1. CONUS context\nVPU bounding boxes, selected VPU {VPU_DEMO}, target catchment",
    fontweight="bold",
)
ax1.set_axis_off()
ax1.set_aspect("equal")

# ------------------------------------------------------------
# Panel 2: selected VPU only
# ------------------------------------------------------------
cats.boundary.plot(
    ax=ax2,
    color="0.82",
    linewidth=0.25,
)

target.plot(
    ax=ax2,
    facecolor="none",
    edgecolor="red",
    linewidth=2.5,
)

ax2.set_title(
    f"2. Selected VPU only\nVPU {VPU_DEMO}, target catchment {target_id}",
    fontweight="bold",
)
ax2.set_axis_off()
ax2.set_aspect("equal")

# ------------------------------------------------------------
# Panel 3: local zoom around target catchment
# ------------------------------------------------------------
local_cats.boundary.plot(
    ax=ax3,
    color="0.75",
    linewidth=0.6,
)

local_pts.plot(
    ax=ax3,
    column="divide_id",
    cmap="tab20",
    markersize=10,
    legend=False,
)

target.boundary.plot(
    ax=ax3,
    color="red",
    linewidth=2.5,
)

ax3.set_xlim(xlim_local)
ax3.set_ylim(ylim_local)

ax3.set_title(
    "3. Local zoom\nMRMS points colored by assigned catchment",
    fontweight="bold",
)
ax3.set_axis_off()
ax3.set_aspect("equal")

# ------------------------------------------------------------
# Panel 4: target catchment only with MRMS points and attributes
# ------------------------------------------------------------
target.plot(
    ax=ax4,
    facecolor="0.90",
    edgecolor="red",
    linewidth=2.5,
    zorder=1,
)

target_pts.plot(
    ax=ax4,
    color="black",
    markersize=28,
    alpha=0.85,
    zorder=2,
)

# Optional: label MRMS row/col for each point if not too many
if len(target_pts) <= 60:
    for _, r in target_pts.iterrows():
        ax4.text(
            r.geometry.x,
            r.geometry.y,
            f"{int(r['row'])},{int(r['col'])}",
            fontsize=6,
            ha="center",
            va="center",
            color="blue",
            zorder=3,
        )

ax4.set_xlim(xlim_target)
ax4.set_ylim(ylim_target)

ax4.set_title(
    "4. Target catchment only\nMRMS points + hydrofabric attributes",
    fontweight="bold",
)
ax4.set_axis_off()
ax4.set_aspect("equal")

# Attribute box
ax4.text(
    1.02,
    1.00,
    attr_text,
    transform=ax4.transAxes,
    ha="left",
    va="top",
    fontsize=8,
    family="monospace",
    bbox={
        "facecolor": "white",
        "edgecolor": "0.5",
        "alpha": 0.95,
        "boxstyle": "round,pad=0.5",
    },
)

# -----------------------------
# Figure legend
# -----------------------------
fig.legend(
    handles=[
        Line2D(
            [0],
            [0],
            color="0.82",
            lw=1.0,
            label="VPU bounding boxes / catchment boundaries",
        ),
        Line2D([0], [0], color="blue", lw=2.2, label=f"Selected VPU box {VPU_DEMO}"),
        Line2D([0], [0], color="red", lw=2.5, label=f"Target catchment {target_id}"),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="black",
            markersize=8,
            label="MRMS points inside target catchment",
        ),
    ],
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=10,
)

fig.suptitle(
    f"MRMS → HydroFabric visual check | VPU {VPU_DEMO} | Catchment {target_id}",
    fontsize=16,
    fontweight="bold",
    y=0.98,
)

plt.tight_layout(rect=[0, 0.04, 1, 0.95])
plt.show()

In [ ]:
# import textwrap
# import matplotlib.pyplot as plt
# from matplotlib.patches import Patch
# from matplotlib.lines import Line2D

# # ============================================================
# # Four-panel visual check:
# # 1. CONUS with real VPU polygons and target catchment location
# # 2. Selected VPU with target catchment highlighted
# # 3. Local zoom around target catchment with MRMS cells
# # 4. Target catchment only, MRMS points inside it, and attributes
# # ============================================================

# # -----------------------------
# # Choose VPU and target catchment
# # -----------------------------
# VPU_DEMO = "03W"

# cwv = pd.read_parquet(CACHE / f"crosswalk_vpu_{VPU_DEMO}.parquet")

# cwg = gpd.GeoDataFrame(
#     cwv,
#     geometry=gpd.points_from_xy(cwv.lon, cwv.lat),
#     crs=4326
# ).to_crs(5070)

# cats = catchments_master[catchments_master["vpuid"] == VPU_DEMO].copy()

# # Pick a mid-sized catchment based on number of MRMS cells
# counts = cwv["divide_id"].value_counts()
# candidate = counts[(counts > 8) & (counts < 40)]

# if len(candidate) > 0:
#     target_id = candidate.index[0]
# else:
#     target_id = counts.index[len(counts) // 2]

# target = cats[cats["divide_id"] == target_id].copy()

# # MRMS points assigned to the target catchment
# target_pts = cwg[cwg["divide_id"] == target_id].copy()

# # -----------------------------
# # Real VPU polygons for CONUS context
# # Not bounding boxes, so they should not artificially overlap
# # -----------------------------
# vpu_cache = CACHE / "vpu_dissolved_polygons.parquet"

# if vpu_cache.exists():
#     vpu_polys = gpd.read_parquet(vpu_cache)
# else:
#     vpu_polys = (
#         catchments_master[["vpuid", "geometry"]]
#         .dissolve(by="vpuid")
#         .reset_index()
#     )
#     vpu_polys.to_parquet(vpu_cache, index=False)

# vpu_demo_poly = vpu_polys[vpu_polys["vpuid"] == VPU_DEMO]

# # -----------------------------
# # Local zoom extent around target catchment
# # -----------------------------
# minx, miny, maxx, maxy = target.total_bounds

# px = (maxx - minx) * 3 + 800
# py = (maxy - miny) * 3 + 800

# xlim_local = (minx - px, maxx + px)
# ylim_local = (miny - py, maxy + py)

# local_cats = cats.cx[xlim_local[0]:xlim_local[1], ylim_local[0]:ylim_local[1]]
# local_pts = cwg.cx[xlim_local[0]:xlim_local[1], ylim_local[0]:ylim_local[1]]

# # -----------------------------
# # Target-only zoom extent
# # -----------------------------
# txmin, tymin, txmax, tymax = target.total_bounds

# target_pad_x = (txmax - txmin) * 0.35 + 300
# target_pad_y = (tymax - tymin) * 0.35 + 300

# xlim_target = (txmin - target_pad_x, txmax + target_pad_x)
# ylim_target = (tymin - target_pad_y, tymax + target_pad_y)

# # -----------------------------
# # Prepare hydrofabric attributes text
# # -----------------------------
# target_attrs = target.drop(columns="geometry").iloc[0].to_dict()
# target_attrs["mrms_cell_count"] = int(len(target_pts))

# attr_lines = []
# for key, value in target_attrs.items():
#     if pd.isna(value):
#         value = "NA"
#     attr_lines.append(f"{key}: {value}")

# attr_text = "\n".join(attr_lines)

# # If the text is too long, wrap it
# attr_text_wrapped = "\n".join(
#     textwrap.wrap(attr_text, width=65, replace_whitespace=False)
# )

# # ============================================================
# # Plot
# # ============================================================
# fig, axes = plt.subplots(2, 2, figsize=(20, 16))
# fig.patch.set_facecolor("white")

# ax1, ax2, ax3, ax4 = axes.ravel()

# # ------------------------------------------------------------
# # Panel 1: CONUS context with real VPU polygons
# # ------------------------------------------------------------
# vpu_polys.boundary.plot(
#     ax=ax1,
#     color="0.82",
#     linewidth=0.5
# )

# vpu_demo_poly.plot(
#     ax=ax1,
#     facecolor="none",
#     edgecolor="blue",
#     linewidth=2.2
# )

# target.centroid.plot(
#     ax=ax1,
#     color="red",
#     markersize=35,
#     zorder=5
# )

# ax1.set_title(
#     f"1. CONUS context\nreal VPU polygons, selected VPU {VPU_DEMO}, target catchment",
#     fontweight="bold"
# )
# ax1.set_axis_off()
# ax1.set_aspect("equal")

# # ------------------------------------------------------------
# # Panel 2: selected VPU only
# # ------------------------------------------------------------
# cats.boundary.plot(
#     ax=ax2,
#     color="0.82",
#     linewidth=0.25
# )

# target.plot(
#     ax=ax2,
#     facecolor="none",
#     edgecolor="red",
#     linewidth=2.5
# )

# ax2.set_title(
#     f"2. Selected VPU only\nVPU {VPU_DEMO}, target catchment {target_id}",
#     fontweight="bold"
# )
# ax2.set_axis_off()
# ax2.set_aspect("equal")

# # ------------------------------------------------------------
# # Panel 3: local zoom around target catchment
# # ------------------------------------------------------------
# local_cats.boundary.plot(
#     ax=ax3,
#     color="0.75",
#     linewidth=0.6
# )

# local_pts.plot(
#     ax=ax3,
#     column="divide_id",
#     cmap="tab20",
#     markersize=10,
#     legend=False
# )

# target.boundary.plot(
#     ax=ax3,
#     color="red",
#     linewidth=2.5
# )

# ax3.set_xlim(xlim_local)
# ax3.set_ylim(ylim_local)

# ax3.set_title(
#     "3. Local zoom\nMRMS cells colored by assigned catchment",
#     fontweight="bold"
# )
# ax3.set_axis_off()
# ax3.set_aspect("equal")

# # ------------------------------------------------------------
# # Panel 4: target catchment only with MRMS points and attributes
# # ------------------------------------------------------------
# target.plot(
#     ax=ax4,
#     facecolor="0.90",
#     edgecolor="red",
#     linewidth=2.5,
#     zorder=1
# )

# target_pts.plot(
#     ax=ax4,
#     color="black",
#     markersize=28,
#     alpha=0.85,
#     zorder=2
# )

# # Optional: label MRMS row/col for each point if not too many
# if len(target_pts) <= 60:
#     for _, r in target_pts.iterrows():
#         ax4.text(
#             r.geometry.x,
#             r.geometry.y,
#             f"{int(r['row'])},{int(r['col'])}",
#             fontsize=6,
#             ha="center",
#             va="center",
#             color="blue",
#             zorder=3
#         )

# ax4.set_xlim(xlim_target)
# ax4.set_ylim(ylim_target)

# ax4.set_title(
#     f"4. Target catchment only\nMRMS points + hydrofabric attributes",
#     fontweight="bold"
# )
# ax4.set_axis_off()
# ax4.set_aspect("equal")

# # Attribute box
# ax4.text(
#     1.02,
#     1.00,
#     attr_text,
#     transform=ax4.transAxes,
#     ha="left",
#     va="top",
#     fontsize=8,
#     family="monospace",
#     bbox=dict(
#         facecolor="white",
#         edgecolor="0.5",
#         alpha=0.95,
#         boxstyle="round,pad=0.5"
#     )
# )

# # -----------------------------
# # Figure legend
# # -----------------------------
# fig.legend(
#     handles=[
#         Line2D([0], [0], color="0.82", lw=1.0, label="VPU or catchment boundaries"),
#         Line2D([0], [0], color="blue", lw=2.2, label=f"Selected VPU {VPU_DEMO}"),
#         Line2D([0], [0], color="red", lw=2.5, label=f"Target catchment {target_id}"),
#         Line2D([0], [0], marker="o", color="none", markerfacecolor="black",
#                markersize=8, label="MRMS points inside target catchment")
#     ],
#     loc="lower center",
#     ncol=4,
#     frameon=True,
#     fontsize=10
# )

# fig.suptitle(
#     f"MRMS → HydroFabric visual check | VPU {VPU_DEMO} | Catchment {target_id}",
#     fontsize=16,
#     fontweight="bold",
#     y=0.98
# )

# plt.tight_layout(rect=[0, 0.04, 1, 0.95])
# plt.show()

In [ ]:
# import textwrap
# import matplotlib.pyplot as plt
# from matplotlib.lines import Line2D
# from shapely.geometry import box

# # ============================================================
# # Four-panel visual check with actual MRMS cell polygons:
# # 1. CONUS context with real VPU polygons
# # 2. Selected VPU with target catchment
# # 3. Local zoom with MRMS cell borders
# # 4. Target catchment with MRMS cells fully inside it
# # ============================================================

# # -----------------------------
# # Choose VPU and target catchment
# # -----------------------------
# VPU_DEMO = "03W"

# cwv = pd.read_parquet(CACHE / f"crosswalk_vpu_{VPU_DEMO}.parquet")

# # MRMS center points
# cwg_pts = gpd.GeoDataFrame(
#     cwv,
#     geometry=gpd.points_from_xy(cwv.lon, cwv.lat),
#     crs=4326
# ).to_crs(5070)

# cats = catchments_master[catchments_master["vpuid"] == VPU_DEMO].copy()

# # Pick a mid-sized catchment based on number of MRMS center points
# counts = cwv["divide_id"].value_counts()
# candidate = counts[(counts > 8) & (counts < 40)]

# if len(candidate) > 0:
#     target_id = candidate.index[0]
# else:
#     target_id = counts.index[len(counts) // 2]

# target = cats[cats["divide_id"] == target_id].copy()
# target_geom = target.geometry.iloc[0]

# # -----------------------------
# # Build actual MRMS cell polygons from center lon/lat
# # Each MRMS cell is 0.01° x 0.01°
# # -----------------------------
# HALF = MRMS_RES / 2.0

# cwv_cells = cwv.copy()

# cwv_cells["geometry"] = [
#     box(lon - HALF, lat - HALF, lon + HALF, lat + HALF)
#     for lon, lat in zip(cwv_cells["lon"], cwv_cells["lat"])
# ]

# mrms_cells = gpd.GeoDataFrame(
#     cwv_cells,
#     geometry="geometry",
#     crs=4326
# ).to_crs(5070)

# # MRMS cells whose centers were assigned to target catchment
# target_center_cells = mrms_cells[mrms_cells["divide_id"] == target_id].copy()

# # MRMS cell-center points assigned to target catchment
# target_pts = cwg_pts[cwg_pts["divide_id"] == target_id].copy()

# # MRMS cells completely inside the target catchment
# target_cells_fully_inside = target_center_cells[
#     target_center_cells.geometry.within(target_geom)
# ].copy()

# print("Target catchment:", target_id)
# print("MRMS center-assigned cells:", len(target_center_cells))
# print("MRMS cells completely inside catchment:", len(target_cells_fully_inside))

# # -----------------------------
# # Real VPU polygons for CONUS context
# # -----------------------------
# vpu_cache = CACHE / "vpu_dissolved_polygons.parquet"

# if vpu_cache.exists():
#     vpu_polys = gpd.read_parquet(vpu_cache)
# else:
#     vpu_polys = (
#         catchments_master[["vpuid", "geometry"]]
#         .dissolve(by="vpuid")
#         .reset_index()
#     )
#     vpu_polys.to_parquet(vpu_cache, index=False)

# vpu_demo_poly = vpu_polys[vpu_polys["vpuid"] == VPU_DEMO]

# # -----------------------------
# # Local zoom extent around target catchment
# # -----------------------------
# minx, miny, maxx, maxy = target.total_bounds

# px = (maxx - minx) * 3 + 800
# py = (maxy - miny) * 3 + 800

# xlim_local = (minx - px, maxx + px)
# ylim_local = (miny - py, maxy + py)

# local_cats = cats.cx[xlim_local[0]:xlim_local[1], ylim_local[0]:ylim_local[1]]
# local_cells = mrms_cells.cx[xlim_local[0]:xlim_local[1], ylim_local[0]:ylim_local[1]]
# local_pts = cwg_pts.cx[xlim_local[0]:xlim_local[1], ylim_local[0]:ylim_local[1]]

# # -----------------------------
# # Target-only extent
# # -----------------------------
# txmin, tymin, txmax, tymax = target.total_bounds

# target_pad_x = (txmax - txmin) * 0.35 + 300
# target_pad_y = (tymax - tymin) * 0.35 + 300

# xlim_target = (txmin - target_pad_x, txmax + target_pad_x)
# ylim_target = (tymin - target_pad_y, tymax + target_pad_y)

# # -----------------------------
# # Prepare attribute text
# # -----------------------------
# target_attrs = target.drop(columns="geometry").iloc[0].to_dict()
# target_attrs["mrms_center_assigned_cells"] = int(len(target_center_cells))
# target_attrs["mrms_cells_fully_inside"] = int(len(target_cells_fully_inside))

# attr_lines = []
# for key, value in target_attrs.items():
#     if pd.isna(value):
#         value = "NA"
#     attr_lines.append(f"{key}: {value}")

# attr_text = "\n".join(attr_lines)

# # ============================================================
# # Plot
# # ============================================================
# fig, axes = plt.subplots(2, 2, figsize=(21, 16))
# fig.patch.set_facecolor("white")

# ax1, ax2, ax3, ax4 = axes.ravel()

# # ------------------------------------------------------------
# # Panel 1: CONUS context
# # ------------------------------------------------------------
# vpu_polys.boundary.plot(
#     ax=ax1,
#     color="0.82",
#     linewidth=0.5
# )

# vpu_demo_poly.plot(
#     ax=ax1,
#     facecolor="none",
#     edgecolor="blue",
#     linewidth=2.2
# )

# target.centroid.plot(
#     ax=ax1,
#     color="red",
#     markersize=35,
#     zorder=5
# )

# ax1.set_title(
#     f"1. CONUS context\nVPU {VPU_DEMO} and target catchment",
#     fontweight="bold"
# )
# ax1.set_axis_off()
# ax1.set_aspect("equal")

# # ------------------------------------------------------------
# # Panel 2: selected VPU
# # ------------------------------------------------------------
# cats.boundary.plot(
#     ax=ax2,
#     color="0.82",
#     linewidth=0.25
# )

# target.plot(
#     ax=ax2,
#     facecolor="none",
#     edgecolor="red",
#     linewidth=2.5
# )

# ax2.set_title(
#     f"2. Selected VPU\nVPU {VPU_DEMO}, target catchment {target_id}",
#     fontweight="bold"
# )
# ax2.set_axis_off()
# ax2.set_aspect("equal")

# # ------------------------------------------------------------
# # Panel 3: local zoom with MRMS cell borders
# # ------------------------------------------------------------
# local_cats.boundary.plot(
#     ax=ax3,
#     color="0.75",
#     linewidth=0.6
# )

# local_cells.boundary.plot(
#     ax=ax3,
#     color="black",
#     linewidth=0.4,
#     linestyle="--",
#     alpha=0.45
# )

# local_pts.plot(
#     ax=ax3,
#     column="divide_id",
#     cmap="tab20",
#     markersize=10,
#     legend=False,
#     alpha=0.8
# )

# target.boundary.plot(
#     ax=ax3,
#     color="red",
#     linewidth=2.5
# )

# ax3.set_xlim(xlim_local)
# ax3.set_ylim(ylim_local)

# ax3.set_title(
#     "3. Local zoom\nMRMS cell borders dashed, centers colored by assigned catchment",
#     fontweight="bold"
# )
# ax3.set_axis_off()
# ax3.set_aspect("equal")

# # ------------------------------------------------------------
# # Panel 4: target catchment with fully inside MRMS cells
# # ------------------------------------------------------------
# target.plot(
#     ax=ax4,
#     facecolor="0.92",
#     edgecolor="red",
#     linewidth=2.8,
#     zorder=1
# )

# # All center-assigned cell borders, light gray
# target_center_cells.boundary.plot(
#     ax=ax4,
#     color="0.65",
#     linewidth=0.7,
#     linestyle="--",
#     alpha=0.7,
#     zorder=2
# )

# # Cells completely inside catchment, stronger dashed black
# if len(target_cells_fully_inside) > 0:
#     target_cells_fully_inside.boundary.plot(
#         ax=ax4,
#         color="black",
#         linewidth=1.2,
#         linestyle="--",
#         zorder=3
#     )

#     target_cells_fully_inside.plot(
#         ax=ax4,
#         facecolor="none",
#         edgecolor="black",
#         linewidth=1.2,
#         linestyle="--",
#         zorder=3
#     )

# # MRMS centers
# target_pts.plot(
#     ax=ax4,
#     color="blue",
#     markersize=20,
#     alpha=0.9,
#     zorder=4
# )

# ax4.set_xlim(xlim_target)
# ax4.set_ylim(ylim_target)

# ax4.set_title(
#     f"4. Target catchment\nMRMS cells completely inside = {len(target_cells_fully_inside)}",
#     fontweight="bold"
# )
# ax4.set_axis_off()
# ax4.set_aspect("equal")

# # Attribute box outside panel 4
# ax4.text(
#     1.02,
#     1.00,
#     attr_text,
#     transform=ax4.transAxes,
#     ha="left",
#     va="top",
#     fontsize=8,
#     family="monospace",
#     bbox=dict(
#         facecolor="white",
#         edgecolor="0.5",
#         alpha=0.95,
#         boxstyle="round,pad=0.5"
#     )
# )

# # -----------------------------
# # Figure legend
# # -----------------------------
# fig.legend(
#     handles=[
#         Line2D([0], [0], color="blue", lw=2.2, label=f"Selected VPU {VPU_DEMO}"),
#         Line2D([0], [0], color="red", lw=2.5, label=f"Target catchment {target_id}"),
#         Line2D([0], [0], color="black", lw=1.2, linestyle="--", label="MRMS cell border"),
#         Line2D([0], [0], marker="o", color="none", markerfacecolor="blue",
#                markersize=7, label="MRMS cell center"),
#         Line2D([0], [0], color="0.65", lw=1.0, linestyle="--",
#                label="Center-assigned MRMS cells"),
#         Line2D([0], [0], color="black", lw=1.5, linestyle="--",
#                label="Cells fully inside catchment")
#     ],
#     loc="lower center",
#     ncol=3,
#     frameon=True,
#     fontsize=10
# )

# fig.suptitle(
#     f"MRMS actual grid-cell check | VPU {VPU_DEMO} | Catchment {target_id}",
#     fontsize=16,
#     fontweight="bold",
#     y=0.98
# )

# plt.tight_layout(rect=[0, 0.06, 1, 0.95])
# plt.show()

In [ ]:
import textwrap
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from shapely.geometry import box

# ============================================================
# Single-plot visual check:
# Target catchment + actual MRMS cell polygons + attributes
# ============================================================

# -----------------------------
# Choose VPU and target catchment
# -----------------------------
VPU_DEMO = "03W"

cwv = pd.read_parquet(CACHE / f"crosswalk_vpu_{VPU_DEMO}.parquet")

# MRMS center points
cwg_pts = gpd.GeoDataFrame(
    cwv,
    geometry=gpd.points_from_xy(cwv.lon, cwv.lat),
    crs=4326,
).to_crs(5070)

cats = catchments_master[catchments_master["vpuid"] == VPU_DEMO].copy()

# Pick a mid-sized catchment based on number of MRMS center points
counts = cwv["divide_id"].value_counts()
candidate = counts[(counts > 8) & (counts < 40)]

if len(candidate) > 0:
    target_id = candidate.index[0]
else:
    target_id = counts.index[len(counts) // 2]

target = cats[cats["divide_id"] == target_id].copy()
target_geom = target.geometry.iloc[0]

# -----------------------------
# Build actual MRMS cell polygons from center lon/lat
# Each MRMS cell is 0.01° x 0.01°
# -----------------------------
HALF = MRMS_RES / 2.0

cwv_cells = cwv.copy()

cwv_cells["geometry"] = [
    box(lon - HALF, lat - HALF, lon + HALF, lat + HALF)
    for lon, lat in zip(cwv_cells["lon"], cwv_cells["lat"])
]

mrms_cells = gpd.GeoDataFrame(
    cwv_cells,
    geometry="geometry",
    crs=4326,
).to_crs(5070)

# MRMS cells whose centers were assigned to the target catchment
target_center_cells = mrms_cells[mrms_cells["divide_id"] == target_id].copy()

# MRMS cell-center points assigned to the target catchment
target_pts = cwg_pts[cwg_pts["divide_id"] == target_id].copy()

# MRMS cells completely inside the target catchment
target_cells_fully_inside = target_center_cells[
    target_center_cells.geometry.within(target_geom)
].copy()

print("Target catchment:", target_id)
print("MRMS center-assigned cells:", len(target_center_cells))
print("MRMS cells completely inside catchment:", len(target_cells_fully_inside))

# -----------------------------
# Plot extent
# -----------------------------
minx, miny, maxx, maxy = target.total_bounds

pad_x = (maxx - minx) * 0.35 + 300
pad_y = (maxy - miny) * 0.35 + 300

xlim = (minx - pad_x, maxx + pad_x)
ylim = (miny - pad_y, maxy + pad_y)

# -----------------------------
# Prepare attribute text
# -----------------------------
target_attrs = target.drop(columns="geometry").iloc[0].to_dict()

target_attrs["mrms_center_assigned_cells"] = int(len(target_center_cells))
target_attrs["mrms_cells_fully_inside"] = int(len(target_cells_fully_inside))

attr_lines = []

for key, value in target_attrs.items():
    if pd.isna(value):
        value = "NA"
    attr_lines.append(f"{key}: {value}")

attr_text = "\n".join(attr_lines)

# ============================================================
# Plot only the target catchment
# ============================================================
fig, ax = plt.subplots(figsize=(12, 12))
fig.patch.set_facecolor("white")

# Target catchment polygon
target.plot(
    ax=ax,
    facecolor="0.92",
    edgecolor="red",
    linewidth=3.0,
    zorder=1,
)

# All MRMS cells whose centers were assigned to this catchment
target_center_cells.boundary.plot(
    ax=ax,
    color="0.65",
    linewidth=0.8,
    linestyle="--",
    alpha=0.75,
    zorder=2,
)

# MRMS cells completely inside the catchment
if len(target_cells_fully_inside) > 0:
    target_cells_fully_inside.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1.4,
        linestyle="--",
        zorder=3,
    )

# MRMS center points
target_pts.plot(
    ax=ax,
    color="blue",
    markersize=25,
    alpha=0.9,
    zorder=4,
)

# Optional row/col labels
if len(target_pts) <= 60:
    for _, r in target_pts.iterrows():
        ax.text(
            r.geometry.x,
            r.geometry.y,
            f"{int(r['row'])},{int(r['col'])}",
            fontsize=6,
            ha="center",
            va="center",
            color="blue",
            zorder=5,
        )

ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_aspect("equal")
ax.set_axis_off()

ax.set_title(
    f"Target catchment {target_id} in VPU {VPU_DEMO}\n"
    f"MRMS actual grid cells: center-assigned vs fully inside",
    fontsize=14,
    fontweight="bold",
)

# Attribute box outside the plot
ax.text(
    1.02,
    1.00,
    attr_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8,
    family="monospace",
    bbox={
        "facecolor": "white",
        "edgecolor": "0.5",
        "alpha": 0.95,
        "boxstyle": "round,pad=0.5",
    },
)

# Legend
ax.legend(
    handles=[
        Line2D([0], [0], color="red", lw=3.0, label="Target catchment boundary"),
        Line2D(
            [0],
            [0],
            color="0.65",
            lw=1.0,
            linestyle="--",
            label="MRMS cells with centers in catchment",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            lw=1.5,
            linestyle="--",
            label="MRMS cells fully inside catchment",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="blue",
            markersize=7,
            label="MRMS cell centers",
        ),
    ],
    loc="lower left",
    frameon=True,
    fontsize=9,
)

plt.tight_layout()
plt.show()

In [ ]:
import textwrap
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from shapely.geometry import box

# ============================================================
# Single catchment plot:
# Actual MRMS cell polygons + fractional area weighting
# ============================================================

# -----------------------------
# Choose VPU and target catchment
# -----------------------------
VPU_DEMO = "03W"

cwv = pd.read_parquet(CACHE / f"crosswalk_vpu_{VPU_DEMO}.parquet")

cats = catchments_master[catchments_master["vpuid"] == VPU_DEMO].copy()

# Pick a mid-sized catchment based on number of MRMS center-assigned cells
counts = cwv["divide_id"].value_counts()
candidate = counts[(counts > 8) & (counts < 40)]

if len(candidate) > 0:
    target_id = candidate.index[0]
else:
    target_id = counts.index[len(counts) // 2]

target = cats[cats["divide_id"] == target_id].copy()
target_geom = target.geometry.iloc[0]

# -----------------------------
# MRMS grid definition
# -----------------------------
MRMS_LON_MIN_EDGE = -130.0
MRMS_LAT_MAX_EDGE = 55.0
MRMS_RES = 0.01
MRMS_NLON = 7000
MRMS_NLAT = 3500
HALF = MRMS_RES / 2.0


def mrms_centers_for_bbox(west, east, south, north, pad=2):
    """
    Return MRMS cell centers covering bbox.
    bbox is in lon/lat EPSG:4326.
    """
    i0 = int(np.floor((west - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) - pad
    i1 = int(np.ceil((east - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) + pad

    j0 = int(np.floor((MRMS_LAT_MAX_EDGE - HALF - north) / MRMS_RES)) - pad
    j1 = int(np.ceil((MRMS_LAT_MAX_EDGE - HALF - south) / MRMS_RES)) + pad

    i0 = max(0, i0)
    i1 = min(MRMS_NLON - 1, i1)

    j0 = max(0, j0)
    j1 = min(MRMS_NLAT - 1, j1)

    cols = np.arange(i0, i1 + 1)
    rows = np.arange(j0, j1 + 1)

    lon = MRMS_LON_MIN_EDGE + HALF + cols * MRMS_RES
    lat = MRMS_LAT_MAX_EDGE - HALF - rows * MRMS_RES

    LON, LAT = np.meshgrid(lon, lat)
    COL, ROW = np.meshgrid(cols, rows)

    return ROW.ravel(), COL.ravel(), LON.ravel(), LAT.ravel()


# -----------------------------
# Build MRMS cells around the target catchment
# Important: this includes cells whose centers may be outside the catchment
# but whose area overlaps the catchment.
# -----------------------------
target_4326 = target.to_crs(4326)
west, south, east, north = target_4326.total_bounds

row, col, lon, lat = mrms_centers_for_bbox(
    west,
    east,
    south,
    north,
    pad=2,
)

mrms_cells_4326 = gpd.GeoDataFrame(
    {
        "cell_id": row.astype(np.int64) * MRMS_NLON + col.astype(np.int64),
        "row": row.astype(np.int32),
        "col": col.astype(np.int32),
        "lon": lon,
        "lat": lat,
        "geometry": [
            box(x - HALF, y - HALF, x + HALF, y + HALF) for x, y in zip(lon, lat)
        ],
    },
    geometry="geometry",
    crs=4326,
)

# Reproject to EPSG:5070 for area calculations
mrms_cells = mrms_cells_4326.to_crs(5070)

# -----------------------------
# Calculate overlap area and fractional count
# -----------------------------
mrms_cells["cell_area_m2"] = mrms_cells.geometry.area

mrms_cells["intersection_geom"] = mrms_cells.geometry.intersection(target_geom)
mrms_cells["intersection_area_m2"] = mrms_cells["intersection_geom"].area

mrms_cells["fraction_inside"] = (
    mrms_cells["intersection_area_m2"] / mrms_cells["cell_area_m2"]
)

# Keep only cells that overlap the catchment at all
overlap_cells = mrms_cells[mrms_cells["fraction_inside"] > 0].copy()

# Classify cells
overlap_cells["cell_class"] = np.where(
    overlap_cells["fraction_inside"] >= 0.999,
    "whole",
    "fractional",
)

whole_cells = overlap_cells[overlap_cells["cell_class"] == "whole"].copy()
fractional_cells = overlap_cells[overlap_cells["cell_class"] == "fractional"].copy()

total_fractional_cell_count = overlap_cells["fraction_inside"].sum()

print("Target catchment:", target_id)
print("All overlapping MRMS cells:", len(overlap_cells))
print("Whole cells completely inside:", len(whole_cells))
print("Fractional edge cells:", len(fractional_cells))
print(f"Total fractional MRMS-cell count: {total_fractional_cell_count:.3f}")

# -----------------------------
# MRMS center points for overlapping cells
# -----------------------------
overlap_pts = gpd.GeoDataFrame(
    overlap_cells.drop(columns=["geometry", "intersection_geom"]),
    geometry=gpd.points_from_xy(overlap_cells["lon"], overlap_cells["lat"]),
    crs=4326,
).to_crs(5070)

# -----------------------------
# Plot extent
# -----------------------------
minx, miny, maxx, maxy = target.total_bounds

pad_x = (maxx - minx) * 0.35 + 300
pad_y = (maxy - miny) * 0.35 + 300

xlim = (minx - pad_x, maxx + pad_x)
ylim = (miny - pad_y, maxy + pad_y)

# -----------------------------
# Prepare attribute text
# -----------------------------
target_attrs = target.drop(columns="geometry").iloc[0].to_dict()

target_attrs["overlapping_mrms_cells"] = int(len(overlap_cells))
target_attrs["whole_cells_inside"] = int(len(whole_cells))
target_attrs["fractional_edge_cells"] = int(len(fractional_cells))
target_attrs["total_fractional_cell_count"] = round(
    float(total_fractional_cell_count),
    3,
)

attr_lines = []

for key, value in target_attrs.items():
    if pd.isna(value):
        value = "NA"
    attr_lines.append(f"{key}: {value}")

attr_text = "\n".join(attr_lines)

# ============================================================
# Plot
# ============================================================
fig, ax = plt.subplots(figsize=(13, 12))
fig.patch.set_facecolor("white")

# Target catchment polygon
target.plot(
    ax=ax,
    facecolor="0.94",
    edgecolor="red",
    linewidth=3.0,
    zorder=1,
)

# All overlapping MRMS cell boundaries
overlap_cells.boundary.plot(
    ax=ax,
    color="0.55",
    linewidth=0.8,
    linestyle="--",
    alpha=0.8,
    zorder=2,
)

# Color cells by fraction inside
overlap_cells.plot(
    ax=ax,
    column="fraction_inside",
    cmap="viridis",
    alpha=0.45,
    edgecolor="0.45",
    linewidth=0.6,
    linestyle="--",
    legend=True,
    legend_kwds={
        "label": "Fraction of MRMS cell inside catchment",
        "shrink": 0.65,
    },
    zorder=3,
)

# Whole cells: strong black dashed border
if len(whole_cells) > 0:
    whole_cells.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1.5,
        linestyle="--",
        zorder=4,
    )

# Fractional edge cells: orange dashed border
if len(fractional_cells) > 0:
    fractional_cells.boundary.plot(
        ax=ax,
        color="orange",
        linewidth=1.5,
        linestyle="--",
        zorder=5,
    )

# MRMS centers
overlap_pts.plot(
    ax=ax,
    color="blue",
    markersize=22,
    alpha=0.95,
    zorder=6,
)

# Label fraction values if not too many cells
if len(overlap_pts) <= 80:
    for _, r in overlap_pts.iterrows():
        ax.text(
            r.geometry.x,
            r.geometry.y,
            f"{r['fraction_inside']:.2f}",
            fontsize=7,
            ha="center",
            va="center",
            color="white",
            fontweight="bold",
            zorder=7,
        )

ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_aspect("equal")
ax.set_axis_off()

ax.set_title(
    f"Target catchment {target_id} in VPU {VPU_DEMO}\n"
    f"Fractional MRMS cell count = {total_fractional_cell_count:.3f}",
    fontsize=14,
    fontweight="bold",
)

# Attribute box
ax.text(
    1.02,
    1.00,
    attr_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8,
    family="monospace",
    bbox={
        "facecolor": "white",
        "edgecolor": "0.5",
        "alpha": 0.95,
        "boxstyle": "round,pad=0.5",
    },
)

# Legend
ax.legend(
    handles=[
        Line2D([0], [0], color="red", lw=3.0, label="Target catchment boundary"),
        Line2D(
            [0],
            [0],
            color="black",
            lw=1.5,
            linestyle="--",
            label="Whole MRMS cell inside catchment",
        ),
        Line2D(
            [0],
            [0],
            color="orange",
            lw=1.5,
            linestyle="--",
            label="Fractional MRMS edge cell",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="blue",
            markersize=7,
            label="MRMS cell center",
        ),
    ],
    loc="lower left",
    frameon=True,
    fontsize=9,
)

plt.tight_layout()
plt.show()

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# from matplotlib.lines import Line2D

# # ============================================================
# # PLOT ONLY:
# # Target catchment + fractional MRMS cells + hydrofabric attributes
# # Uses existing overlap_cells / whole_cells / fractional_cells
# # Does NOT recalculate intersections
# # ============================================================

# # -----------------------------
# # Helper: format attributes
# # -----------------------------
# def format_attrs(title, row_dict, max_lines=40):
#     lines = [title, "-" * len(title)]

#     if row_dict is None or len(row_dict) == 0:
#         lines.append("No matching record found.")
#         return "\n".join(lines)

#     for k, v in row_dict.items():
#         if k == "geometry":
#             continue
#         if pd.isna(v):
#             v = "NA"
#         lines.append(f"{k}: {v}")

#     if len(lines) > max_lines:
#         lines = lines[:max_lines]
#         lines.append("...")

#     return "\n".join(lines)


# # -----------------------------
# # 1. Catchment / divide attributes
# # -----------------------------
# catchment_attrs = target.drop(columns="geometry").iloc[0].to_dict()

# catchment_attrs["overlapping_mrms_cells"] = int(len(overlap_cells))
# catchment_attrs["whole_cells_inside"] = int(len(whole_cells))
# catchment_attrs["fractional_edge_cells"] = int(len(fractional_cells))
# catchment_attrs["total_fractional_cell_count"] = round(float(total_fractional_cell_count), 3)

# catchment_text = format_attrs(
#     "CATCHMENT / DIVIDE ATTRIBUTES",
#     catchment_attrs,
#     max_lines=35
# )


# # -----------------------------
# # 2. Network attributes
# # network links: divide_id ↔ flowpath id ↔ toid ↔ vpuid
# # -----------------------------
# network_row = None
# flowpath_ids = []

# if "network" in globals():
#     net_match = network[
#         network["divide_id"].astype(str) == str(target_id)
#     ].copy()

#     if len(net_match) > 0:
#         network_row = net_match.iloc[0].to_dict()

#         if "id" in net_match.columns:
#             flowpath_ids = net_match["id"].astype(str).unique().tolist()

# network_text = format_attrs(
#     "NETWORK ATTRIBUTES",
#     network_row,
#     max_lines=25
# )


# # -----------------------------
# # 3. Flowpath attributes and geometry
# # -----------------------------
# flowpath_gdf = None
# flowpath_row = None

# if "flowpaths" in globals() and len(flowpath_ids) > 0:
#     flowpath_gdf = flowpaths[
#         flowpaths["id"].astype(str).isin(flowpath_ids)
#     ].copy()

#     if len(flowpath_gdf) > 0:
#         flowpath_row = flowpath_gdf.drop(columns="geometry").iloc[0].to_dict()

#         if flowpath_gdf.crs != target.crs:
#             flowpath_gdf = flowpath_gdf.to_crs(target.crs)

# flowpath_text = format_attrs(
#     "FLOWPATH ATTRIBUTES",
#     flowpath_row,
#     max_lines=25
# )


# # -----------------------------
# # 4. Nexus attributes and geometry
# # -----------------------------
# nexus_gdf = None
# nexus_row = None

# target_nexus_id = None

# if "nexus_id" in catchment_attrs:
#     target_nexus_id = catchment_attrs["nexus_id"]

# if "nexus" in globals() and target_nexus_id is not None:
#     nexus_match = nexus[
#         nexus["id"].astype(str) == str(target_nexus_id)
#     ].copy()

#     if len(nexus_match) > 0:
#         nexus_row = nexus_match.drop(columns="geometry").iloc[0].to_dict()
#         nexus_gdf = nexus_match.copy()

#         if nexus_gdf.crs != target.crs:
#             nexus_gdf = nexus_gdf.to_crs(target.crs)

# nexus_text = format_attrs(
#     "NEXUS ATTRIBUTES",
#     nexus_row,
#     max_lines=25
# )


# # -----------------------------
# # Plot extent
# # -----------------------------
# minx, miny, maxx, maxy = target.total_bounds

# pad_x = (maxx - minx) * 0.45 + 300
# pad_y = (maxy - miny) * 0.45 + 300

# xlim = (minx - pad_x, maxx + pad_x)
# ylim = (miny - pad_y, maxy + pad_y)


# # ============================================================
# # Plot
# # ============================================================
# fig = plt.figure(figsize=(20, 12))
# fig.patch.set_facecolor("white")

# gs = fig.add_gridspec(
#     nrows=1,
#     ncols=2,
#     width_ratios=[1.35, 1.0],
#     wspace=0.08
# )

# ax = fig.add_subplot(gs[0, 0])
# ax_text = fig.add_subplot(gs[0, 1])

# # -----------------------------
# # Map: catchment + MRMS cells
# # -----------------------------
# target.plot(
#     ax=ax,
#     facecolor="0.94",
#     edgecolor="red",
#     linewidth=3.0,
#     zorder=1
# )

# overlap_cells.plot(
#     ax=ax,
#     column="fraction_inside",
#     cmap="jet",
#     alpha=0.50,
#     edgecolor="0.45",
#     linewidth=0.7,
#     linestyle="--",
#     legend=True,
#     legend_kwds={
#         "label": "Fraction of MRMS cell inside catchment",
#         "shrink": 0.70
#     },
#     zorder=2
# )

# # Whole cells: black dashed border
# if len(whole_cells) > 0:
#     whole_cells.boundary.plot(
#         ax=ax,
#         color="black",
#         linewidth=1.5,
#         linestyle="--",
#         zorder=3
#     )

# # Fractional edge cells: orange dashed border
# if len(fractional_cells) > 0:
#     fractional_cells.boundary.plot(
#         ax=ax,
#         color="orange",
#         linewidth=1.6,
#         linestyle="--",
#         zorder=4
#     )

# # Flowpath line
# if flowpath_gdf is not None and len(flowpath_gdf) > 0:
#     flowpath_gdf.plot(
#         ax=ax,
#         color="cyan",
#         linewidth=2.2,
#         zorder=5
#     )

# # Nexus point
# if nexus_gdf is not None and len(nexus_gdf) > 0:
#     nexus_gdf.plot(
#         ax=ax,
#         color="magenta",
#         marker="*",
#         markersize=250,
#         edgecolor="black",
#         linewidth=0.8,
#         zorder=6
#     )

# # MRMS centers
# overlap_pts.plot(
#     ax=ax,
#     color="blue",
#     markersize=24,
#     alpha=0.95,
#     zorder=7
# )

# # Label fraction values
# if len(overlap_pts) <= 100:
#     for _, r in overlap_pts.iterrows():
#         ax.text(
#             r.geometry.x,
#             r.geometry.y,
#             f"{r['fraction_inside']:.2f}",
#             fontsize=7,
#             ha="center",
#             va="center",
#             color="white",
#             fontweight="bold",
#             zorder=8
#         )

# ax.set_xlim(xlim)
# ax.set_ylim(ylim)
# ax.set_aspect("equal")
# ax.set_axis_off()

# ax.set_title(
#     f"Catchment {target_id} | VPU {VPU_DEMO}\n"
#     f"Fractional MRMS cell count = {total_fractional_cell_count:.3f}",
#     fontsize=14,
#     fontweight="bold"
# )

# ax.legend(
#     handles=[
#         Line2D([0], [0], color="red", lw=3.0, label="Catchment boundary"),
#         Line2D([0], [0], color="black", lw=1.5, linestyle="--",
#                label="Whole MRMS cell"),
#         Line2D([0], [0], color="orange", lw=1.6, linestyle="--",
#                label="Fractional MRMS cell"),
#         Line2D([0], [0], marker="o", color="none", markerfacecolor="blue",
#                markersize=7, label="MRMS cell center"),
#         Line2D([0], [0], color="cyan", lw=2.2, label="Flowpath"),
#         Line2D([0], [0], marker="*", color="none", markerfacecolor="magenta",
#                markeredgecolor="black", markersize=14, label="Nexus")
#     ],
#     loc="lower left",
#     frameon=True,
#     fontsize=9
# )

# # -----------------------------
# # Right-side attribute panel
# # -----------------------------
# ax_text.set_axis_off()

# full_text = (
#     catchment_text
#     + "\n\n"
#     + network_text
#     + "\n\n"
#     + flowpath_text
#     + "\n\n"
#     + nexus_text
# )

# ax_text.text(
#     0.00,
#     1.00,
#     full_text,
#     transform=ax_text.transAxes,
#     ha="left",
#     va="top",
#     fontsize=8,
#     family="monospace",
#     bbox=dict(
#         facecolor="white",
#         edgecolor="0.6",
#         alpha=0.97,
#         boxstyle="round,pad=0.6"
#     )
# )

# plt.tight_layout()
# plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# PLOT ONLY:
# Target catchment + fractional MRMS cells + hydrofabric attributes
#
# This block assumes these already exist from the previous calculation:
# overlap_cells, whole_cells, fractional_cells, overlap_pts
# target, target_id, VPU_DEMO, total_fractional_cell_count
# network, flowpaths, nexus
# ============================================================


# -----------------------------
# Helper: format attributes
# -----------------------------
def format_attrs(title, row_dict):
    lines = [title, "-" * len(title)]

    if row_dict is None or len(row_dict) == 0:
        lines.append("No matching record found.")
        return "\n".join(lines)

    for k, v in row_dict.items():
        if k == "geometry":
            continue

        if pd.isna(v):
            v = "NA"

        lines.append(f"{k}: {v}")

    return "\n".join(lines)


# ============================================================
# 1. Catchment / divide attributes
# ============================================================
catchment_attrs = target.drop(columns="geometry").iloc[0].to_dict()

catchment_attrs["overlapping_mrms_cells"] = int(len(overlap_cells))
catchment_attrs["whole_cells_inside"] = int(len(whole_cells))
catchment_attrs["fractional_edge_cells"] = int(len(fractional_cells))
catchment_attrs["total_fractional_cell_count"] = round(
    float(total_fractional_cell_count),
    3,
)

catchment_text = format_attrs(
    "CATCHMENT / DIVIDE ATTRIBUTES",
    catchment_attrs,
)


# ============================================================
# 2. Network attributes
# network links divide_id ↔ flowpath id ↔ toid ↔ vpuid
# ============================================================
network_row = None
flowpath_ids = []

if "network" in globals():
    net_match = network[network["divide_id"].astype(str) == str(target_id)].copy()

    if len(net_match) > 0:
        network_row = net_match.iloc[0].to_dict()

        if "id" in net_match.columns:
            flowpath_ids = net_match["id"].astype(str).dropna().unique().tolist()

network_text = format_attrs(
    "NETWORK ATTRIBUTES",
    network_row,
)


# ============================================================
# 3. Flowpath attributes and geometry
# ============================================================
flowpath_gdf = None
flowpath_row = None

if "flowpaths" in globals() and len(flowpath_ids) > 0:
    flowpath_gdf = flowpaths[flowpaths["id"].astype(str).isin(flowpath_ids)].copy()

    if len(flowpath_gdf) > 0:
        flowpath_row = flowpath_gdf.drop(columns="geometry").iloc[0].to_dict()

        if flowpath_gdf.crs != target.crs:
            flowpath_gdf = flowpath_gdf.to_crs(target.crs)

flowpath_text = format_attrs(
    "FLOWPATH ATTRIBUTES",
    flowpath_row,
)


# ============================================================
# 4. Nexus attributes and geometry
# ============================================================
nexus_gdf = None
nexus_row = None

target_nexus_id = None

if "nexus_id" in catchment_attrs:
    target_nexus_id = catchment_attrs["nexus_id"]

if "nexus" in globals() and target_nexus_id is not None:
    nexus_match = nexus[nexus["id"].astype(str) == str(target_nexus_id)].copy()

    if len(nexus_match) > 0:
        nexus_row = nexus_match.drop(columns="geometry").iloc[0].to_dict()
        nexus_gdf = nexus_match.copy()

        if nexus_gdf.crs != target.crs:
            nexus_gdf = nexus_gdf.to_crs(target.crs)

nexus_text = format_attrs(
    "NEXUS ATTRIBUTES",
    nexus_row,
)


# ============================================================
# Plot extent
# ============================================================
minx, miny, maxx, maxy = target.total_bounds

pad_x = (maxx - minx) * 0.45 + 300
pad_y = (maxy - miny) * 0.45 + 300

xlim = (minx - pad_x, maxx + pad_x)
ylim = (miny - pad_y, maxy + pad_y)


# ============================================================
# Plot
# ============================================================
fig = plt.figure(figsize=(22, 13))
fig.patch.set_facecolor("white")

gs = fig.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[1.35, 1.05],
    wspace=0.08,
)

ax = fig.add_subplot(gs[0, 0])
ax_text = fig.add_subplot(gs[0, 1])

# -----------------------------
# Catchment polygon
# -----------------------------
# target.plot(
#     ax=ax,
#     facecolor="0.94",
#     edgecolor="red",
#     linewidth=3.0,
#     zorder=1
# )

# # -----------------------------
# # MRMS cells colored by fraction inside
# # -----------------------------
# overlap_cells.plot(
#     ax=ax,
#     column="fraction_inside",
#     cmap="jet",
#     vmin=0,
#     vmax=1,
#     alpha=0.55,
#     edgecolor="0.45",
#     linewidth=0.7,
#     linestyle="--",
#     legend=True,
#     legend_kwds={
#         "label": "Fraction of MRMS cell inside catchment",
#         "shrink": 0.70
#     },
#     zorder=2
# )

# Plot MRMS cells first, with stronger opacity
overlap_cells.plot(
    ax=ax,
    column="fraction_inside",
    cmap="jet",
    vmin=0,
    vmax=1,
    alpha=0.90,  # closer to true colorbar color
    edgecolor="0.35",
    linewidth=0.7,
    linestyle="--",
    legend=True,
    legend_kwds={
        "label": "Fraction of MRMS cell inside catchment",
        "shrink": 0.70,
    },
    zorder=2,
)

# Plot catchment boundary only, no gray fill
target.boundary.plot(
    ax=ax,
    color="red",
    linewidth=3.0,
    zorder=6,
)

# -----------------------------
# Whole cells: black dashed border
# -----------------------------
if len(whole_cells) > 0:
    whole_cells.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1.6,
        linestyle="--",
        zorder=3,
    )

# -----------------------------
# Fractional edge cells: orange dashed border
# -----------------------------
if len(fractional_cells) > 0:
    fractional_cells.boundary.plot(
        ax=ax,
        color="orange",
        linewidth=1.8,
        linestyle="--",
        zorder=4,
    )

# -----------------------------
# Flowpath line
# -----------------------------
if flowpath_gdf is not None and len(flowpath_gdf) > 0:
    flowpath_gdf.plot(
        ax=ax,
        color="cyan",
        linewidth=2.4,
        zorder=5,
    )

# -----------------------------
# Nexus point
# -----------------------------
if nexus_gdf is not None and len(nexus_gdf) > 0:
    nexus_gdf.plot(
        ax=ax,
        color="magenta",
        marker="*",
        markersize=280,
        edgecolor="black",
        linewidth=0.9,
        zorder=6,
    )

# -----------------------------
# MRMS cell centers
# -----------------------------
overlap_pts.plot(
    ax=ax,
    color="blue",
    markersize=25,
    alpha=0.95,
    zorder=7,
)

# -----------------------------
# Label fraction values
# -----------------------------
if len(overlap_pts) <= 100:
    for _, r in overlap_pts.iterrows():
        label_color = "white" if r["fraction_inside"] > 0.45 else "black"

        ax.text(
            r.geometry.x,
            r.geometry.y,
            f"{r['fraction_inside']:.2f}",
            fontsize=7,
            ha="center",
            va="center",
            color=label_color,
            fontweight="bold",
            zorder=8,
        )

# -----------------------------
# Map formatting
# -----------------------------
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_aspect("equal")
ax.set_axis_off()

ax.set_title(
    f"Catchment {target_id} | VPU {VPU_DEMO}\n"
    f"Fractional MRMS cell count = {total_fractional_cell_count:.3f}",
    fontsize=14,
    fontweight="bold",
)

# -----------------------------
# Legend
# -----------------------------
ax.legend(
    handles=[
        Line2D(
            [0],
            [0],
            color="red",
            lw=3.0,
            label="Catchment boundary",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            lw=1.6,
            linestyle="--",
            label="Whole MRMS cell",
        ),
        Line2D(
            [0],
            [0],
            color="orange",
            lw=1.8,
            linestyle="--",
            label="Fractional MRMS cell",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="blue",
            markersize=7,
            label="MRMS cell center",
        ),
        Line2D(
            [0],
            [0],
            color="cyan",
            lw=2.4,
            label="Flowpath",
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="none",
            markerfacecolor="magenta",
            markeredgecolor="black",
            markersize=14,
            label="Nexus",
        ),
    ],
    loc="lower left",
    frameon=True,
    fontsize=9,
)


# ============================================================
# Right-side hydrofabric attribute panel
# ============================================================
ax_text.set_axis_off()

full_text = (
    catchment_text
    + "\n\n"
    + network_text
    + "\n\n"
    + flowpath_text
    + "\n\n"
    + nexus_text
)

ax_text.text(
    0.00,
    1.00,
    full_text,
    transform=ax_text.transAxes,
    ha="left",
    va="top",
    fontsize=7.5,
    family="monospace",
    bbox={
        "facecolor": "white",
        "edgecolor": "0.6",
        "alpha": 0.97,
        "boxstyle": "round,pad=0.6",
    },
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# PLOT ONLY + NORMALIZED WEIGHTS:
# Target catchment + fractional MRMS cells + hydrofabric attributes
#
# This block assumes these already exist from the previous calculation:
# overlap_cells, target, target_id, VPU_DEMO
# network, flowpaths, nexus
#
# It does NOT recalculate intersections.
# It only scales fraction_inside so the final MRMS weights sum to 1.
# ============================================================

# ============================================================
# 0. Normalize MRMS weights
# ============================================================
overlap_cells = overlap_cells.copy()

if "fraction_inside" not in overlap_cells.columns:
    raise ValueError(
        "overlap_cells must already contain 'fraction_inside' from the previous calculation.",
    )

total_fractional_cell_count = overlap_cells["fraction_inside"].sum()

if total_fractional_cell_count <= 0:
    raise ValueError("Total fractional cell count is zero. Check overlap_cells.")

# This is the weight to use later for precipitation averaging
overlap_cells["mrms_weight"] = (
    overlap_cells["fraction_inside"] / total_fractional_cell_count
)

# Rebuild whole/fractional subsets using the existing fraction_inside values
whole_cells = overlap_cells[overlap_cells["fraction_inside"] >= 0.999].copy()
fractional_cells = overlap_cells[
    (overlap_cells["fraction_inside"] > 0) & (overlap_cells["fraction_inside"] < 0.999)
].copy()

# Rebuild MRMS center points for plotting only
drop_cols = ["geometry"]

if "intersection_geom" in overlap_cells.columns:
    drop_cols.append("intersection_geom")

overlap_pts = gpd.GeoDataFrame(
    overlap_cells.drop(columns=drop_cols),
    geometry=gpd.points_from_xy(overlap_cells["lon"], overlap_cells["lat"]),
    crs=4326,
).to_crs(target.crs)

print("Target catchment:", target_id)
print(f"Total fractional cell count        : {total_fractional_cell_count:.6f}")
print(f"Sum of normalized MRMS weights     : {overlap_cells['mrms_weight'].sum():.6f}")
print(f"Whole cells                        : {len(whole_cells)}")
print(f"Fractional edge cells              : {len(fractional_cells)}")

# Later rainfall averaging will be:
# catchment_avg_precip = sum(MRMS_precip_rate * mrms_weight)


# -----------------------------
# Helper: format attributes
# -----------------------------
def format_attrs(title, row_dict):
    lines = [title, "-" * len(title)]

    if row_dict is None or len(row_dict) == 0:
        lines.append("No matching record found.")
        return "\n".join(lines)

    for k, v in row_dict.items():
        if k == "geometry":
            continue

        if pd.isna(v):
            v = "NA"

        lines.append(f"{k}: {v}")

    return "\n".join(lines)


# ============================================================
# 1. Catchment / divide attributes
# ============================================================
catchment_attrs = target.drop(columns="geometry").iloc[0].to_dict()

catchment_attrs["overlapping_mrms_cells"] = int(len(overlap_cells))
catchment_attrs["whole_cells_inside"] = int(len(whole_cells))
catchment_attrs["fractional_edge_cells"] = int(len(fractional_cells))
catchment_attrs["total_fractional_cell_count"] = round(
    float(total_fractional_cell_count),
    6,
)
catchment_attrs["sum_normalized_mrms_weight"] = round(
    float(overlap_cells["mrms_weight"].sum()),
    6,
)

catchment_text = format_attrs(
    "CATCHMENT / DIVIDE ATTRIBUTES",
    catchment_attrs,
)


# ============================================================
# 2. Network attributes
# ============================================================
network_row = None
flowpath_ids = []

if "network" in globals():
    net_match = network[network["divide_id"].astype(str) == str(target_id)].copy()

    if len(net_match) > 0:
        network_row = net_match.iloc[0].to_dict()

        if "id" in net_match.columns:
            flowpath_ids = net_match["id"].astype(str).dropna().unique().tolist()

network_text = format_attrs(
    "NETWORK ATTRIBUTES",
    network_row,
)


# ============================================================
# 3. Flowpath attributes and geometry
# ============================================================
flowpath_gdf = None
flowpath_row = None

if "flowpaths" in globals() and len(flowpath_ids) > 0:
    flowpath_gdf = flowpaths[flowpaths["id"].astype(str).isin(flowpath_ids)].copy()

    if len(flowpath_gdf) > 0:
        flowpath_row = flowpath_gdf.drop(columns="geometry").iloc[0].to_dict()

        if flowpath_gdf.crs != target.crs:
            flowpath_gdf = flowpath_gdf.to_crs(target.crs)

flowpath_text = format_attrs(
    "FLOWPATH ATTRIBUTES",
    flowpath_row,
)


# ============================================================
# 4. Nexus attributes and geometry
# ============================================================
nexus_gdf = None
nexus_row = None

target_nexus_id = None

if "nexus_id" in catchment_attrs:
    target_nexus_id = catchment_attrs["nexus_id"]

if "nexus" in globals() and target_nexus_id is not None:
    nexus_match = nexus[nexus["id"].astype(str) == str(target_nexus_id)].copy()

    if len(nexus_match) > 0:
        nexus_row = nexus_match.drop(columns="geometry").iloc[0].to_dict()
        nexus_gdf = nexus_match.copy()

        if nexus_gdf.crs != target.crs:
            nexus_gdf = nexus_gdf.to_crs(target.crs)

nexus_text = format_attrs(
    "NEXUS ATTRIBUTES",
    nexus_row,
)


# ============================================================
# Plot extent
# ============================================================
minx, miny, maxx, maxy = target.total_bounds

pad_x = (maxx - minx) * 0.45 + 300
pad_y = (maxy - miny) * 0.45 + 300

xlim = (minx - pad_x, maxx + pad_x)
ylim = (miny - pad_y, maxy + pad_y)


# ============================================================
# Plot
# ============================================================
fig = plt.figure(figsize=(22, 13))
fig.patch.set_facecolor("white")

gs = fig.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[1.35, 1.05],
    wspace=0.08,
)

ax = fig.add_subplot(gs[0, 0])
ax_text = fig.add_subplot(gs[0, 1])


# -----------------------------
# MRMS cells colored by normalized precipitation weight
# -----------------------------
overlap_cells.plot(
    ax=ax,
    column="mrms_weight",
    cmap="jet",
    vmin=0,
    vmax=overlap_cells["mrms_weight"].max(),
    alpha=0.90,
    edgecolor="0.35",
    linewidth=0.7,
    linestyle="--",
    legend=True,
    legend_kwds={
        "label": "Normalized MRMS weight for catchment average",
        "shrink": 0.70,
    },
    zorder=2,
)

# Catchment boundary only
target.boundary.plot(
    ax=ax,
    color="red",
    linewidth=3.0,
    zorder=6,
)

# Whole cells: black dashed border
if len(whole_cells) > 0:
    whole_cells.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1.6,
        linestyle="--",
        zorder=3,
    )

# Fractional edge cells: orange dashed border
if len(fractional_cells) > 0:
    fractional_cells.boundary.plot(
        ax=ax,
        color="orange",
        linewidth=1.8,
        linestyle="--",
        zorder=4,
    )

# Flowpath line
if flowpath_gdf is not None and len(flowpath_gdf) > 0:
    flowpath_gdf.plot(
        ax=ax,
        color="cyan",
        linewidth=2.4,
        zorder=5,
    )

# Nexus point
if nexus_gdf is not None and len(nexus_gdf) > 0:
    nexus_gdf.plot(
        ax=ax,
        color="magenta",
        marker="*",
        markersize=280,
        edgecolor="black",
        linewidth=0.9,
        zorder=7,
    )

# MRMS cell centers
overlap_pts.plot(
    ax=ax,
    color="blue",
    markersize=25,
    alpha=0.95,
    zorder=8,
)

# Label normalized weights
if len(overlap_pts) <= 100:
    for _, r in overlap_pts.iterrows():
        label_color = (
            "white"
            if r["mrms_weight"] > overlap_cells["mrms_weight"].max() * 0.45
            else "black"
        )

        ax.text(
            r.geometry.x,
            r.geometry.y,
            f"{r['mrms_weight']:.3f}",
            fontsize=7,
            ha="center",
            va="center",
            color=label_color,
            fontweight="bold",
            zorder=9,
        )

# Map formatting
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_aspect("equal")
ax.set_axis_off()

ax.set_title(
    f"Catchment {target_id} | VPU {VPU_DEMO}\n"
    f"Normalized MRMS weights sum = {overlap_cells['mrms_weight'].sum():.3f}",
    fontsize=14,
    fontweight="bold",
)

# Legend
ax.legend(
    handles=[
        Line2D(
            [0],
            [0],
            color="red",
            lw=3.0,
            label="Catchment boundary",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            lw=1.6,
            linestyle="--",
            label="Whole MRMS cell",
        ),
        Line2D(
            [0],
            [0],
            color="orange",
            lw=1.8,
            linestyle="--",
            label="Fractional MRMS cell",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="blue",
            markersize=7,
            label="MRMS cell center",
        ),
        Line2D(
            [0],
            [0],
            color="cyan",
            lw=2.4,
            label="Flowpath",
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="none",
            markerfacecolor="magenta",
            markeredgecolor="black",
            markersize=14,
            label="Nexus",
        ),
    ],
    loc="lower left",
    frameon=True,
    fontsize=9,
)


# ============================================================
# Right-side hydrofabric attribute panel
# ============================================================
ax_text.set_axis_off()

full_text = (
    catchment_text
    + "\n\n"
    + network_text
    + "\n\n"
    + flowpath_text
    + "\n\n"
    + nexus_text
)

ax_text.text(
    0.00,
    1.00,
    full_text,
    transform=ax_text.transAxes,
    ha="left",
    va="top",
    fontsize=7.5,
    family="monospace",
    bbox={
        "facecolor": "white",
        "edgecolor": "0.6",
        "alpha": 0.97,
        "boxstyle": "round,pad=0.6",
    },
)

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

# ============================================================
# Step 2 readiness check
# Checks:
# 1. catchments_master
# 2. crosswalk
# 3. COMBINED
# 4. mrms_crosswalk_cache
# ============================================================

print("Checking Step 2 outputs...\n")

# -----------------------------
# 1. Check catchments_master
# -----------------------------
assert "catchments_master" in globals(), (
    "❌ catchments_master is not in memory. Run Step 1 first."
)

assert isinstance(catchments_master, gpd.GeoDataFrame), (
    "❌ catchments_master exists, but it is not a GeoDataFrame."
)

needed_catchment_cols = [
    "divide_id",
    "flowpath_id",
    "nexus_id",
    "vpuid",
    "nexus_type",
    "is_terminal",
    "geometry",
]

missing = [c for c in needed_catchment_cols if c not in catchments_master.columns]

assert len(missing) == 0, f"❌ catchments_master is missing columns: {missing}"

print("✅ catchments_master exists")
print(f"   rows: {len(catchments_master):,}")
print(f"   VPUs: {catchments_master['vpuid'].nunique()}")
print(f"   CRS : {catchments_master.crs}\n")


# -----------------------------
# 2. Check CACHE / mrms_crosswalk_cache
# -----------------------------
assert "CACHE" in globals(), "❌ CACHE variable is not defined."
assert isinstance(CACHE, Path), "❌ CACHE exists, but it is not a Path object."
assert CACHE.exists(), f"❌ Cache folder does not exist: {CACHE}"

cache_files = sorted(CACHE.glob("crosswalk_vpu_*.parquet"))

assert len(cache_files) > 0, (
    "❌ No VPU crosswalk parquet files found in mrms_crosswalk_cache."
)

print("✅ mrms_crosswalk_cache exists")
print(f"   folder: {CACHE.resolve()}")
print(f"   VPU cache files: {len(cache_files)}\n")


# -----------------------------
# 3. Check COMBINED path
# -----------------------------
assert "COMBINED" in globals(), "❌ COMBINED variable is not defined."

assert COMBINED.exists(), f"❌ Combined crosswalk file does not exist: {COMBINED}"

print("✅ COMBINED exists")
print(f"   file: {COMBINED.resolve()}")
print(f"   size: {COMBINED.stat().st_size / 1e6:.1f} MB\n")


# -----------------------------
# 4. Check crosswalk table
# -----------------------------
if "crosswalk" not in globals():
    print("⚠️ crosswalk is not in memory, so I am loading it from COMBINED...")
    crosswalk = pd.read_parquet(COMBINED)

assert isinstance(crosswalk, pd.DataFrame), (
    "❌ crosswalk exists, but it is not a DataFrame."
)

needed_crosswalk_cols = [
    "cell_id",
    "row",
    "col",
    "lon",
    "lat",
    "divide_id",
    "vpuid",
    "nexus_id",
]

missing = [c for c in needed_crosswalk_cols if c not in crosswalk.columns]

assert len(missing) == 0, f"❌ crosswalk is missing columns: {missing}"

assert len(crosswalk) > 0, "❌ crosswalk is empty."

duplicate_cells = crosswalk["cell_id"].duplicated().sum()

print("✅ crosswalk exists")
print(f"   rows: {len(crosswalk):,}")
print(f"   unique MRMS cells: {crosswalk['cell_id'].nunique():,}")
print(f"   duplicate cell_id rows: {duplicate_cells:,}")
print(f"   catchments hit: {crosswalk['divide_id'].nunique():,}")
print(f"   VPUs represented: {crosswalk['vpuid'].nunique()}\n")


# -----------------------------
# 5. Check whether every VPU has a cache file
# -----------------------------
expected_vpus = set(catchments_master["vpuid"].dropna().astype(str).unique())

cached_vpus = {f.stem.replace("crosswalk_vpu_", "") for f in cache_files}

missing_vpus = sorted(expected_vpus - cached_vpus)
extra_vpus = sorted(cached_vpus - expected_vpus)

if len(missing_vpus) == 0:
    print("✅ Every VPU has a cached crosswalk file")
else:
    print("⚠️ Missing VPU cache files:")
    print(missing_vpus)

if len(extra_vpus) > 0:
    print("⚠️ Extra cache files not matching catchments_master VPUs:")
    print(extra_vpus)

print("\nSTEP 2 READY ✅")

# The block below takes too long to finish


# Step 2b — Build AREA-WEIGHTED / FRACTIONAL MRMS crosswalk

# Output:
fractional_crosswalk
mrms_fractional_crosswalk_cache/
mrms_hf_fractional_crosswalk_conus.parquet

# Important:
Unlike the center-point crosswalk, this table can have
repeated cell_id values because one MRMS cell can overlap
more than one catchment.

In [ ]:
# ============================================================
# Step 2b — Build AREA-WEIGHTED / FRACTIONAL MRMS crosswalk
#
# Output:
#   fractional_crosswalk
#   mrms_fractional_crosswalk_cache/
#   mrms_hf_fractional_crosswalk_conus.parquet
#
# Important:
#   Unlike the center-point crosswalk, this table can have
#   repeated cell_id values because one MRMS cell can overlap
#   more than one catchment.
# ============================================================

import time
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import box

# ------------------------------------------------------------
# Required objects from previous steps
# ------------------------------------------------------------
assert "catchments_master" in globals(), "Run Step 1 first: catchments_master missing."
assert "CACHE" in globals(), "Run Step 2 first: CACHE missing."

# ------------------------------------------------------------
# MRMS grid definition
# Same as Step 2
# ------------------------------------------------------------
MRMS_LON_MIN_EDGE = -130.0
MRMS_LAT_MAX_EDGE = 55.0

MRMS_RES = 0.01
MRMS_NLON = 7000
MRMS_NLAT = 3500

HALF = MRMS_RES / 2.0
AREA_CRS = 5070  # EPSG:5070, meters; good for area calculations


# ------------------------------------------------------------
# Output folders/files
# ------------------------------------------------------------
FRAC_CACHE = CACHE / "mrms_fractional_crosswalk_cache"
FRAC_CACHE.mkdir(exist_ok=True)

FRAC_COMBINED = CACHE / "mrms_hf_fractional_crosswalk_conus.parquet"


# ------------------------------------------------------------
# Columns to carry from HydroFabric
# ------------------------------------------------------------
CARRY_FRAC = [
    "divide_id",
    "vpuid",
    "nexus_id",
    "nexus_type",
    "is_terminal",
    "geometry",
]

if "terminal_nexus_id" in catchments_master.columns:
    CARRY_FRAC.insert(-1, "terminal_nexus_id")


# ------------------------------------------------------------
# Helper: MRMS centers covering a lon/lat bbox
# ------------------------------------------------------------
def mrms_centers_for_bbox(west, east, south, north, pad=2):
    """
    Return MRMS grid-cell centers covering a lon/lat bbox.

    Returns
    -------
        row, col, lon, lat
    """
    i0 = int(np.floor((west - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) - pad
    i1 = int(np.ceil((east - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) + pad

    j0 = int(np.floor((MRMS_LAT_MAX_EDGE - HALF - north) / MRMS_RES)) - pad
    j1 = int(np.ceil((MRMS_LAT_MAX_EDGE - HALF - south) / MRMS_RES)) + pad

    i0 = max(0, i0)
    i1 = min(MRMS_NLON - 1, i1)

    j0 = max(0, j0)
    j1 = min(MRMS_NLAT - 1, j1)

    if i1 < i0 or j1 < j0:
        return None

    cols = np.arange(i0, i1 + 1)
    rows = np.arange(j0, j1 + 1)

    lon = MRMS_LON_MIN_EDGE + HALF + cols * MRMS_RES
    lat = MRMS_LAT_MAX_EDGE - HALF - rows * MRMS_RES

    LON, LAT = np.meshgrid(lon, lat)
    COL, ROW = np.meshgrid(cols, rows)

    return ROW.ravel(), COL.ravel(), LON.ravel(), LAT.ravel()


# ------------------------------------------------------------
# Helper: build actual MRMS cell polygons from center lon/lat
# ------------------------------------------------------------
def build_mrms_cell_polygons(row, col, lon, lat):
    """
    Build actual 0.01 x 0.01 degree MRMS grid-cell polygons.
    Geometry is first made in EPSG:4326, then reprojected to EPSG:5070.
    """
    gdf_4326 = gpd.GeoDataFrame(
        {
            "cell_id": row.astype(np.int64) * MRMS_NLON + col.astype(np.int64),
            "row": row.astype(np.int32),
            "col": col.astype(np.int32),
            "lon": lon.astype(float),
            "lat": lat.astype(float),
            "geometry": [
                box(x - HALF, y - HALF, x + HALF, y + HALF) for x, y in zip(lon, lat)
            ],
        },
        geometry="geometry",
        crs=4326,
    )

    gdf_5070 = gdf_4326.to_crs(AREA_CRS)

    gdf_5070["cell_area_m2"] = gdf_5070.geometry.area

    return gdf_5070


# ------------------------------------------------------------
# Helper: calculate intersection area in chunks
# ------------------------------------------------------------
def calculate_intersection_area(candidates, chunk_size=250_000):
    """
    Calculate area of MRMS cell polygon ∩ catchment polygon.
    Chunking prevents memory from exploding for large VPUs.
    """
    areas = []

    for start in range(0, len(candidates), chunk_size):
        end = min(start + chunk_size, len(candidates))

        chunk = candidates.iloc[start:end].copy()

        catchment_geoms = gpd.GeoSeries(
            chunk["catchment_geometry"],
            crs=AREA_CRS,
            index=chunk.index,
        )

        inter = chunk.geometry.intersection(catchment_geoms)

        areas.append(inter.area.to_numpy())

    return np.concatenate(areas)


# ------------------------------------------------------------
# Choose VPUs to run
# ------------------------------------------------------------
# Full CONUS:
VPUS_TO_RUN = sorted(catchments_master["vpuid"].dropna().unique())

# For quick testing first, you can temporarily use:
# VPUS_TO_RUN = ["03W"]


# ------------------------------------------------------------
# Build fractional crosswalk, one VPU at a time
# ------------------------------------------------------------
parts = []
t_all = time.time()

for vpu in VPUS_TO_RUN:
    out = FRAC_CACHE / f"fractional_crosswalk_vpu_{vpu}.parquet"

    if out.exists():
        df_cached = pd.read_parquet(out)
        parts.append(df_cached)

        print(f"{vpu}: cached fractional crosswalk ({len(df_cached):,} rows)")
        continue

    print(f"\nWorking on VPU {vpu}...")

    sub = catchments_master[catchments_master["vpuid"] == vpu][CARRY_FRAC].copy()

    if len(sub) == 0:
        print(f"{vpu}: no catchments")
        continue

    # Make sure catchments are in EPSG:5070
    if sub.crs is None:
        raise ValueError("catchments_master has no CRS. It should be EPSG:5070.")

    sub = sub.to_crs(AREA_CRS)

    # Fix invalid geometries if needed
    try:
        sub["geometry"] = sub.geometry.make_valid()
    except Exception:
        sub["geometry"] = sub.geometry.buffer(0)

    # Bounding box in lon/lat
    west, south, east, north = sub.to_crs(4326).total_bounds

    res = mrms_centers_for_bbox(
        west,
        east,
        south,
        north,
        pad=2,
    )

    if res is None:
        print(f"{vpu}: no MRMS cells in bbox")
        continue

    row, col, lon, lat = res

    # Build actual MRMS polygons
    mrms_cells = build_mrms_cell_polygons(row, col, lon, lat)

    print(f"  MRMS candidate cells in VPU bbox: {len(mrms_cells):,}")
    print(f"  Catchments in VPU: {len(sub):,}")

    # Spatial join gives candidate MRMS-cell / catchment pairs
    candidates = gpd.sjoin(
        mrms_cells,
        sub,
        how="inner",
        predicate="intersects",
    )

    if len(candidates) == 0:
        print(f"{vpu}: no overlapping cell/catchment candidates")
        continue

    print(f"  Candidate cell/catchment pairs: {len(candidates):,}")

    # Attach right-side catchment geometry using index_right
    catchment_geom_lookup = sub.geometry.rename("catchment_geometry")

    candidates = candidates.join(
        catchment_geom_lookup,
        on="index_right",
    )

    # Calculate true intersection area
    candidates["intersection_area_m2"] = calculate_intersection_area(
        candidates,
        chunk_size=250_000,
    )

    # Remove zero-area boundary touches
    candidates = candidates[candidates["intersection_area_m2"] > 0.01].copy()

    # Fraction of each MRMS cell inside each catchment
    candidates["fraction_inside"] = (
        candidates["intersection_area_m2"] / candidates["cell_area_m2"]
    )

    # Guard against tiny numerical overflows like 1.0000000002
    candidates["fraction_inside"] = candidates["fraction_inside"].clip(
        lower=0,
        upper=1,
    )

    # Keep only useful columns
    keep_cols = [
        "cell_id",
        "row",
        "col",
        "lon",
        "lat",
        "divide_id",
        "vpuid",
        "nexus_id",
        "nexus_type",
        "is_terminal",
        "cell_area_m2",
        "intersection_area_m2",
        "fraction_inside",
    ]

    if "terminal_nexus_id" in candidates.columns:
        keep_cols.insert(9, "terminal_nexus_id")

    df = (
        pd.DataFrame(candidates[keep_cols])
        .drop_duplicates(subset=["divide_id", "cell_id"])
        .sort_values(["divide_id", "cell_id"])
        .reset_index(drop=True)
    )

    df.to_parquet(out, index=False)
    parts.append(df)

    print(f"  Saved: {out}")
    print(f"  Final fractional rows: {len(df):,}")
    print(f"  Catchments hit: {df['divide_id'].nunique():,}")
    print(f"  Unique MRMS cells touched: {df['cell_id'].nunique():,}")


# ------------------------------------------------------------
# Combine all VPU files
# ------------------------------------------------------------
fractional_crosswalk = (
    pd.concat(parts, ignore_index=True)
    .drop_duplicates(subset=["divide_id", "cell_id"])
    .reset_index(drop=True)
)

fractional_crosswalk.to_parquet(FRAC_COMBINED, index=False)

print("\n============================================================")
print("FRACTIONAL CROSSWALK DONE ✅")
print("============================================================")
print(f"Total rows              : {len(fractional_crosswalk):,}")
print(f"Unique catchments       : {fractional_crosswalk['divide_id'].nunique():,}")
print(f"Unique MRMS cells       : {fractional_crosswalk['cell_id'].nunique():,}")
print(f"Saved combined file     : {FRAC_COMBINED}")
print(f"Total time              : {(time.time() - t_all) / 60:.1f} minutes")

In [ ]:
# ============================================================
# Step 2b readiness check
# ============================================================

assert "fractional_crosswalk" in globals(), "fractional_crosswalk is not in memory."
assert FRAC_COMBINED.exists(), f"Missing file: {FRAC_COMBINED}"

needed_cols = [
    "cell_id",
    "row",
    "col",
    "lon",
    "lat",
    "divide_id",
    "vpuid",
    "nexus_id",
    "cell_area_m2",
    "intersection_area_m2",
    "fraction_inside",
]

missing = [c for c in needed_cols if c not in fractional_crosswalk.columns]
assert len(missing) == 0, f"Missing columns: {missing}"

assert len(fractional_crosswalk) > 0, "fractional_crosswalk is empty."
assert fractional_crosswalk["fraction_inside"].between(0, 1).all(), (
    "Some fractions are outside 0–1."
)

print("fractional_crosswalk rows:", f"{len(fractional_crosswalk):,}")
print("unique catchments:", f"{fractional_crosswalk['divide_id'].nunique():,}")
print("unique MRMS cells:", f"{fractional_crosswalk['cell_id'].nunique():,}")
print("fraction range:")
print(fractional_crosswalk["fraction_inside"].describe())

print("\nSTEP 2b FRACTIONAL CROSSWALK READY ✅")

In [ ]:
# # Example structure:
# # mrms_values should have:
# #   cell_id
# #   precip_mm

# tmp = mrms_values.merge(
#     fractional_crosswalk[["divide_id", "cell_id", "fraction_inside"]],
#     on="cell_id",
#     how="inner"
# )

# catchment_precip = (
#     tmp.assign(weighted_precip=lambda d: d["precip_mm"] * d["fraction_inside"])
#        .groupby("divide_id")
#        .agg(
#            precip_sum_weighted=("weighted_precip", "sum"),
#            weight_sum=("fraction_inside", "sum")
#        )
#        .reset_index()
# )

# catchment_precip["precip_mm_area_weighted"] = (
#     catchment_precip["precip_sum_weighted"] /
#     catchment_precip["weight_sum"]
# )

# Will do later for step 2

1. Use center-based crosswalk to quickly find event-relevant catchments.
2. Build fractional_crosswalk only for those catchments.
3. Save it.
4. Download MRMS.
5. Multiply MRMS precip by fractional weights.
6. Summarize by catchment and time.

# Step 3 — Download MRMS precipitation and compute catchment-averaged rainfall rate

Step 1 built the HydroFabric catchment table.
Step 2 built the MRMS-to-catchment spatial relationship, including fractional MRMS cell coverage inside each catchment.

Step 3 now uses that spatial relationship to download MRMS precipitation data and summarize it at the catchment scale.

## Goal

For each selected catchment and time step, we will compute an average MRMS precipitation rate using the fraction of each MRMS grid cell that lies inside the catchment.

This avoids treating edge cells as simply inside or outside. Instead, each MRMS cell contributes according to the percentage of its area that overlaps the catchment.

## Fractional averaging method

For each catchment:

`weighted MRMS rate = sum(MRMS rate × fraction_inside) / sum(fraction_inside)`

where:

* `MRMS rate` is the precipitation rate from the MRMS grid cell
* `fraction_inside` is the fraction of that MRMS cell area inside the catchment
* `sum(fraction_inside)` is the total fractional number of MRMS cells contributing to the catchment

For example, if a cell is 90% inside the catchment, it contributes `0.9` of that cell to the average.

## What this step will do

1. Select the catchments and MRMS cells needed for the analysis.

2. Download MRMS precipitation data for the target event dates and time steps.

3. Extract MRMS values for the grid cells that overlap each catchment.

4. Weight each MRMS value by its `fraction_inside`.

5. Compute a catchment-averaged precipitation rate for every catchment and time step.

6. Save the output as a time-series table.

## Expected output

The main output will be a catchment-scale MRMS precipitation table, such as:

`time → divide_id → vpuid → weighted_mean_precip_rate`

This output will be used as rainfall forcing or rainfall-summary input for later flash-flood modeling, event analysis, and USGS gage comparison.


In [ ]:
# # ============================================================
# # Step 3.1 — Load flash-flood event catalog and make event points
# #
# # Inputs needed from previous steps:
# #   catchments_master
# #   crosswalk
# #
# # Outputs from this cell:
# #   events_raw
# #   events_gdf_4326
# #   events_gdf_5070
# # ============================================================

# import pandas as pd
# import geopandas as gpd
# import numpy as np
# from pathlib import Path

# # ------------------------------------------------------------
# # 0. Check previous-step objects
# # ------------------------------------------------------------
# assert "catchments_master" in globals(), "❌ catchments_master missing. Run Step 1 first."
# assert "crosswalk" in globals(), "❌ crosswalk missing. Run Step 2 first."

# print("✅ Step 1 and Step 2 objects found")
# print(f"catchments_master rows: {len(catchments_master):,}")
# print(f"crosswalk rows        : {len(crosswalk):,}")


# # ------------------------------------------------------------
# # 1. Set event catalog path
# # ------------------------------------------------------------
# # Change this filename if your event catalog has a different name.
# EVENT_CATALOG = Path("merged_flash_flood_events_2021_2025.parquet")

# # If the exact file is not found, search nearby files automatically
# if not EVENT_CATALOG.exists():
#     possible_files = (
#         list(Path(".").glob("*event*.parquet")) +
#         list(Path(".").glob("*Event*.parquet")) +
#         list(Path(".").glob("*flash*.parquet")) +
#         list(Path(".").glob("*flood*.parquet")) +
#         list(Path(".").glob("*event*.csv")) +
#         list(Path(".").glob("*Event*.csv")) +
#         list(Path(".").glob("*flash*.csv")) +
#         list(Path(".").glob("*flood*.csv"))
#     )

#     if len(possible_files) == 0:
#         raise FileNotFoundError(
#             "❌ No event catalog found. Put the merged event catalog in this folder "
#             "or update EVENT_CATALOG manually."
#         )

#     EVENT_CATALOG = possible_files[0]
#     print(f"⚠️ Exact filename not found. Using detected file: {EVENT_CATALOG}")


# # ------------------------------------------------------------
# # 2. Load event catalog
# # ------------------------------------------------------------
# suffix = EVENT_CATALOG.suffix.lower()

# if suffix == ".parquet":
#     events_raw = pd.read_parquet(EVENT_CATALOG)
# elif suffix == ".csv":
#     events_raw = pd.read_csv(EVENT_CATALOG)
# elif suffix in [".xlsx", ".xls"]:
#     events_raw = pd.read_excel(EVENT_CATALOG)
# else:
#     raise ValueError(f"Unsupported event file type: {suffix}")

# print("\n✅ Event catalog loaded")
# print(f"file: {EVENT_CATALOG}")
# print(f"rows: {len(events_raw):,}")
# print("columns:")
# print(list(events_raw.columns))


# # ------------------------------------------------------------
# # 3. Standardize column names
# # ------------------------------------------------------------
# events_raw = events_raw.copy()
# events_raw.columns = (
#     events_raw.columns
#     .str.strip()
#     .str.lower()
#     .str.replace(" ", "_")
#     .str.replace("-", "_")
# )

# # ------------------------------------------------------------
# # 4. Find longitude / latitude columns
# # ------------------------------------------------------------
# lon_candidates = [
#     "lon", "longitude", "centroid_lon", "centroid_longitude",
#     "event_lon", "event_longitude", "begin_lon", "begin_longitude"
# ]

# lat_candidates = [
#     "lat", "latitude", "centroid_lat", "centroid_latitude",
#     "event_lat", "event_latitude", "begin_lat", "begin_latitude"
# ]

# lon_col = next((c for c in lon_candidates if c in events_raw.columns), None)
# lat_col = next((c for c in lat_candidates if c in events_raw.columns), None)

# assert lon_col is not None, "❌ Could not find longitude column."
# assert lat_col is not None, "❌ Could not find latitude column."

# print(f"\n✅ Coordinate columns found: {lon_col}, {lat_col}")


# # ------------------------------------------------------------
# # 5. Create event_id if missing
# # ------------------------------------------------------------
# event_id_candidates = [
#     "event_id", "merged_event_id", "storm_event_id", "id"
# ]

# event_id_col = next((c for c in event_id_candidates if c in events_raw.columns), None)

# if event_id_col is None:
#     events_raw["event_id"] = np.arange(1, len(events_raw) + 1)
#     event_id_col = "event_id"
#     print("⚠️ No event_id column found. Created event_id from row number.")
# else:
#     events_raw = events_raw.rename(columns={event_id_col: "event_id"})
#     event_id_col = "event_id"

# print("✅ event_id ready")


# # ------------------------------------------------------------
# # 6. Optional: parse begin/end time if present
# # ------------------------------------------------------------
# begin_candidates = [
#     "begin_time", "begin_datetime", "start_time", "event_begin_time",
#     "begin_date_time", "begin_date"
# ]

# end_candidates = [
#     "end_time", "end_datetime", "stop_time", "event_end_time",
#     "end_date_time", "end_date"
# ]

# begin_col = next((c for c in begin_candidates if c in events_raw.columns), None)
# end_col = next((c for c in end_candidates if c in events_raw.columns), None)

# if begin_col is not None:
#     events_raw["begin_time"] = pd.to_datetime(events_raw[begin_col], errors="coerce", utc=True)
#     print(f"✅ begin_time parsed from: {begin_col}")
# else:
#     print("⚠️ No begin_time column found yet.")

# if end_col is not None:
#     events_raw["end_time"] = pd.to_datetime(events_raw[end_col], errors="coerce", utc=True)
#     print(f"✅ end_time parsed from: {end_col}")
# else:
#     print("⚠️ No end_time column found yet.")


# # ------------------------------------------------------------
# # 7. Remove events without valid coordinates
# # ------------------------------------------------------------
# events_raw[lon_col] = pd.to_numeric(events_raw[lon_col], errors="coerce")
# events_raw[lat_col] = pd.to_numeric(events_raw[lat_col], errors="coerce")

# before = len(events_raw)

# events_raw = events_raw.dropna(subset=[lon_col, lat_col]).copy()

# events_raw = events_raw[
#     events_raw[lon_col].between(-180, 180) &
#     events_raw[lat_col].between(-90, 90)
# ].copy()

# after = len(events_raw)

# print(f"\nRemoved events with invalid coordinates: {before - after:,}")
# print(f"Remaining events: {after:,}")


# # ------------------------------------------------------------
# # 8. Convert to GeoDataFrame
# # ------------------------------------------------------------
# events_gdf_4326 = gpd.GeoDataFrame(
#     events_raw,
#     geometry=gpd.points_from_xy(events_raw[lon_col], events_raw[lat_col]),
#     crs="EPSG:4326"
# )

# events_gdf_5070 = events_gdf_4326.to_crs(catchments_master.crs)

# print("\n✅ Event GeoDataFrames created")
# print(f"events_gdf_4326 CRS: {events_gdf_4326.crs}")
# print(f"events_gdf_5070 CRS: {events_gdf_5070.crs}")
# print(f"events_gdf_5070 rows: {len(events_gdf_5070):,}")

# events_gdf_5070.head()


# ============================================================
# Step 3.1 — Load final event-gage CSV
#
# Input:
#   final_events (5).csv
#
# Outputs:
#   events_raw
#   event_gage_pairs_4326
#   event_gage_pairs_5070
#   storm_points_5070
#   gage_points_5070
# ============================================================


# ============================================================# ============================================================# ============================================================
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 0. Check previous-step objects
# ------------------------------------------------------------
assert "catchments_master" in globals(), (
    "❌ catchments_master missing. Run Step 1 first."
)
assert "crosswalk" in globals(), "❌ crosswalk missing. Run Step 2 first."

# ------------------------------------------------------------
# 1. Load filtered event-gage CSV
# ------------------------------------------------------------
EVENT_GAGE_CSV = Path("final_events (5).csv")

events_raw = pd.read_csv(EVENT_GAGE_CSV)

print("✅ Loaded event-gage CSV")
print(f"rows: {len(events_raw):,}")
print("columns:")
print(list(events_raw.columns))


# ------------------------------------------------------------
# 2. Standardize key columns
# ------------------------------------------------------------
events_raw = events_raw.copy()

events_raw["event_id"] = events_raw["storm_index"]

events_raw["cluster_id"] = events_raw["cluster_id"]

events_raw["begin_time"] = pd.to_datetime(
    events_raw["begin"],
    errors="coerce",
    utc=True,
)

events_raw["end_time"] = pd.to_datetime(
    events_raw["end"],
    errors="coerce",
    utc=True,
)

events_raw["storm_lat"] = pd.to_numeric(
    events_raw["centroid_lat"],
    errors="coerce",
)

events_raw["storm_lon"] = pd.to_numeric(
    events_raw["centroid_lon"],
    errors="coerce",
)

events_raw["gage_lat"] = pd.to_numeric(
    events_raw["gage_lat"],
    errors="coerce",
)

events_raw["gage_lon"] = pd.to_numeric(
    events_raw["gage_lon"],
    errors="coerce",
)

events_raw["footprint_radius_km"] = pd.to_numeric(
    events_raw["footprint_radius_km"],
    errors="coerce",
)

events_raw["centroid_to_gage_dist_km"] = pd.to_numeric(
    events_raw["centroid_to_gage_dist_km"],
    errors="coerce",
)


# ------------------------------------------------------------
# 3. Remove rows missing storm or gage coordinates
# ------------------------------------------------------------
before = len(events_raw)

events_raw = events_raw.dropna(
    subset=[
        "event_id",
        "begin_time",
        "end_time",
        "storm_lat",
        "storm_lon",
        "gage_lat",
        "gage_lon",
        "usgs_gage_id",
    ],
).copy()

after = len(events_raw)

print(f"\nRemoved invalid rows: {before - after:,}")
print(f"Remaining event-gage rows: {after:,}")


# ------------------------------------------------------------
# 4. Create storm-point GeoDataFrame
# one row per event-gage pair, geometry = storm centroid
# ------------------------------------------------------------
event_gage_pairs_4326 = gpd.GeoDataFrame(
    events_raw,
    geometry=gpd.points_from_xy(
        events_raw["storm_lon"],
        events_raw["storm_lat"],
    ),
    crs="EPSG:4326",
)

event_gage_pairs_5070 = event_gage_pairs_4326.to_crs(catchments_master.crs)

print("\n✅ Created event_gage_pairs_5070")
print(f"rows: {len(event_gage_pairs_5070):,}")
print(f"CRS : {event_gage_pairs_5070.crs}")


# ------------------------------------------------------------
# 5. Create unique storm points
# one row per storm/event
# ------------------------------------------------------------
storm_cols = [
    "event_id",
    "cluster_id",
    "begin_time",
    "end_time",
    "duration_hours",
    "storm_lat",
    "storm_lon",
    "footprint_radius_km",
    "primary_state",
    "states",
    "wfos",
    "deaths_total",
    "injuries_total",
    "damage_total_usd",
    "flood_causes",
]

storm_points_4326 = (
    events_raw[storm_cols].drop_duplicates(subset=["event_id"]).reset_index(drop=True)
)

storm_points_4326 = gpd.GeoDataFrame(
    storm_points_4326,
    geometry=gpd.points_from_xy(
        storm_points_4326["storm_lon"],
        storm_points_4326["storm_lat"],
    ),
    crs="EPSG:4326",
)

storm_points_5070 = storm_points_4326.to_crs(catchments_master.crs)

print("\n✅ Created storm_points_5070")
print(f"unique storms: {len(storm_points_5070):,}")


# ------------------------------------------------------------
# 6. Create unique gage points
# one row per USGS gage
# ------------------------------------------------------------
gage_cols = [
    "usgs_gage_id",
    "STATID",
    "DRAIN_SQKM",
    "gage_lat",
    "gage_lon",
]

gage_points_4326 = (
    events_raw[gage_cols]
    .drop_duplicates(subset=["usgs_gage_id"])
    .reset_index(drop=True)
)

gage_points_4326 = gpd.GeoDataFrame(
    gage_points_4326,
    geometry=gpd.points_from_xy(
        gage_points_4326["gage_lon"],
        gage_points_4326["gage_lat"],
    ),
    crs="EPSG:4326",
)

gage_points_5070 = gage_points_4326.to_crs(catchments_master.crs)

print("\n✅ Created gage_points_5070")
print(f"unique gages: {len(gage_points_5070):,}")


# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------
print("\n============================================================")
print("STEP 3.1 READY ✅")
print("============================================================")
print(f"event-gage pairs : {len(event_gage_pairs_5070):,}")
print(f"unique storms    : {len(storm_points_5070):,}")
print(f"unique USGS gages: {len(gage_points_5070):,}")
print(
    f"time range       : {events_raw['begin_time'].min()} to {events_raw['end_time'].max()}",
)

event_gage_pairs_5070.head()

In [ ]:
# events_raw["event_id"] = events_raw["cluster_id"]

# events_raw["begin_time"] = pd.to_datetime(events_raw["begin"], errors="coerce", utc=True)
# events_raw["end_time"]   = pd.to_datetime(events_raw["end"], errors="coerce", utc=True)

# events_raw["lat"] = pd.to_numeric(events_raw["centroid_lat"], errors="coerce")
# events_raw["lon"] = pd.to_numeric(events_raw["centroid_lon"], errors="coerce")

# events_raw["footprint_radius_km"] = pd.to_numeric(
#     events_raw["footprint_radius_km"],
#     errors="coerce"
# )


################################################################
# Event/storm ID
events_raw["event_id"] = events_raw[
    "storm_index"
]  # or cluster_id if that is your main event ID

# Storm time
events_raw["begin_time"] = pd.to_datetime(
    events_raw["begin"],
    errors="coerce",
    utc=True,
)
events_raw["end_time"] = pd.to_datetime(events_raw["end"], errors="coerce", utc=True)

# Storm location
events_raw["storm_lat"] = pd.to_numeric(events_raw["centroid_lat"], errors="coerce")
events_raw["storm_lon"] = pd.to_numeric(events_raw["centroid_lon"], errors="coerce")

# Storm footprint
events_raw["footprint_radius_km"] = pd.to_numeric(
    events_raw["footprint_radius_km"],
    errors="coerce",
)

# Gage location
events_raw["gage_lat"] = pd.to_numeric(events_raw["gage_lat"], errors="coerce")
events_raw["gage_lon"] = pd.to_numeric(events_raw["gage_lon"], errors="coerce")

# Gage distance from storm centroid
events_raw["centroid_to_gage_dist_km"] = pd.to_numeric(
    events_raw["centroid_to_gage_dist_km"],
    errors="coerce",
)

## Step 3.2 — Create storm/gage points and identify event catchments

This step converts the filtered event-gage CSV into geospatial layers. Each row has two important locations: the storm centroid and the matched USGS gage. The storm centroid is used to define the rainfall/event footprint, while the gage point is kept for later comparison with observed streamflow.

First, invalid rows with missing time, storm coordinates, gage coordinates, footprint radius, or gage ID are removed. Then the code creates three GeoDataFrames: one for all event-gage pairs, one for unique storm centroids, and one for unique USGS gage locations. All layers are projected to the HydroFabric CRS so distances and buffers are measured in meters.

The storm footprint is created by buffering each storm centroid using `footprint_radius_km`. These storm buffers are then spatially intersected with the HydroFabric catchment polygons. The output, `event_catchments`, tells us which catchments fall inside each storm footprint.

The main purpose of this step is to reduce the problem size. Instead of processing all CONUS catchments, we identify only the catchments relevant to the filtered flash-flood events. These selected catchments will later be used to build a much smaller fractional MRMS crosswalk and extract area-weighted precipitation.

In [ ]:
# ============================================================
# Step 3.2 — Create storm/gage points and find event catchments
#
# Inputs:
#   events_raw
#   catchments_master
#
# Outputs:
#   event_gage_pairs_5070
#   storm_points_5070
#   gage_points_5070
#   storm_buffers_5070
#   event_catchments
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Remove invalid rows
# ------------------------------------------------------------
before = len(events_raw)

events_raw = events_raw.dropna(
    subset=[
        "event_id",
        "begin_time",
        "end_time",
        "storm_lat",
        "storm_lon",
        "gage_lat",
        "gage_lon",
        "footprint_radius_km",
        "usgs_gage_id",
    ],
).copy()

after = len(events_raw)

print(f"Removed invalid rows: {before - after:,}")
print(f"Remaining rows: {after:,}")


# ------------------------------------------------------------
# 2. Create event-gage pair GeoDataFrame
# geometry = storm centroid
# ------------------------------------------------------------
event_gage_pairs_4326 = gpd.GeoDataFrame(
    events_raw,
    geometry=gpd.points_from_xy(
        events_raw["storm_lon"],
        events_raw["storm_lat"],
    ),
    crs="EPSG:4326",
)

event_gage_pairs_5070 = event_gage_pairs_4326.to_crs(catchments_master.crs)

print("✅ event_gage_pairs_5070 created")
print(f"rows: {len(event_gage_pairs_5070):,}")


# ------------------------------------------------------------
# 3. Create unique storm points
# ------------------------------------------------------------
storm_cols = [
    "event_id",
    "cluster_id",
    "begin_time",
    "end_time",
    "duration_hours",
    "storm_lat",
    "storm_lon",
    "footprint_radius_km",
    "primary_state",
    "states",
    "wfos",
    "flood_causes",
]

storm_cols = [c for c in storm_cols if c in events_raw.columns]

storm_points_4326 = (
    events_raw[storm_cols].drop_duplicates(subset=["event_id"]).reset_index(drop=True)
)

storm_points_4326 = gpd.GeoDataFrame(
    storm_points_4326,
    geometry=gpd.points_from_xy(
        storm_points_4326["storm_lon"],
        storm_points_4326["storm_lat"],
    ),
    crs="EPSG:4326",
)

storm_points_5070 = storm_points_4326.to_crs(catchments_master.crs)

print("✅ storm_points_5070 created")
print(f"unique storms: {len(storm_points_5070):,}")


# ------------------------------------------------------------
# 4. Create unique gage points
# ------------------------------------------------------------
gage_cols = [
    "usgs_gage_id",
    "STATID",
    "DRAIN_SQKM",
    "gage_lat",
    "gage_lon",
]

gage_cols = [c for c in gage_cols if c in events_raw.columns]

gage_points_4326 = (
    events_raw[gage_cols]
    .drop_duplicates(subset=["usgs_gage_id"])
    .reset_index(drop=True)
)

gage_points_4326 = gpd.GeoDataFrame(
    gage_points_4326,
    geometry=gpd.points_from_xy(
        gage_points_4326["gage_lon"],
        gage_points_4326["gage_lat"],
    ),
    crs="EPSG:4326",
)

gage_points_5070 = gage_points_4326.to_crs(catchments_master.crs)

print("✅ gage_points_5070 created")
print(f"unique gages: {len(gage_points_5070):,}")


# ------------------------------------------------------------
# 5. Create storm footprint buffers
# ------------------------------------------------------------
storm_buffers_5070 = storm_points_5070.copy()

storm_buffers_5070["buffer_radius_m"] = (
    storm_buffers_5070["footprint_radius_km"] * 1000.0
)

storm_buffers_5070["geometry"] = storm_buffers_5070.geometry.buffer(
    storm_buffers_5070["buffer_radius_m"],
)

print("✅ storm_buffers_5070 created")


# ------------------------------------------------------------
# 6. Intersect storm buffers with HydroFabric catchments
# ------------------------------------------------------------
catchment_cols = [
    "divide_id",
    "vpuid",
    "nexus_id",
    "flowpath_id",
    "nexus_type",
    "is_terminal",
    "geometry",
]

catchment_cols = [c for c in catchment_cols if c in catchments_master.columns]

catchments_for_join = catchments_master[catchment_cols].copy()

if catchments_for_join.crs != storm_buffers_5070.crs:
    catchments_for_join = catchments_for_join.to_crs(storm_buffers_5070.crs)

event_catchments = gpd.sjoin(
    storm_buffers_5070,
    catchments_for_join,
    how="inner",
    predicate="intersects",
)

event_catchments = (
    pd.DataFrame(event_catchments.drop(columns="geometry"))
    .drop(columns=["index_right"], errors="ignore")
    .drop_duplicates(subset=["event_id", "divide_id"])
    .reset_index(drop=True)
)

print("\n============================================================")
print("STEP 3.2 READY ✅")
print("============================================================")
print(f"event-catchment pairs : {len(event_catchments):,}")
print(f"unique events         : {event_catchments['event_id'].nunique():,}")
print(f"unique catchments     : {event_catchments['divide_id'].nunique():,}")
print(f"unique VPUs           : {event_catchments['vpuid'].nunique():,}")

event_catchments.head()

## Step 3.3 — Select only event-relevant catchments and MRMS cells

This step uses the `event_catchments` table from the previous block to reduce the dataset size. Instead of working with all CONUS HydroFabric catchments and all MRMS cells, we keep only the catchments that intersect the filtered storm footprints.

The code creates `selected_catchments`, which contains only the HydroFabric catchments needed for the selected storm events. Then it joins these catchments with the existing center-based MRMS `crosswalk` to identify the MRMS grid cells associated with those event catchments.

The output from this step is not the final area-weighted precipitation yet. It is a smaller working dataset that tells us which catchments and MRMS cells are relevant. The next step will use this reduced set to build a much faster fractional crosswalk only for these selected catchments.

In [ ]:
# ============================================================
# Step 3.3 — Select event-relevant catchments and MRMS cells
#
# Inputs:
#   event_catchments
#   catchments_master
#   crosswalk
#
# Outputs:
#   selected_divide_ids
#   selected_catchments
#   event_crosswalk
#   selected_mrms_cells
# ============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path

# ------------------------------------------------------------
# 0. Check required objects
# ------------------------------------------------------------
assert "event_catchments" in globals(), (
    "❌ event_catchments missing. Run Step 3.2 first."
)
assert "catchments_master" in globals(), "❌ catchments_master missing."
assert "crosswalk" in globals(), "❌ crosswalk missing."

print("✅ Required objects found")


# ------------------------------------------------------------
# 1. Get selected catchment IDs
# ------------------------------------------------------------
selected_divide_ids = event_catchments["divide_id"].dropna().unique()

print(f"Selected event-relevant catchments: {len(selected_divide_ids):,}")


# ------------------------------------------------------------
# 2. Subset HydroFabric catchments
# ------------------------------------------------------------
selected_catchments = catchments_master[
    catchments_master["divide_id"].isin(selected_divide_ids)
].copy()

print(f"selected_catchments rows: {len(selected_catchments):,}")
print(f"selected VPUs: {selected_catchments['vpuid'].nunique():,}")


# ------------------------------------------------------------
# 3. Subset center-based MRMS crosswalk
# ------------------------------------------------------------
selected_crosswalk = crosswalk[crosswalk["divide_id"].isin(selected_divide_ids)].copy()

print(f"selected_crosswalk rows: {len(selected_crosswalk):,}")
print(f"unique MRMS cells: {selected_crosswalk['cell_id'].nunique():,}")


# ------------------------------------------------------------
# 4. Connect events to MRMS cells through catchments
# ------------------------------------------------------------
event_crosswalk = event_catchments[["event_id", "divide_id"]].merge(
    selected_crosswalk,
    on="divide_id",
    how="inner",
)

event_crosswalk = event_crosswalk.drop_duplicates(
    subset=["event_id", "divide_id", "cell_id"],
).reset_index(drop=True)

print(f"event_crosswalk rows: {len(event_crosswalk):,}")
print(f"unique events in event_crosswalk: {event_crosswalk['event_id'].nunique():,}")
print(
    f"unique catchments in event_crosswalk: {event_crosswalk['divide_id'].nunique():,}",
)
print(f"unique MRMS cells in event_crosswalk: {event_crosswalk['cell_id'].nunique():,}")


# ------------------------------------------------------------
# 5. Unique MRMS cells needed for all selected events
# ------------------------------------------------------------
selected_mrms_cells = (
    event_crosswalk[["cell_id", "row", "col", "lon", "lat"]]
    .drop_duplicates(subset=["cell_id"])
    .sort_values(["row", "col"])
    .reset_index(drop=True)
)

print(f"selected_mrms_cells rows: {len(selected_mrms_cells):,}")


# ------------------------------------------------------------
# 6. Save reduced files
# ------------------------------------------------------------
STEP3_CACHE = CACHE / "step3_event_relevant"
STEP3_CACHE.mkdir(exist_ok=True)

selected_catchments_file = STEP3_CACHE / "selected_catchments.parquet"
event_crosswalk_file = STEP3_CACHE / "event_crosswalk_center_based.parquet"
selected_mrms_cells_file = STEP3_CACHE / "selected_mrms_cells.parquet"

selected_catchments.to_parquet(selected_catchments_file, index=False)
event_crosswalk.to_parquet(event_crosswalk_file, index=False)
selected_mrms_cells.to_parquet(selected_mrms_cells_file, index=False)

print("\n============================================================")
print("STEP 3.3 READY ✅")
print("============================================================")
print(f"Saved selected_catchments : {selected_catchments_file}")
print(f"Saved event_crosswalk     : {event_crosswalk_file}")
print(f"Saved selected_mrms_cells : {selected_mrms_cells_file}")

event_crosswalk.head()

In [ ]:
print(sorted(selected_catchments["vpuid"].dropna().astype(str).unique()))
print("Count:", selected_catchments["vpuid"].dropna().astype(str).nunique())

In [ ]:
print(sorted(catchments_master["vpuid"].dropna().astype(str).unique()))
print(
    "Total in catchments_master:",
    catchments_master["vpuid"].dropna().astype(str).nunique(),
)

## Step 3.4 — Build fractional MRMS crosswalk only for selected event catchments

This step creates the area-weighted MRMS-to-catchment crosswalk, but only for the catchments selected in Step 3.3. This is much faster than building a fractional crosswalk for all CONUS.

For each selected catchment, the code finds the nearby MRMS grid cells, builds actual MRMS cell polygons, intersects them with the catchment polygon, and calculates the fraction of each MRMS cell that lies inside the catchment.

The output is `selected_fractional_crosswalk`. This table contains `divide_id`, `cell_id`, `row`, `col`, `fraction_inside`, and `weight`. The `weight` values sum to 1 for each catchment, so they can be used later to compute area-weighted MRMS precipitation.

In [ ]:
# ============================================================
# Step 3.4 — Build fractional crosswalk for selected catchments only
#
# Inputs:
#   selected_catchments
#   selected_crosswalk
#   CACHE
#
# Outputs:
#   selected_fractional_crosswalk
#   selected_fractional_crosswalk.parquet
# ============================================================

import time
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from pathlib import Path

# ------------------------------------------------------------
# 0. Checks
# ------------------------------------------------------------
assert "selected_catchments" in globals(), (
    "❌ selected_catchments missing. Run Step 3.3 first."
)
assert "selected_crosswalk" in globals(), (
    "❌ selected_crosswalk missing. Run Step 3.3 first."
)
assert "CACHE" in globals(), "❌ CACHE missing."

print("✅ Required objects found")

# ------------------------------------------------------------
# 1. MRMS grid settings
# ------------------------------------------------------------
MRMS_LON_MIN_EDGE = -130.0
MRMS_LAT_MAX_EDGE = 55.0
MRMS_RES = 0.01
MRMS_NLON = 7000
MRMS_NLAT = 3500
HALF = MRMS_RES / 2.0
AREA_CRS = 5070

# Neighbor padding around center-based MRMS cells
# 1 is usually enough; 2 is safer for edge cases.
PAD_CELLS = 2

# ------------------------------------------------------------
# 2. Output folder
# ------------------------------------------------------------
FRAC_SELECTED_CACHE = (
    CACHE / "step3_event_relevant" / "selected_fractional_crosswalk_cache"
)
FRAC_SELECTED_CACHE.mkdir(parents=True, exist_ok=True)

SELECTED_FRAC_FILE = (
    CACHE / "step3_event_relevant" / "selected_fractional_crosswalk.parquet"
)


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------
def lon_from_col(col):
    return MRMS_LON_MIN_EDGE + HALF + col * MRMS_RES


def lat_from_row(row):
    return MRMS_LAT_MAX_EDGE - HALF - row * MRMS_RES


def cell_id_from_row_col(row, col):
    return row.astype(np.int64) * MRMS_NLON + col.astype(np.int64)


def build_mrms_cells_from_rows_cols(rows, cols):
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)

    lon = lon_from_col(cols)
    lat = lat_from_row(rows)

    gdf = gpd.GeoDataFrame(
        {
            "cell_id": cell_id_from_row_col(rows, cols),
            "row": rows,
            "col": cols,
            "lon": lon,
            "lat": lat,
            "geometry": [
                box(x - HALF, y - HALF, x + HALF, y + HALF) for x, y in zip(lon, lat)
            ],
        },
        geometry="geometry",
        crs="EPSG:4326",
    ).to_crs(AREA_CRS)

    gdf["cell_area_m2"] = gdf.geometry.area

    return gdf


def rows_cols_for_bbox(west, south, east, north, pad=2):
    i0 = int(np.floor((west - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) - pad
    i1 = int(np.ceil((east - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) + pad

    j0 = int(np.floor((MRMS_LAT_MAX_EDGE - HALF - north) / MRMS_RES)) - pad
    j1 = int(np.ceil((MRMS_LAT_MAX_EDGE - HALF - south) / MRMS_RES)) + pad

    i0 = max(0, i0)
    i1 = min(MRMS_NLON - 1, i1)

    j0 = max(0, j0)
    j1 = min(MRMS_NLAT - 1, j1)

    if i1 < i0 or j1 < j0:
        return pd.DataFrame(columns=["row", "col"])

    cols = np.arange(i0, i1 + 1)
    rows = np.arange(j0, j1 + 1)

    COL, ROW = np.meshgrid(cols, rows)

    return pd.DataFrame(
        {
            "row": ROW.ravel().astype(np.int32),
            "col": COL.ravel().astype(np.int32),
        },
    )


def add_neighbor_cells(df, pad=2):
    base = df[["row", "col"]].drop_duplicates().copy()

    pieces = []

    for dr in range(-pad, pad + 1):
        for dc in range(-pad, pad + 1):
            tmp = base.copy()
            tmp["row"] = tmp["row"] + dr
            tmp["col"] = tmp["col"] + dc
            pieces.append(tmp)

    out = pd.concat(pieces, ignore_index=True)

    out = out[
        out["row"].between(0, MRMS_NLAT - 1) & out["col"].between(0, MRMS_NLON - 1)
    ]

    return out.drop_duplicates().reset_index(drop=True)


def intersection_area_chunked(candidates, chunk_size=150_000):
    areas = []

    for start in range(0, len(candidates), chunk_size):
        end = min(start + chunk_size, len(candidates))
        chunk = candidates.iloc[start:end].copy()

        catchment_geoms = gpd.GeoSeries(
            chunk["catchment_geometry"],
            crs=AREA_CRS,
            index=chunk.index,
        )

        inter = chunk.geometry.intersection(catchment_geoms)
        areas.append(inter.area.to_numpy())

    return np.concatenate(areas)


# ------------------------------------------------------------
# 4. Run by VPU to control memory
# ------------------------------------------------------------
parts = []
t0 = time.time()

vpus = sorted(selected_catchments["vpuid"].dropna().astype(str).unique())

print(f"VPUs to process: {len(vpus)}")
print(vpus)

for vpu in vpus:
    out_file = FRAC_SELECTED_CACHE / f"selected_fractional_crosswalk_vpu_{vpu}.parquet"

    if out_file.exists():
        df_cached = pd.read_parquet(out_file)
        parts.append(df_cached)
        print(f"{vpu}: cached ({len(df_cached):,} rows)")
        continue

    print(f"\nWorking on VPU {vpu}...")

    sub = selected_catchments[selected_catchments["vpuid"].astype(str) == vpu].copy()

    if len(sub) == 0:
        print(f"{vpu}: no selected catchments")
        continue

    sub = sub.to_crs(AREA_CRS)

    try:
        sub["geometry"] = sub.geometry.make_valid()
    except Exception:
        sub["geometry"] = sub.geometry.buffer(0)

    divide_ids_vpu = sub["divide_id"].dropna().unique()

    # --------------------------------------------------------
    # Candidate cells from center-based crosswalk + neighbors
    # --------------------------------------------------------
    cw_vpu = selected_crosswalk[selected_crosswalk["divide_id"].isin(divide_ids_vpu)][
        ["divide_id", "cell_id", "row", "col"]
    ].copy()

    candidate_cells = add_neighbor_cells(
        cw_vpu[["row", "col"]],
        pad=PAD_CELLS,
    )

    # --------------------------------------------------------
    # Add cells for catchments with no center-based MRMS cell
    # --------------------------------------------------------
    catchments_with_center_cell = set(cw_vpu["divide_id"].dropna().unique())
    missing_center_catchments = sub[
        ~sub["divide_id"].isin(catchments_with_center_cell)
    ].copy()

    print(f"  selected catchments: {len(sub):,}")
    print(f"  catchments without center MRMS cell: {len(missing_center_catchments):,}")

    extra_cells = []

    if len(missing_center_catchments) > 0:
        missing_4326 = missing_center_catchments.to_crs(4326)

        for geom in missing_4326.geometry:
            west, south, east, north = geom.bounds
            extra_cells.append(
                rows_cols_for_bbox(west, south, east, north, pad=PAD_CELLS),
            )

    if len(extra_cells) > 0:
        extra_cells = pd.concat(extra_cells, ignore_index=True)
        candidate_cells = pd.concat(
            [candidate_cells, extra_cells],
            ignore_index=True,
        )

    candidate_cells = candidate_cells.drop_duplicates(
        subset=["row", "col"],
    ).reset_index(drop=True)

    print(f"  candidate MRMS cells: {len(candidate_cells):,}")

    if len(candidate_cells) == 0:
        print(f"{vpu}: no candidate cells")
        continue

    # --------------------------------------------------------
    # Build MRMS cell polygons
    # --------------------------------------------------------
    mrms_cells = build_mrms_cells_from_rows_cols(
        candidate_cells["row"].values,
        candidate_cells["col"].values,
    )

    # --------------------------------------------------------
    # Spatial join: MRMS cell polygons intersect catchments
    # --------------------------------------------------------
    join_cols = [
        "divide_id",
        "vpuid",
        "nexus_id",
        "nexus_type",
        "is_terminal",
        "geometry",
    ]

    if "terminal_nexus_id" in sub.columns:
        join_cols.insert(-1, "terminal_nexus_id")

    join_cols = [c for c in join_cols if c in sub.columns]

    sub_join = sub[join_cols].copy()

    candidates = gpd.sjoin(
        mrms_cells,
        sub_join,
        how="inner",
        predicate="intersects",
    )

    if len(candidates) == 0:
        print(f"{vpu}: no intersecting cells")
        continue

    print(f"  candidate cell/catchment pairs: {len(candidates):,}")

    # Attach catchment geometry
    catchment_geom_lookup = sub_join.geometry.rename("catchment_geometry")

    candidates = candidates.join(
        catchment_geom_lookup,
        on="index_right",
    )

    # --------------------------------------------------------
    # True intersection area
    # --------------------------------------------------------
    candidates["intersection_area_m2"] = intersection_area_chunked(
        candidates,
        chunk_size=150_000,
    )

    candidates = candidates[candidates["intersection_area_m2"] > 0.01].copy()

    candidates["fraction_inside"] = (
        candidates["intersection_area_m2"] / candidates["cell_area_m2"]
    ).clip(0, 1)

    # --------------------------------------------------------
    # Save useful columns
    # --------------------------------------------------------
    keep_cols = [
        "cell_id",
        "row",
        "col",
        "lon",
        "lat",
        "divide_id",
        "vpuid",
        "nexus_id",
        "cell_area_m2",
        "intersection_area_m2",
        "fraction_inside",
    ]

    optional_cols = ["nexus_type", "is_terminal", "terminal_nexus_id"]

    for c in optional_cols:
        if c in candidates.columns and c not in keep_cols:
            keep_cols.append(c)

    df = (
        pd.DataFrame(candidates[keep_cols])
        .drop_duplicates(subset=["divide_id", "cell_id"])
        .sort_values(["divide_id", "cell_id"])
        .reset_index(drop=True)
    )

    # Normalize to weights that sum to 1 for each catchment
    df["weight"] = df["fraction_inside"] / df.groupby("divide_id")[
        "fraction_inside"
    ].transform("sum")

    df.to_parquet(out_file, index=False)
    parts.append(df)

    print(f"  final rows: {len(df):,}")
    print(f"  catchments hit: {df['divide_id'].nunique():,}")
    print(f"  saved: {out_file}")


# ------------------------------------------------------------
# 5. Combine all selected VPU fractional files
# ------------------------------------------------------------
selected_fractional_crosswalk = (
    pd.concat(parts, ignore_index=True)
    .drop_duplicates(subset=["divide_id", "cell_id"])
    .reset_index(drop=True)
)

selected_fractional_crosswalk["weight"] = selected_fractional_crosswalk[
    "fraction_inside"
] / selected_fractional_crosswalk.groupby("divide_id")["fraction_inside"].transform(
    "sum",
)

selected_fractional_crosswalk.to_parquet(SELECTED_FRAC_FILE, index=False)

print("\n============================================================")
print("STEP 3.4 READY ✅")
print("============================================================")
print(f"rows               : {len(selected_fractional_crosswalk):,}")
print(f"unique catchments  : {selected_fractional_crosswalk['divide_id'].nunique():,}")
print(f"unique MRMS cells  : {selected_fractional_crosswalk['cell_id'].nunique():,}")
print(f"saved file         : {SELECTED_FRAC_FILE}")
print(f"total time         : {(time.time() - t0) / 60:.1f} minutes")

# Quick weight check
weight_check = selected_fractional_crosswalk.groupby("divide_id")["weight"].sum()

print("\nWeight sum check:")
print(weight_check.describe())

selected_fractional_crosswalk.head()

In [ ]:
# ============================================================
# Step 3.4 alternative — Use existing full fractional crosswalk
# ============================================================

import pandas as pd
from pathlib import Path

# Full fractional crosswalk file from previous full-CONUS run
FRAC_COMBINED = CACHE / "mrms_hf_fractional_crosswalk_conus.parquet"

assert FRAC_COMBINED.exists(), f"❌ Full fractional file not found: {FRAC_COMBINED}"
assert "selected_divide_ids" in globals(), (
    "❌ selected_divide_ids missing. Run Step 3.3 first."
)

# Load full fractional crosswalk
fractional_crosswalk = pd.read_parquet(FRAC_COMBINED)

print(f"Full fractional crosswalk rows: {len(fractional_crosswalk):,}")

# Keep only event-relevant catchments
selected_fractional_crosswalk = fractional_crosswalk[
    fractional_crosswalk["divide_id"].isin(selected_divide_ids)
].copy()

# Recompute normalized weights for selected catchments
selected_fractional_crosswalk["weight"] = selected_fractional_crosswalk[
    "fraction_inside"
] / selected_fractional_crosswalk.groupby("divide_id")["fraction_inside"].transform(
    "sum",
)

# Save reduced file
SELECTED_FRAC_FILE = (
    CACHE / "step3_event_relevant" / "selected_fractional_crosswalk.parquet"
)
SELECTED_FRAC_FILE.parent.mkdir(parents=True, exist_ok=True)

selected_fractional_crosswalk.to_parquet(SELECTED_FRAC_FILE, index=False)

print("\nSTEP 3.4 ALTERNATIVE READY ✅")
print(f"selected rows      : {len(selected_fractional_crosswalk):,}")
print(f"selected catchments: {selected_fractional_crosswalk['divide_id'].nunique():,}")
print(f"selected MRMS cells: {selected_fractional_crosswalk['cell_id'].nunique():,}")
print(f"saved file         : {SELECTED_FRAC_FILE}")

# Check weights
print("\nWeight sum check:")
print(selected_fractional_crosswalk.groupby("divide_id")["weight"].sum().describe())

## Step 3.5 — Build event-specific MRMS weighting table

This step connects each flash-flood event to the fractional MRMS cells inside its affected HydroFabric catchments. We already know which catchments intersect each storm footprint from `event_catchments`, and we already have area-weighted MRMS cell fractions from `selected_fractional_crosswalk`.

The output, `event_mrms_weights`, is the main lookup table for rainfall extraction. Each row tells us how much one MRMS cell contributes to one catchment for one event. Later, after MRMS precipitation is downloaded, we will merge rainfall values with this table and calculate area-weighted precipitation for each event, catchment, and time step.

In [ ]:
# ============================================================
# Step 3.5 — Build event-specific MRMS weighting table
#
# Inputs:
#   event_catchments
#   selected_fractional_crosswalk
#   events_raw
#
# Output:
#   event_mrms_weights
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 0. Checks
# ------------------------------------------------------------
assert "event_catchments" in globals(), "❌ event_catchments missing. Run Step 3.2."
assert "selected_fractional_crosswalk" in globals(), (
    "❌ selected_fractional_crosswalk missing."
)
assert "events_raw" in globals(), "❌ events_raw missing."

print("✅ Required objects found")


# ------------------------------------------------------------
# 1. Keep event-catchment relationship
# ------------------------------------------------------------
event_catchment_lookup = (
    event_catchments[["event_id", "divide_id"]].drop_duplicates().copy()
)

print(f"event-catchment pairs: {len(event_catchment_lookup):,}")


# ------------------------------------------------------------
# 2. Attach fractional MRMS weights
# ------------------------------------------------------------
event_mrms_weights = event_catchment_lookup.merge(
    selected_fractional_crosswalk[
        [
            "divide_id",
            "cell_id",
            "row",
            "col",
            "lon",
            "lat",
            "fraction_inside",
            "weight",
        ]
    ],
    on="divide_id",
    how="inner",
)

event_mrms_weights = event_mrms_weights.drop_duplicates(
    subset=["event_id", "divide_id", "cell_id"],
).reset_index(drop=True)

print(f"event_mrms_weights rows: {len(event_mrms_weights):,}")
print(f"unique events          : {event_mrms_weights['event_id'].nunique():,}")
print(f"unique catchments      : {event_mrms_weights['divide_id'].nunique():,}")
print(f"unique MRMS cells      : {event_mrms_weights['cell_id'].nunique():,}")


# ------------------------------------------------------------
# 3. Attach event time and gage metadata
# ------------------------------------------------------------
event_meta_cols = [
    "event_id",
    "cluster_id",
    "begin_time",
    "end_time",
    "duration_hours",
    "storm_lat",
    "storm_lon",
    "footprint_radius_km",
    "usgs_gage_id",
    "gage_lat",
    "gage_lon",
    "centroid_to_gage_dist_km",
]

event_meta_cols = [c for c in event_meta_cols if c in events_raw.columns]

event_meta = (
    events_raw[event_meta_cols]
    .drop_duplicates(subset=["event_id", "usgs_gage_id"])
    .copy()
)

event_mrms_weights = event_mrms_weights.merge(
    event_meta,
    on="event_id",
    how="left",
)


# ------------------------------------------------------------
# 4. Save output
# ------------------------------------------------------------
STEP3_CACHE = CACHE / "step3_event_relevant"
STEP3_CACHE.mkdir(exist_ok=True)

EVENT_MRMS_WEIGHTS_FILE = STEP3_CACHE / "event_mrms_weights.parquet"

event_mrms_weights.to_parquet(EVENT_MRMS_WEIGHTS_FILE, index=False)

print("\n============================================================")
print("STEP 3.5 READY ✅")
print("============================================================")
print(f"saved file: {EVENT_MRMS_WEIGHTS_FILE}")

event_mrms_weights.head()

## Step 3.6 — Build MRMS time manifest for selected events

This step creates the list of MRMS timestamps needed for each flash-flood event. For each event, the code uses the storm `begin_time` and `end_time`, optionally adds a time buffer before and after the storm, and rounds the time window to the chosen MRMS interval.

The output `event_time_manifest` contains one row per event-time combination. The output `unique_mrms_times` contains the unique timestamps that need to be downloaded only once. This avoids downloading the same MRMS file repeatedly for multiple events.

In [ ]:
# ============================================================
# Step 3.6 — Build MRMS time manifest
#
# Inputs:
#   events_raw
#   event_mrms_weights
#
# Outputs:
#   event_time_manifest
#   unique_mrms_times
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 0. Checks
# ------------------------------------------------------------
assert "events_raw" in globals(), "❌ events_raw missing."
assert "event_mrms_weights" in globals(), (
    "❌ event_mrms_weights missing. Run Step 3.5 first."
)
assert "CACHE" in globals(), "❌ CACHE missing."

print("✅ Required objects found")


# ------------------------------------------------------------
# 1. User settings
# ------------------------------------------------------------
# Change this depending on your MRMS product.
# Examples: "2min", "5min", "15min", "1H"
MRMS_TIME_FREQ = "2min"

# Add extra time around each storm event
PRE_EVENT_HOURS = 168
POST_EVENT_HOURS = 24


# ------------------------------------------------------------
# 2. Create unique event time table
# ------------------------------------------------------------
event_time_table = (
    events_raw[
        [
            "event_id",
            "cluster_id",
            "begin_time",
            "end_time",
            "duration_hours",
            "storm_lat",
            "storm_lon",
            "footprint_radius_km",
        ]
    ]
    .drop_duplicates(subset=["event_id"])
    .copy()
)

event_time_table["begin_time"] = pd.to_datetime(
    event_time_table["begin_time"],
    errors="coerce",
    utc=True,
)

event_time_table["end_time"] = pd.to_datetime(
    event_time_table["end_time"],
    errors="coerce",
    utc=True,
)

event_time_table = event_time_table.dropna(
    subset=["event_id", "begin_time", "end_time"],
).copy()

print(f"Unique events with valid times: {len(event_time_table):,}")


# ------------------------------------------------------------
# 3. Add pre/post event buffer and round to MRMS interval
# ------------------------------------------------------------
event_time_table["mrms_start"] = event_time_table["begin_time"] - pd.to_timedelta(
    PRE_EVENT_HOURS,
    unit="h",
)

event_time_table["mrms_end"] = event_time_table["end_time"] + pd.to_timedelta(
    POST_EVENT_HOURS,
    unit="h",
)

event_time_table["mrms_start"] = event_time_table["mrms_start"].dt.floor(MRMS_TIME_FREQ)
event_time_table["mrms_end"] = event_time_table["mrms_end"].dt.ceil(MRMS_TIME_FREQ)


# ------------------------------------------------------------
# 4. Expand each event into all MRMS timestamps
# ------------------------------------------------------------
rows = []

for_i = 0

for _, r in event_time_table.iterrows():
    times = pd.date_range(
        start=r["mrms_start"],
        end=r["mrms_end"],
        freq=MRMS_TIME_FREQ,
    )

    tmp = pd.DataFrame(
        {
            "event_id": r["event_id"],
            "cluster_id": r["cluster_id"],
            "begin_time": r["begin_time"],
            "end_time": r["end_time"],
            "mrms_start": r["mrms_start"],
            "mrms_end": r["mrms_end"],
            "mrms_time": times,
        },
    )

    rows.append(tmp)

event_time_manifest = pd.concat(rows, ignore_index=True)

print(f"event_time_manifest rows: {len(event_time_manifest):,}")
print(f"unique events: {event_time_manifest['event_id'].nunique():,}")


# ------------------------------------------------------------
# 5. Create unique MRMS download times
# ------------------------------------------------------------
unique_mrms_times = (
    event_time_manifest[["mrms_time"]]
    .drop_duplicates()
    .sort_values("mrms_time")
    .reset_index(drop=True)
)

unique_mrms_times["year"] = unique_mrms_times["mrms_time"].dt.year
unique_mrms_times["month"] = unique_mrms_times["mrms_time"].dt.month
unique_mrms_times["day"] = unique_mrms_times["mrms_time"].dt.day
unique_mrms_times["hour"] = unique_mrms_times["mrms_time"].dt.hour
unique_mrms_times["minute"] = unique_mrms_times["mrms_time"].dt.minute

print(f"unique MRMS timestamps to download: {len(unique_mrms_times):,}")
print(
    f"time range: {unique_mrms_times['mrms_time'].min()} to {unique_mrms_times['mrms_time'].max()}",
)


# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------
STEP3_CACHE = CACHE / "step3_event_relevant"
STEP3_CACHE.mkdir(exist_ok=True)

EVENT_TIME_MANIFEST_FILE = STEP3_CACHE / "event_time_manifest.parquet"
UNIQUE_MRMS_TIMES_FILE = STEP3_CACHE / "unique_mrms_times.parquet"

event_time_manifest.to_parquet(EVENT_TIME_MANIFEST_FILE, index=False)
unique_mrms_times.to_parquet(UNIQUE_MRMS_TIMES_FILE, index=False)

print("\n============================================================")
print("STEP 3.6 READY ✅")
print("============================================================")
print(f"Saved event_time_manifest: {EVENT_TIME_MANIFEST_FILE}")
print(f"Saved unique_mrms_times  : {UNIQUE_MRMS_TIMES_FILE}")

unique_mrms_times.head()

# Check Readiness for downloading



In [ ]:
print("Unique MRMS times:", len(unique_mrms_times))
print(
    "Time range:",
    unique_mrms_times["mrms_time"].min(),
    "to",
    unique_mrms_times["mrms_time"].max(),
)
print("Unique events:", event_time_manifest["event_id"].nunique())
print("Unique MRMS cells needed:", event_mrms_weights["cell_id"].nunique())

In [ ]:
# Pick one short event for testing
test_event_id = event_time_table.sort_values("duration_hours").iloc[0]["event_id"]

test_event_times = event_time_manifest[
    event_time_manifest["event_id"] == test_event_id
].copy()

test_event_weights = event_mrms_weights[
    event_mrms_weights["event_id"] == test_event_id
].copy()

print("Test event:", test_event_id)
print("MRMS timestamps for this event:", len(test_event_times))
print("MRMS cells for this event:", test_event_weights["cell_id"].nunique())
print(
    "Time range:",
    test_event_times["mrms_time"].min(),
    "to",
    test_event_times["mrms_time"].max(),
)

## Step 3.7 — Test-download MRMS files for one event

This step downloads MRMS 2-minute `PrecipRate` files for one selected event only. Because the time window includes 7 days before the event and 1 day after the event, this test event may still require several thousand MRMS files.

The code builds the MRMS archive URL for each timestamp, checks whether the file already exists locally, downloads missing files, and saves them in an event-specific folder. This is still a test run before scaling to all events.

In [ ]:
# ============================================================
# Step 3.7 — Download MRMS PrecipRate files for one test event
#
# Inputs:
#   test_event_id
#   test_event_times
#
# Output:
#   downloaded .grib2.gz MRMS files for one event
# ============================================================

import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Checks
# ------------------------------------------------------------
assert "test_event_id" in globals(), "❌ test_event_id missing."
assert "test_event_times" in globals(), "❌ test_event_times missing."

# ------------------------------------------------------------
# 1. Settings
# ------------------------------------------------------------
MRMS_PRODUCT = "PrecipRate"
MRMS_FIELD = "PrecipRate_00.00"

# Historical archive base
MRMS_ARCHIVE_BASE = "https://mtarchive.geol.iastate.edu"

# Download folder for this event
MRMS_DOWNLOAD_DIR = CACHE / "mrms_downloads" / f"event_{test_event_id}" / MRMS_PRODUCT
MRMS_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Number of parallel download workers
N_WORKERS = 8

print(f"Downloading MRMS files for event: {test_event_id}")
print(f"Number of timestamps: {len(test_event_times):,}")
print(f"Output folder: {MRMS_DOWNLOAD_DIR}")


# ------------------------------------------------------------
# 2. Build archive URL and local filename
# ------------------------------------------------------------
def mrms_archive_url_and_path(t):
    """
    Build Iowa State MTArchive URL for MRMS PrecipRate.

    Example archive structure:
    /YYYY/MM/DD/mrms/ncep/PrecipRate/PrecipRate_00.00_YYYYMMDD-HHMMSS.grib2.gz
    """
    t = pd.Timestamp(t).tz_convert("UTC")

    yyyy = t.strftime("%Y")
    mm = t.strftime("%m")
    dd = t.strftime("%d")
    stamp = t.strftime("%Y%m%d-%H%M%S")

    fname = f"{MRMS_FIELD}_{stamp}.grib2.gz"

    url = f"{MRMS_ARCHIVE_BASE}/{yyyy}/{mm}/{dd}/mrms/ncep/{MRMS_PRODUCT}/{fname}"

    local_path = MRMS_DOWNLOAD_DIR / fname

    return url, local_path


# ------------------------------------------------------------
# 3. Download one file
# ------------------------------------------------------------
def download_one(t, timeout=30):
    url, local_path = mrms_archive_url_and_path(t)

    if local_path.exists() and local_path.stat().st_size > 0:
        return {
            "mrms_time": t,
            "status": "exists",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": local_path.stat().st_size,
        }

    try:
        r = requests.get(url, timeout=timeout, stream=True)

        if r.status_code != 200:
            return {
                "mrms_time": t,
                "status": f"missing_http_{r.status_code}",
                "url": url,
                "local_path": str(local_path),
                "size_bytes": 0,
            }

        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)

        return {
            "mrms_time": t,
            "status": "downloaded",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": local_path.stat().st_size,
        }

    except Exception as e:
        return {
            "mrms_time": t,
            "status": f"error_{type(e).__name__}",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": 0,
        }


# ------------------------------------------------------------
# 4. Run downloads
# ------------------------------------------------------------
times_to_download = (
    test_event_times["mrms_time"].drop_duplicates().sort_values().tolist()
)

download_results = []

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(download_one, t): t for t in times_to_download}

    for fut in tqdm(as_completed(futures), total=len(futures)):
        download_results.append(fut.result())


# ------------------------------------------------------------
# 5. Save download manifest
# ------------------------------------------------------------
test_event_download_manifest = pd.DataFrame(download_results)

TEST_EVENT_DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{test_event_id}" / "download_manifest.parquet"
)

test_event_download_manifest.to_parquet(
    TEST_EVENT_DOWNLOAD_MANIFEST_FILE,
    index=False,
)

print("\n============================================================")
print("STEP 3.7 DOWNLOAD TEST DONE ✅")
print("============================================================")
print(test_event_download_manifest["status"].value_counts())
print(f"Saved manifest: {TEST_EVENT_DOWNLOAD_MANIFEST_FILE}")

test_event_download_manifest.head()

In [ ]:
# ============================================================
# Step 3.7 — Download MRMS PrecipRate files for one test event
#
# Inputs:
#   test_event_id
#   test_event_times
#
# Output:
#   downloaded .grib2.gz MRMS files for one event
#   download manifest with status and timing summary
# ============================================================

import time
import requests
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Checks
# ------------------------------------------------------------
assert "test_event_id" in globals(), "❌ test_event_id missing."
assert "test_event_times" in globals(), "❌ test_event_times missing."
assert "CACHE" in globals(), "❌ CACHE missing."

# ------------------------------------------------------------
# 1. Settings
# ------------------------------------------------------------
MRMS_PRODUCT = "PrecipRate"
MRMS_FIELD = "PrecipRate_00.00"

# Historical archive base
MRMS_ARCHIVE_BASE = "https://mtarchive.geol.iastate.edu"

# Download folder for this event
MRMS_DOWNLOAD_DIR = CACHE / "mrms_downloads" / f"event_{test_event_id}" / MRMS_PRODUCT
MRMS_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Number of parallel download workers
N_WORKERS = 8

print(f"Downloading MRMS files for event: {test_event_id}")
print(f"Number of timestamps: {len(test_event_times):,}")
print(f"Output folder: {MRMS_DOWNLOAD_DIR}")


# ------------------------------------------------------------
# 2. Build archive URL and local filename
# ------------------------------------------------------------
def mrms_archive_url_and_path(t):
    """
    Build Iowa State MTArchive URL for MRMS PrecipRate.

    Example archive structure:
    /YYYY/MM/DD/mrms/ncep/PrecipRate/PrecipRate_00.00_YYYYMMDD-HHMMSS.grib2.gz
    """
    t = pd.Timestamp(t).tz_convert("UTC")

    yyyy = t.strftime("%Y")
    mm = t.strftime("%m")
    dd = t.strftime("%d")
    stamp = t.strftime("%Y%m%d-%H%M%S")

    fname = f"{MRMS_FIELD}_{stamp}.grib2.gz"

    url = f"{MRMS_ARCHIVE_BASE}/{yyyy}/{mm}/{dd}/mrms/ncep/{MRMS_PRODUCT}/{fname}"

    local_path = MRMS_DOWNLOAD_DIR / fname

    return url, local_path


# ------------------------------------------------------------
# 3. Download one file
# ------------------------------------------------------------
def download_one(t, timeout=30):
    url, local_path = mrms_archive_url_and_path(t)

    if local_path.exists() and local_path.stat().st_size > 0:
        return {
            "mrms_time": t,
            "status": "exists",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": local_path.stat().st_size,
        }

    try:
        r = requests.get(url, timeout=timeout, stream=True)

        if r.status_code != 200:
            return {
                "mrms_time": t,
                "status": f"missing_http_{r.status_code}",
                "url": url,
                "local_path": str(local_path),
                "size_bytes": 0,
            }

        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)

        return {
            "mrms_time": t,
            "status": "downloaded",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": local_path.stat().st_size,
        }

    except Exception as e:
        return {
            "mrms_time": t,
            "status": f"error_{type(e).__name__}",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": 0,
        }


# ------------------------------------------------------------
# 4. Run downloads and measure time
# ------------------------------------------------------------
times_to_download = (
    test_event_times["mrms_time"].drop_duplicates().sort_values().tolist()
)

download_results = []

download_start_time = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(download_one, t): t for t in times_to_download}

    for fut in tqdm(as_completed(futures), total=len(futures)):
        download_results.append(fut.result())

download_end_time = time.time()

download_elapsed_seconds = download_end_time - download_start_time
download_elapsed_minutes = download_elapsed_seconds / 60


# ------------------------------------------------------------
# 5. Save download manifest
# ------------------------------------------------------------
test_event_download_manifest = pd.DataFrame(download_results)

TEST_EVENT_DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{test_event_id}" / "download_manifest.parquet"
)

test_event_download_manifest.to_parquet(
    TEST_EVENT_DOWNLOAD_MANIFEST_FILE,
    index=False,
)


# ------------------------------------------------------------
# 6. Print summary
# ------------------------------------------------------------
print("\n============================================================")
print("STEP 3.7 DOWNLOAD TEST DONE ✅")
print("============================================================")

print("\nDownload status:")
print(test_event_download_manifest["status"].value_counts())

print(f"\nSaved manifest: {TEST_EVENT_DOWNLOAD_MANIFEST_FILE}")

print("\nDownload timing:")
print(f"Total time: {download_elapsed_minutes:.2f} minutes")
print(f"Total time: {download_elapsed_seconds:.1f} seconds")
print(
    f"Average time per timestamp: {download_elapsed_seconds / len(times_to_download):.3f} seconds",
)

print("\nDownloaded file size summary:")
print(test_event_download_manifest["size_bytes"].describe())

test_event_download_manifest.head()

In [ ]:
# ============================================================
# Step 3.8 — Download largest event and check time
# ============================================================

import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Find largest event by number of MRMS timestamps
# ------------------------------------------------------------
largest_event_summary = (
    event_time_manifest.groupby("event_id")
    .agg(
        n_mrms_times=("mrms_time", "count"),
        start_time=("mrms_time", "min"),
        end_time=("mrms_time", "max"),
    )
    .reset_index()
    .sort_values("n_mrms_times", ascending=False)
)

largest_event_id = largest_event_summary.iloc[0]["event_id"]

largest_event_times = event_time_manifest[
    event_time_manifest["event_id"] == largest_event_id
].copy()

print("Largest event selected:")
print(largest_event_summary.head(10))

print("\nUsing largest event:")
print("event_id:", largest_event_id)
print("MRMS timestamps:", len(largest_event_times))
print(
    "Time range:",
    largest_event_times["mrms_time"].min(),
    "to",
    largest_event_times["mrms_time"].max(),
)


# ------------------------------------------------------------
# 1. Settings
# ------------------------------------------------------------
MRMS_PRODUCT = "PrecipRate"
MRMS_FIELD = "PrecipRate_00.00"
MRMS_ARCHIVE_BASE = "https://mtarchive.geol.iastate.edu"

MRMS_DOWNLOAD_DIR = (
    CACHE / "mrms_downloads" / f"event_{largest_event_id}" / MRMS_PRODUCT
)
MRMS_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

N_WORKERS = 8

print(f"\nDownloading MRMS files for largest event: {largest_event_id}")
print(f"Output folder: {MRMS_DOWNLOAD_DIR}")


# ------------------------------------------------------------
# 2. Build URL and local path
# ------------------------------------------------------------
def mrms_archive_url_and_path(t):
    t = pd.Timestamp(t).tz_convert("UTC")

    yyyy = t.strftime("%Y")
    mm = t.strftime("%m")
    dd = t.strftime("%d")
    stamp = t.strftime("%Y%m%d-%H%M%S")

    fname = f"{MRMS_FIELD}_{stamp}.grib2.gz"

    url = f"{MRMS_ARCHIVE_BASE}/{yyyy}/{mm}/{dd}/mrms/ncep/{MRMS_PRODUCT}/{fname}"

    local_path = MRMS_DOWNLOAD_DIR / fname

    return url, local_path


# ------------------------------------------------------------
# 3. Download one file
# ------------------------------------------------------------
def download_one(t, timeout=30):
    url, local_path = mrms_archive_url_and_path(t)

    if local_path.exists() and local_path.stat().st_size > 0:
        return {
            "mrms_time": t,
            "status": "exists",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": local_path.stat().st_size,
        }

    try:
        r = requests.get(url, timeout=timeout, stream=True)

        if r.status_code != 200:
            return {
                "mrms_time": t,
                "status": f"missing_http_{r.status_code}",
                "url": url,
                "local_path": str(local_path),
                "size_bytes": 0,
            }

        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)

        return {
            "mrms_time": t,
            "status": "downloaded",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": local_path.stat().st_size,
        }

    except Exception as e:
        return {
            "mrms_time": t,
            "status": f"error_{type(e).__name__}",
            "url": url,
            "local_path": str(local_path),
            "size_bytes": 0,
        }


# ------------------------------------------------------------
# 4. Run download and time it
# ------------------------------------------------------------
times_to_download = (
    largest_event_times["mrms_time"].drop_duplicates().sort_values().tolist()
)

download_results = []

download_start_time = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(download_one, t): t for t in times_to_download}

    for fut in tqdm(as_completed(futures), total=len(futures)):
        download_results.append(fut.result())

download_end_time = time.time()

download_elapsed_seconds = download_end_time - download_start_time
download_elapsed_minutes = download_elapsed_seconds / 60


# ------------------------------------------------------------
# 5. Save manifest
# ------------------------------------------------------------
largest_event_download_manifest = pd.DataFrame(download_results)

LARGEST_EVENT_DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{largest_event_id}" / "download_manifest.parquet"
)

largest_event_download_manifest.to_parquet(
    LARGEST_EVENT_DOWNLOAD_MANIFEST_FILE,
    index=False,
)


# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------
print("\n============================================================")
print("LARGEST EVENT DOWNLOAD DONE ✅")
print("============================================================")

print("\nDownload status:")
print(largest_event_download_manifest["status"].value_counts())

print(f"\nSaved manifest: {LARGEST_EVENT_DOWNLOAD_MANIFEST_FILE}")

print("\nDownload timing:")
print(f"Total time: {download_elapsed_minutes:.2f} minutes")
print(f"Total time: {download_elapsed_seconds:.1f} seconds")
print(
    f"Average time per timestamp: {download_elapsed_seconds / len(times_to_download):.3f} seconds",
)

print("\nDownloaded file size summary:")
print(largest_event_download_manifest["size_bytes"].describe())

largest_event_download_manifest.head()

In [ ]:
print("test_event_id:", test_event_id)

events_raw[events_raw["event_id"] == test_event_id][
    ["event_id", "cluster_id", "begin_time", "end_time", "usgs_gage_id"]
].drop_duplicates()

In [ ]:
print("unique storm_index:", events_raw["storm_index"].nunique())
print("unique cluster_id:", events_raw["cluster_id"].nunique())
print("rows:", len(events_raw))

In [ ]:
events_raw["event_id"] = events_raw["cluster_id"]

In [ ]:
import sys

!{sys.executable} -m pip install cfgrib eccodes

In [ ]:
import sys

!{sys.executable} -m pip install -U cfgrib eccodes ecmwflibs

In [ ]:
import xarray as xr
import cfgrib
import eccodes

print(xr.backends.list_engines())

In [ ]:
import xarray as xr
import cfgrib
import eccodes

print("cfgrib works")
print(xr.backends.list_engines())

## Inspect one MRMS `.grib2.gz` file

A `.grib2.gz` file is a compressed GRIB2 file. The `.gz` part only means the file is compressed. After decompression, the actual data are in GRIB2 format, which stores gridded meteorological data such as MRMS precipitation rate.

This cell selects one downloaded MRMS file, decompresses it temporarily, opens it with `cfgrib`, and prints its structure: dimensions, coordinates, data variables, attributes, grid shape, and value range.

In [ ]:
from pathlib import Path
import pandas as pd

# Re-define cache folder after kernel restart
CACHE = Path("mrms_crosswalk_cache")

assert CACHE.exists(), f"Cache folder not found: {CACHE}"

# Optional: load one download manifest if needed
manifest_files = sorted(
    (CACHE / "mrms_downloads").glob("event_*/download_manifest.parquet"),
)

print("Found manifests:", len(manifest_files))
print("First manifest:", manifest_files[0])

download_manifest = pd.read_parquet(manifest_files[0])

print(download_manifest["status"].value_counts())
download_manifest.head()

In [ ]:
# ============================================================
# Inspect structure of one downloaded MRMS .grib2.gz file
# ============================================================

import gzip
import tempfile
from pathlib import Path
import xarray as xr
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Select one local MRMS file
# ------------------------------------------------------------
# Prefer using the download manifest if available
if "download_manifest" in globals():
    manifest_to_use = download_manifest.copy()
elif "test_event_download_manifest" in globals():
    manifest_to_use = test_event_download_manifest.copy()
elif "largest_event_download_manifest" in globals():
    manifest_to_use = largest_event_download_manifest.copy()
else:
    manifest_to_use = None

if manifest_to_use is not None:
    one_file = (
        manifest_to_use[
            manifest_to_use["status"].isin(["downloaded", "exists"])
            & (manifest_to_use["size_bytes"] > 0)
        ]
        .sort_values("mrms_time")
        .iloc[0]["local_path"]
    )
    one_file = Path(one_file)
else:
    # Fallback: search inside CACHE
    files = sorted((CACHE / "mrms_downloads").glob("**/*.grib2.gz"))
    assert len(files) > 0, "❌ No .grib2.gz files found."
    one_file = files[0]

print("Selected file:")
print(one_file)
print(f"Compressed file size: {one_file.stat().st_size / 1e6:.2f} MB")


# ------------------------------------------------------------
# 2. Decompress and open with cfgrib
# ------------------------------------------------------------
with gzip.open(one_file, "rb") as gz:
    raw_grib = gz.read()

print(f"Decompressed GRIB2 size: {len(raw_grib) / 1e6:.2f} MB")

with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
    tmp.write(raw_grib)
    tmp.flush()

    ds = xr.load_dataset(
        tmp.name,
        engine="cfgrib",
        backend_kwargs={"indexpath": ""},
        decode_timedelta=False,
    )

print("\n============================================================")
print("XARRAY DATASET STRUCTURE")
print("============================================================")
print(ds)


# ------------------------------------------------------------
# 3. Print dimensions
# ------------------------------------------------------------
print("\n============================================================")
print("DIMENSIONS")
print("============================================================")
for dim, size in ds.sizes.items():
    print(f"{dim}: {size}")


# ------------------------------------------------------------
# 4. Print coordinates
# ------------------------------------------------------------
print("\n============================================================")
print("COORDINATES")
print("============================================================")
for coord in ds.coords:
    c = ds[coord]
    print(f"{coord}: shape={c.shape}, dtype={c.dtype}")

    if coord in ["latitude", "longitude"]:
        print(f"  min={float(c.min()):.4f}, max={float(c.max()):.4f}")


# ------------------------------------------------------------
# 5. Print data variables
# ------------------------------------------------------------
print("\n============================================================")
print("DATA VARIABLES")
print("============================================================")
for var in ds.data_vars:
    da = ds[var]

    print(f"\nVariable: {var}")
    print(f"  shape: {da.shape}")
    print(f"  dims : {da.dims}")
    print(f"  dtype: {da.dtype}")

    values = da.values.astype(float)
    values[values < 0] = np.nan

    print(f"  valid min : {np.nanmin(values):.4f}")
    print(f"  valid max : {np.nanmax(values):.4f}")
    print(f"  valid mean: {np.nanmean(values):.4f}")
    print(f"  NaN count : {np.isnan(values).sum():,}")

    print("  attributes:")
    for k, v in da.attrs.items():
        print(f"    {k}: {v}")


# ------------------------------------------------------------
# 6. Print global attributes
# ------------------------------------------------------------
print("\n============================================================")
print("GLOBAL ATTRIBUTES")
print("============================================================")
for k, v in ds.attrs.items():
    print(f"{k}: {v}")


# ------------------------------------------------------------
# 7. Show first few latitude/longitude values
# ------------------------------------------------------------
print("\n============================================================")
print("GRID SAMPLE")
print("============================================================")

print("First 5 latitudes:")
print(ds["latitude"].values[:5])

print("\nFirst 5 longitudes:")
print(ds["longitude"].values[:5])

print("\nLast 5 latitudes:")
print(ds["latitude"].values[-5:])

print("\nLast 5 longitudes:")
print(ds["longitude"].values[-5:])

## Step 3.9 — Convert 2-minute MRMS PrecipRate files to 15-minute rainfall time series

This step reads the downloaded local `.grib2.gz` MRMS files directly, so we do not need to convert the raw GRIB files to CSV first.

For each MRMS file, the code extracts precipitation rate values only at the MRMS cells needed for the selected event. Then it applies the fractional catchment weights from `event_mrms_weights` to compute area-weighted precipitation rate for each catchment.

Because `PrecipRate` is a rainfall rate in mm/hr, the 15-minute rainfall depth is calculated as:

`15-minute accumulation = mean precipitation rate over the 15-minute window × 0.25 hours`

The outputs are catchment-level 2-minute precipitation rate, catchment-level 15-minute rainfall, and an event-average 15-minute rainfall time series plot.

In [ ]:
# ============================================================
# Reload needed variables after kernel restart
# and choose one downloaded event to process
# ============================================================

from pathlib import Path
import pandas as pd

CACHE = Path("mrms_crosswalk_cache")

# Load event_mrms_weights if not already in memory
EVENT_MRMS_WEIGHTS_FILE = CACHE / "step3_event_relevant" / "event_mrms_weights.parquet"

if "event_mrms_weights" not in globals():
    event_mrms_weights = pd.read_parquet(EVENT_MRMS_WEIGHTS_FILE)
    print("Loaded event_mrms_weights")

# Find downloaded event manifests
manifest_files = sorted(
    (CACHE / "mrms_downloads").glob("event_*/download_manifest.parquet"),
)

assert len(manifest_files) > 0, "No downloaded event manifests found."

summary = []

for mf in manifest_files:
    event_id = mf.parent.name.replace("event_", "")
    dm = pd.read_parquet(mf)

    good = dm[dm["status"].isin(["downloaded", "exists"]) & (dm["size_bytes"] > 0)]

    summary.append(
        {
            "event_id": event_id,
            "manifest_file": mf,
            "total_rows": len(dm),
            "good_files": len(good),
            "missing_files": (dm["status"].astype(str).str.contains("missing")).sum(),
        },
    )

downloaded_event_summary = pd.DataFrame(summary).sort_values(
    "good_files",
    ascending=False,
)

display(downloaded_event_summary)

# Choose the downloaded event with the most good files
EVENT_TO_PROCESS = downloaded_event_summary.iloc[0]["event_id"]

print("Selected EVENT_TO_PROCESS:", EVENT_TO_PROCESS)

In [ ]:
# ============================================================
# Simple 2-min MRMS time series for ONE point
# Example: nearest MRMS grid cell to the event gage
# ============================================================

import gzip
import tempfile
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Required objects
# ------------------------------------------------------------
assert "EVENT_TO_PROCESS" in globals(), "❌ EVENT_TO_PROCESS missing."
assert "CACHE" in globals(), "❌ CACHE missing."

print("Processing event:", EVENT_TO_PROCESS)

# ------------------------------------------------------------
# 1. Load download manifest
# ------------------------------------------------------------
DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{EVENT_TO_PROCESS}" / "download_manifest.parquet"
)

download_manifest = pd.read_parquet(DOWNLOAD_MANIFEST_FILE)

download_manifest["mrms_time"] = pd.to_datetime(
    download_manifest["mrms_time"],
    utc=True,
)

good_files = download_manifest[
    download_manifest["status"].isin(["downloaded", "exists"])
    & (download_manifest["size_bytes"] > 0)
].copy()

good_files = good_files.sort_values("mrms_time").reset_index(drop=True)

print(f"Good files: {len(good_files):,}")
print(f"Time range: {good_files['mrms_time'].min()} to {good_files['mrms_time'].max()}")


# ------------------------------------------------------------
# 2. Choose ONE point
# ------------------------------------------------------------
# Option A: manually set a point
# POINT_LAT = 30.1234
# POINT_LON = -97.1234

# Option B: use the gage point from event_mrms_weights
if "POINT_LAT" not in globals() or "POINT_LON" not in globals():
    assert "event_mrms_weights" in globals(), "❌ event_mrms_weights missing."

    tmp_event = event_mrms_weights[
        event_mrms_weights["event_id"].astype(str) == str(EVENT_TO_PROCESS)
    ].copy()

    assert len(tmp_event) > 0, "❌ No matching event found in event_mrms_weights."

    if {"gage_lat", "gage_lon"}.issubset(tmp_event.columns):
        POINT_LAT = float(tmp_event["gage_lat"].dropna().iloc[0])
        POINT_LON = float(tmp_event["gage_lon"].dropna().iloc[0])
        POINT_NAME = "USGS gage"
    else:
        # fallback: use first MRMS cell already connected to this event
        POINT_ROW = int(tmp_event["row"].iloc[0])
        POINT_COL = int(tmp_event["col"].iloc[0])
        POINT_LAT = 55.0 - 0.005 - POINT_ROW * 0.01
        POINT_LON = -130.0 + 0.005 + POINT_COL * 0.01
        POINT_NAME = "first event MRMS cell"

print(f"Point used: {POINT_NAME}")
print(f"POINT_LAT: {POINT_LAT}")
print(f"POINT_LON: {POINT_LON}")


# ------------------------------------------------------------
# 3. Convert lat/lon to nearest MRMS row/col
# ------------------------------------------------------------
MRMS_LON_MIN_EDGE = -130.0
MRMS_LAT_MAX_EDGE = 55.0
MRMS_RES = 0.01
HALF = MRMS_RES / 2.0

point_col = int(round((POINT_LON - (MRMS_LON_MIN_EDGE + HALF)) / MRMS_RES))
point_row = int(round(((MRMS_LAT_MAX_EDGE - HALF) - POINT_LAT) / MRMS_RES))

print(f"Nearest MRMS row: {point_row}")
print(f"Nearest MRMS col: {point_col}")

nearest_lat = MRMS_LAT_MAX_EDGE - HALF - point_row * MRMS_RES
nearest_lon = MRMS_LON_MIN_EDGE + HALF + point_col * MRMS_RES

print(f"Nearest MRMS cell center lat/lon: {nearest_lat}, {nearest_lon}")


# ------------------------------------------------------------
# 4. Helper: read one GRIB.GZ file
# ------------------------------------------------------------
def read_one_point_from_grib_gz(gz_path, row, col):
    gz_path = Path(gz_path)

    with gzip.open(gz_path, "rb") as gz:
        raw = gz.read()

    with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
        tmp.write(raw)
        tmp.flush()

        ds = xr.load_dataset(
            tmp.name,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""},
            decode_timedelta=False,
        )

    varname = list(ds.data_vars)[0]

    value = float(ds[varname].values[row, col])

    # Negative MRMS values are no-data
    if value < 0:
        value = np.nan

    return value


# ------------------------------------------------------------
# 5. Extract 2-min point time series
# ------------------------------------------------------------
records = []
failed = []

t0 = time.time()

for _, r in tqdm(good_files.iterrows(), total=len(good_files)):
    try:
        precip_rate = read_one_point_from_grib_gz(
            r["local_path"],
            point_row,
            point_col,
        )

        records.append(
            {
                "event_id": EVENT_TO_PROCESS,
                "mrms_time": r["mrms_time"],
                "point_lat": POINT_LAT,
                "point_lon": POINT_LON,
                "mrms_row": point_row,
                "mrms_col": point_col,
                "mrms_cell_center_lat": nearest_lat,
                "mrms_cell_center_lon": nearest_lon,
                "precip_rate_mm_hr": precip_rate,
                "precip_mm_2min": precip_rate * (2 / 60),
            },
        )

    except Exception as e:
        failed.append(
            {
                "mrms_time": r["mrms_time"],
                "local_path": r["local_path"],
                "error": str(e),
            },
        )

elapsed_min = (time.time() - t0) / 60

point_2min_timeseries = pd.DataFrame(records).sort_values("mrms_time")

print("\n============================================================")
print("ONE-POINT 2-MIN TIME SERIES DONE ✅")
print("============================================================")
print(f"Successful files: {len(point_2min_timeseries):,}")
print(f"Failed files    : {len(failed):,}")
print(f"Elapsed time    : {elapsed_min:.2f} minutes")


# ------------------------------------------------------------
# 6. Save CSV
# ------------------------------------------------------------
PROCESSED_DIR = CACHE / "mrms_processed" / f"event_{EVENT_TO_PROCESS}"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

POINT_2MIN_CSV = PROCESSED_DIR / "point_2min_timeseries.csv"
point_2min_timeseries.to_csv(POINT_2MIN_CSV, index=False)

print(f"Saved: {POINT_2MIN_CSV}")


# ------------------------------------------------------------
# 7. Plot point time series
# ------------------------------------------------------------
plt.figure(figsize=(13, 4))

plt.plot(
    point_2min_timeseries["mrms_time"],
    point_2min_timeseries["precip_rate_mm_hr"],
    linewidth=1.2,
)

plt.xlabel("Time")
plt.ylabel("Precipitation rate (mm/hr)")
plt.title(f"Event {EVENT_TO_PROCESS} — MRMS 2-min precipitation at one point")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

point_2min_timeseries.head()

In [ ]:
# ============================================================
# Simple 2-min MRMS precipitation time series for one event
# ============================================================

import gzip
import tempfile
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Checks
# ------------------------------------------------------------
assert "EVENT_TO_PROCESS" in globals(), "❌ EVENT_TO_PROCESS missing."
assert "event_mrms_weights" in globals(), "❌ event_mrms_weights missing."
assert "CACHE" in globals(), "❌ CACHE missing."

print("Processing event:", EVENT_TO_PROCESS)

# ------------------------------------------------------------
# 1. Load download manifest
# ------------------------------------------------------------
DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{EVENT_TO_PROCESS}" / "download_manifest.parquet"
)

download_manifest = pd.read_parquet(DOWNLOAD_MANIFEST_FILE)

download_manifest["mrms_time"] = pd.to_datetime(
    download_manifest["mrms_time"],
    utc=True,
)

good_files = download_manifest[
    download_manifest["status"].isin(["downloaded", "exists"])
    & (download_manifest["size_bytes"] > 0)
].copy()

good_files = good_files.sort_values("mrms_time").reset_index(drop=True)

print(f"Good GRIB files: {len(good_files):,}")
print(f"Time range: {good_files['mrms_time'].min()} to {good_files['mrms_time'].max()}")

# ------------------------------------------------------------
# 2. Get event-specific MRMS weights
# ------------------------------------------------------------
event_mrms_weights["_event_id_str"] = event_mrms_weights["event_id"].astype(str)

weights = (
    event_mrms_weights[event_mrms_weights["_event_id_str"] == str(EVENT_TO_PROCESS)][
        ["divide_id", "cell_id", "row", "col", "weight"]
    ]
    .drop_duplicates(subset=["divide_id", "cell_id"])
    .copy()
)

assert len(weights) > 0, f"❌ No weights found for event {EVENT_TO_PROCESS}"

weights["row"] = weights["row"].astype(int)
weights["col"] = weights["col"].astype(int)
weights["weight"] = weights["weight"].astype(float)

rows_idx = weights["row"].values
cols_idx = weights["col"].values

print(f"Catchments: {weights['divide_id'].nunique():,}")
print(f"MRMS cells: {weights['cell_id'].nunique():,}")


# ------------------------------------------------------------
# 3. Helper: read one GRIB.GZ file
# ------------------------------------------------------------
def read_mrms_grib_gz(gz_path):
    gz_path = Path(gz_path)

    with gzip.open(gz_path, "rb") as gz:
        raw = gz.read()

    with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
        tmp.write(raw)
        tmp.flush()

        ds = xr.load_dataset(
            tmp.name,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""},
            decode_timedelta=False,
        )

    varname = list(ds.data_vars)[0]
    arr = ds[varname].values.astype(float)

    # Negative values are no-data
    arr[arr < 0] = np.nan

    return arr


# ------------------------------------------------------------
# 4. Read all files and create 2-min event time series
# ------------------------------------------------------------
records = []
failed = []

t0 = time.time()

for _, r in tqdm(good_files.iterrows(), total=len(good_files)):
    mrms_time = r["mrms_time"]
    local_path = r["local_path"]

    try:
        arr = read_mrms_grib_gz(local_path)

        vals = arr[rows_idx, cols_idx]

        tmp = weights.copy()
        tmp["precip_rate_mm_hr"] = vals
        tmp["weighted_rate"] = tmp["precip_rate_mm_hr"] * tmp["weight"]

        # Catchment-average precipitation rate
        catchment_rates = tmp.groupby("divide_id", as_index=False).agg(
            precip_rate_mm_hr=("weighted_rate", "sum"),
        )

        # Event-level simple time series
        mean_rate = catchment_rates["precip_rate_mm_hr"].mean()
        max_rate = catchment_rates["precip_rate_mm_hr"].max()

        records.append(
            {
                "event_id": EVENT_TO_PROCESS,
                "mrms_time": mrms_time,
                "mean_precip_rate_mm_hr": mean_rate,
                "max_catchment_precip_rate_mm_hr": max_rate,
                "mean_precip_mm_2min": mean_rate * (2 / 60),
                "max_catchment_precip_mm_2min": max_rate * (2 / 60),
            },
        )

    except Exception as e:
        failed.append(
            {
                "mrms_time": mrms_time,
                "local_path": local_path,
                "error": str(e),
            },
        )

elapsed_min = (time.time() - t0) / 60

event_2min_timeseries = pd.DataFrame(records).sort_values("mrms_time")

print("\n============================================================")
print("2-MIN TIME SERIES DONE ✅")
print("============================================================")
print(f"Successful files: {len(records):,}")
print(f"Failed files    : {len(failed):,}")
print(f"Elapsed time    : {elapsed_min:.2f} minutes")
print(f"Time steps      : {len(event_2min_timeseries):,}")

# ------------------------------------------------------------
# 5. Save output
# ------------------------------------------------------------
PROCESSED_DIR = CACHE / "mrms_processed" / f"event_{EVENT_TO_PROCESS}"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EVENT_2MIN_CSV = PROCESSED_DIR / "event_2min_timeseries.csv"

event_2min_timeseries.to_csv(EVENT_2MIN_CSV, index=False)

if len(failed) > 0:
    pd.DataFrame(failed).to_csv(
        PROCESSED_DIR / "failed_grib_reads.csv",
        index=False,
    )

print(f"Saved: {EVENT_2MIN_CSV}")

# ------------------------------------------------------------
# 6. Plot simple 2-min time series
# ------------------------------------------------------------
plt.figure(figsize=(13, 4))

plt.plot(
    event_2min_timeseries["mrms_time"],
    event_2min_timeseries["mean_precip_rate_mm_hr"],
    linewidth=1.3,
    label="Mean catchment precipitation rate",
)

plt.plot(
    event_2min_timeseries["mrms_time"],
    event_2min_timeseries["max_catchment_precip_rate_mm_hr"],
    linewidth=1.0,
    linestyle="--",
    label="Max catchment precipitation rate",
)

plt.xlabel("Time")
plt.ylabel("Precipitation rate (mm/hr)")
plt.title(f"Event {EVENT_TO_PROCESS} — MRMS 2-min precipitation-rate time series")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

event_2min_timeseries.head()

In [ ]:
# ============================================================
# Create event rainfall time series directly from downloaded GRIB.GZ files
# Reads each GRIB file once, extracts selected MRMS cells, aggregates to 15 min
# ============================================================

import gzip
import tempfile
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Required inputs
# ------------------------------------------------------------
assert "EVENT_TO_PROCESS" in globals(), "❌ EVENT_TO_PROCESS missing."
assert "event_mrms_weights" in globals(), "❌ event_mrms_weights missing."
assert "CACHE" in globals(), "❌ CACHE missing."

print("Processing event:", EVENT_TO_PROCESS)

# ------------------------------------------------------------
# 1. Load download manifest for this event
# ------------------------------------------------------------
DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{EVENT_TO_PROCESS}" / "download_manifest.parquet"
)

assert DOWNLOAD_MANIFEST_FILE.exists(), f"❌ Missing manifest: {DOWNLOAD_MANIFEST_FILE}"

download_manifest = pd.read_parquet(DOWNLOAD_MANIFEST_FILE)

download_manifest["mrms_time"] = pd.to_datetime(
    download_manifest["mrms_time"],
    utc=True,
)

good_files = download_manifest[
    download_manifest["status"].isin(["downloaded", "exists"])
    & (download_manifest["size_bytes"] > 0)
].copy()

good_files = good_files.sort_values("mrms_time").reset_index(drop=True)

print(f"Good files: {len(good_files):,}")
print(f"Time range: {good_files['mrms_time'].min()} to {good_files['mrms_time'].max()}")


# ------------------------------------------------------------
# 2. Get event-specific weights
# ------------------------------------------------------------
event_mrms_weights["_event_id_str"] = event_mrms_weights["event_id"].astype(str)

weights = (
    event_mrms_weights[event_mrms_weights["_event_id_str"] == str(EVENT_TO_PROCESS)][
        ["divide_id", "cell_id", "row", "col", "weight"]
    ]
    .drop_duplicates(subset=["divide_id", "cell_id"])
    .copy()
)

assert len(weights) > 0, f"❌ No weights found for event {EVENT_TO_PROCESS}"

weights["row"] = weights["row"].astype(int)
weights["col"] = weights["col"].astype(int)
weights["weight"] = weights["weight"].astype(float)

rows_idx = weights["row"].values
cols_idx = weights["col"].values

print(f"Catchments: {weights['divide_id'].nunique():,}")
print(f"MRMS cells: {weights['cell_id'].nunique():,}")
print(f"Weight rows: {len(weights):,}")


# ------------------------------------------------------------
# 3. Helper: read one GRIB.GZ file
# ------------------------------------------------------------
def read_mrms_grib_gz(gz_path):
    gz_path = Path(gz_path)

    with gzip.open(gz_path, "rb") as gz:
        raw = gz.read()

    with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
        tmp.write(raw)
        tmp.flush()

        ds = xr.load_dataset(
            tmp.name,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""},
            decode_timedelta=False,
        )

    varname = list(ds.data_vars)[0]
    arr = ds[varname].values.astype(float)

    # MRMS missing/no-data values
    arr[arr < 0] = np.nan

    return arr


# ------------------------------------------------------------
# 4. Read all GRIB files once and create 2-min catchment rates
# ------------------------------------------------------------
records = []
failed = []

t0 = time.time()

for _, r in tqdm(good_files.iterrows(), total=len(good_files)):
    mrms_time = r["mrms_time"]
    local_path = r["local_path"]

    try:
        arr = read_mrms_grib_gz(local_path)

        vals = arr[rows_idx, cols_idx]

        tmp = weights.copy()
        tmp["precip_rate_mm_hr"] = vals
        tmp["weighted_rate"] = tmp["precip_rate_mm_hr"] * tmp["weight"]

        catchment_rates = (
            tmp.groupby("divide_id", as_index=False)["weighted_rate"]
            .sum()
            .rename(columns={"weighted_rate": "precip_rate_mm_hr"})
        )

        catchment_rates["mrms_time"] = mrms_time
        catchment_rates["event_id"] = EVENT_TO_PROCESS

        records.append(catchment_rates)

    except Exception as e:
        failed.append(
            {
                "mrms_time": mrms_time,
                "local_path": local_path,
                "error": str(e),
            },
        )

elapsed_min = (time.time() - t0) / 60

print(f"\nFinished reading files in {elapsed_min:.2f} minutes")
print(f"Successful files: {len(records):,}")
print(f"Failed files: {len(failed):,}")

if len(records) == 0:
    raise RuntimeError("❌ No GRIB files were successfully read.")

catchment_2min_rate = pd.concat(records, ignore_index=True)


# ------------------------------------------------------------
# 5. Aggregate to 15-min rainfall
# ------------------------------------------------------------
catchment_2min_rate["mrms_time"] = pd.to_datetime(
    catchment_2min_rate["mrms_time"],
    utc=True,
)

catchment_15min = catchment_2min_rate.groupby(
    [
        "event_id",
        "divide_id",
        pd.Grouper(key="mrms_time", freq="15min"),
    ],
    as_index=False,
).agg(
    precip_rate_mm_hr=("precip_rate_mm_hr", "mean"),
)

# PrecipRate is mm/hr, so 15-min depth = mean rate × 0.25 hr
catchment_15min["precip_mm_15min"] = catchment_15min["precip_rate_mm_hr"] * 0.25


# ------------------------------------------------------------
# 6. Create event-level time series
# ------------------------------------------------------------
event_15min_timeseries = catchment_15min.groupby(
    ["event_id", "mrms_time"],
    as_index=False,
).agg(
    mean_precip_mm_15min=("precip_mm_15min", "mean"),
    max_catchment_precip_mm_15min=("precip_mm_15min", "max"),
    mean_precip_rate_mm_hr=("precip_rate_mm_hr", "mean"),
)

print(f"\n2-min rows: {len(catchment_2min_rate):,}")
print(f"15-min rows: {len(catchment_15min):,}")
print(f"15-min event timestamps: {len(event_15min_timeseries):,}")


# ------------------------------------------------------------
# 7. Save smaller processed outputs
# ------------------------------------------------------------
PROCESSED_DIR = CACHE / "mrms_processed" / f"event_{EVENT_TO_PROCESS}"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

catchment_2min_rate.to_parquet(
    PROCESSED_DIR / "catchment_2min_precip_rate.parquet",
    index=False,
)

catchment_15min.to_parquet(
    PROCESSED_DIR / "catchment_15min_precip.parquet",
    index=False,
)

event_15min_timeseries.to_csv(
    PROCESSED_DIR / "event_15min_timeseries.csv",
    index=False,
)

if len(failed) > 0:
    pd.DataFrame(failed).to_csv(
        PROCESSED_DIR / "failed_grib_reads.csv",
        index=False,
    )

print("\nSaved outputs in:")
print(PROCESSED_DIR)


# ------------------------------------------------------------
# 8. Plot 15-min time series
# ------------------------------------------------------------
plt.figure(figsize=(13, 4))

plt.plot(
    event_15min_timeseries["mrms_time"],
    event_15min_timeseries["mean_precip_mm_15min"],
    linewidth=1.5,
    label="Mean catchment rainfall",
)

plt.plot(
    event_15min_timeseries["mrms_time"],
    event_15min_timeseries["max_catchment_precip_mm_15min"],
    linewidth=1.0,
    linestyle="--",
    label="Max catchment rainfall",
)

plt.xlabel("Time")
plt.ylabel("15-min precipitation depth (mm)")
plt.title(f"Event {EVENT_TO_PROCESS} — MRMS 15-min precipitation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

event_15min_timeseries.head()

In [ ]:
# ============================================================
# Step 3.9 — Read downloaded 2-min MRMS files,
#            create 15-min catchment rainfall,
#            and plot event rainfall time series
# ============================================================

import gzip
import tempfile
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Reload needed paths/objects after kernel restart
# ------------------------------------------------------------
CACHE = Path("mrms_crosswalk_cache")

assert CACHE.exists(), f"❌ CACHE folder not found: {CACHE}"

EVENT_MRMS_WEIGHTS_FILE = CACHE / "step3_event_relevant" / "event_mrms_weights.parquet"

assert EVENT_MRMS_WEIGHTS_FILE.exists(), f"❌ Missing file: {EVENT_MRMS_WEIGHTS_FILE}"

if "event_mrms_weights" not in globals():
    event_mrms_weights = pd.read_parquet(EVENT_MRMS_WEIGHTS_FILE)
    print("✅ Loaded event_mrms_weights")


# ------------------------------------------------------------
# 1. Choose event to process
# ------------------------------------------------------------
# If EVENT_TO_PROCESS already exists from your downloaded-event summary, use it.
# Otherwise choose the downloaded event with the most good files.

if "EVENT_TO_PROCESS" not in globals():
    manifest_files = sorted(
        (CACHE / "mrms_downloads").glob("event_*/download_manifest.parquet"),
    )

    assert len(manifest_files) > 0, "❌ No downloaded event manifests found."

    summary = []

    for mf in manifest_files:
        event_id = mf.parent.name.replace("event_", "")
        dm = pd.read_parquet(mf)

        good = dm[dm["status"].isin(["downloaded", "exists"]) & (dm["size_bytes"] > 0)]

        summary.append(
            {
                "event_id": event_id,
                "manifest_file": mf,
                "total_rows": len(dm),
                "good_files": len(good),
                "missing_files": dm["status"].astype(str).str.contains("missing").sum(),
            },
        )

    downloaded_event_summary = (
        pd.DataFrame(summary)
        .sort_values("good_files", ascending=False)
        .reset_index(drop=True)
    )

    display(downloaded_event_summary)

    EVENT_TO_PROCESS = downloaded_event_summary.iloc[0]["event_id"]

print("Processing event:", EVENT_TO_PROCESS)


# ------------------------------------------------------------
# 2. Load download manifest for selected event
# ------------------------------------------------------------
DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{EVENT_TO_PROCESS}" / "download_manifest.parquet"
)

assert DOWNLOAD_MANIFEST_FILE.exists(), (
    f"❌ Missing download manifest: {DOWNLOAD_MANIFEST_FILE}"
)

download_manifest = pd.read_parquet(DOWNLOAD_MANIFEST_FILE)

download_manifest["mrms_time"] = pd.to_datetime(
    download_manifest["mrms_time"],
    utc=True,
)

good_files = download_manifest[
    download_manifest["status"].isin(["downloaded", "exists"])
    & (download_manifest["size_bytes"] > 0)
].copy()

good_files = good_files.sort_values("mrms_time").reset_index(drop=True)

assert len(good_files) > 0, "❌ No usable downloaded MRMS files found."

print(f"Good MRMS files: {len(good_files):,}")
print(f"Time range: {good_files['mrms_time'].min()} to {good_files['mrms_time'].max()}")


# ------------------------------------------------------------
# 3. Get event-specific MRMS fractional weights
# ------------------------------------------------------------
# Convert event IDs to string for safe matching
event_mrms_weights["_event_id_str"] = event_mrms_weights["event_id"].astype(str)

weights = event_mrms_weights[
    event_mrms_weights["_event_id_str"] == str(EVENT_TO_PROCESS)
].copy()

weights = weights[["divide_id", "cell_id", "row", "col", "weight"]].drop_duplicates(
    subset=["divide_id", "cell_id"],
)

assert len(weights) > 0, f"❌ No MRMS weights found for event {EVENT_TO_PROCESS}."

weights["row"] = weights["row"].astype(int)
weights["col"] = weights["col"].astype(int)
weights["weight"] = weights["weight"].astype(float)

print(f"Catchments for event: {weights['divide_id'].nunique():,}")
print(f"Unique MRMS cells for event: {weights['cell_id'].nunique():,}")
print(f"Weight rows: {len(weights):,}")


# ------------------------------------------------------------
# 4. Helper to read one local GRIB2.GZ file
# ------------------------------------------------------------
def read_mrms_preciprate_array(gz_path):
    """
    Read one local MRMS .grib2.gz file and return the precipitation-rate array.
    MRMS PrecipRate is usually read by cfgrib as variable 'unknown'.
    """
    gz_path = Path(gz_path)

    with gzip.open(gz_path, "rb") as gz:
        raw_grib = gz.read()

    with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
        tmp.write(raw_grib)
        tmp.flush()

        ds = xr.load_dataset(
            tmp.name,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""},
            decode_timedelta=False,
        )

    varname = list(ds.data_vars)[0]

    arr = ds[varname].values.astype(float)

    # MRMS no-data values are commonly negative
    arr[arr < 0] = np.nan

    return arr


# ------------------------------------------------------------
# 5. Quick test on one file before full loop
# ------------------------------------------------------------
test_file = good_files.iloc[0]["local_path"]

test_arr = read_mrms_preciprate_array(test_file)

print("\nOne-file test:")
print("array shape:", test_arr.shape)
print("valid min :", np.nanmin(test_arr))
print("valid max :", np.nanmax(test_arr))
print("valid mean:", np.nanmean(test_arr))


# ------------------------------------------------------------
# 6. Loop through files and compute catchment 2-min rates
# ------------------------------------------------------------
records = []
failed_files = []

t0 = time.time()

for _, r in tqdm(good_files.iterrows(), total=len(good_files)):
    mrms_time = r["mrms_time"]
    local_path = r["local_path"]

    try:
        arr = read_mrms_preciprate_array(local_path)

        vals = arr[
            weights["row"].values,
            weights["col"].values,
        ]

        tmp = weights.copy()
        tmp["precip_rate_mm_hr"] = vals

        # Weighted catchment-average precipitation rate
        tmp["weighted_rate"] = tmp["precip_rate_mm_hr"] * tmp["weight"]

        catchment_rates = (
            tmp.groupby("divide_id", as_index=False)["weighted_rate"]
            .sum()
            .rename(columns={"weighted_rate": "precip_rate_mm_hr"})
        )

        catchment_rates["mrms_time"] = mrms_time
        catchment_rates["event_id"] = EVENT_TO_PROCESS

        records.append(catchment_rates)

    except Exception as e:
        failed_files.append(
            {
                "mrms_time": mrms_time,
                "local_path": local_path,
                "error_type": type(e).__name__,
                "error_message": str(e),
            },
        )

elapsed_min = (time.time() - t0) / 60

print(f"\nFinished reading GRIB files in {elapsed_min:.2f} minutes")
print(f"Successful files: {len(records):,}")
print(f"Failed files    : {len(failed_files):,}")

if len(records) == 0:
    raise RuntimeError(
        "❌ No MRMS files were successfully read. Check cfgrib and file paths.",
    )

catchment_2min_rate = pd.concat(records, ignore_index=True)

print(f"\n2-min catchment rows: {len(catchment_2min_rate):,}")
print(f"Unique catchments: {catchment_2min_rate['divide_id'].nunique():,}")
print(f"Unique times: {catchment_2min_rate['mrms_time'].nunique():,}")


# ------------------------------------------------------------
# 7. Aggregate 2-min rate to 15-min rainfall
# ------------------------------------------------------------
catchment_2min_rate["mrms_time"] = pd.to_datetime(
    catchment_2min_rate["mrms_time"],
    utc=True,
)

catchment_15min = catchment_2min_rate.groupby(
    [
        "event_id",
        "divide_id",
        pd.Grouper(key="mrms_time", freq="15min"),
    ],
    as_index=False,
).agg(
    precip_rate_mm_hr=("precip_rate_mm_hr", "mean"),
)

# Convert mean rate mm/hr to 15-min accumulation depth mm
catchment_15min["precip_mm_15min"] = catchment_15min["precip_rate_mm_hr"] * 0.25

print(f"\n15-min catchment rows: {len(catchment_15min):,}")
print(f"15-min timestamps: {catchment_15min['mrms_time'].nunique():,}")


# ------------------------------------------------------------
# 8. Create event-average 15-min time series
# ------------------------------------------------------------
event_15min_timeseries = catchment_15min.groupby(
    ["event_id", "mrms_time"],
    as_index=False,
).agg(
    mean_precip_rate_mm_hr=("precip_rate_mm_hr", "mean"),
    mean_precip_mm_15min=("precip_mm_15min", "mean"),
    max_catchment_precip_mm_15min=("precip_mm_15min", "max"),
)


# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------
PROCESSED_DIR = CACHE / "mrms_processed" / f"event_{EVENT_TO_PROCESS}"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CATCHMENT_2MIN_FILE = PROCESSED_DIR / "catchment_2min_precip_rate.parquet"
CATCHMENT_15MIN_FILE = PROCESSED_DIR / "catchment_15min_precip.parquet"
EVENT_15MIN_CSV = PROCESSED_DIR / "event_15min_timeseries.csv"
FAILED_FILES_CSV = PROCESSED_DIR / "failed_grib_reads.csv"

catchment_2min_rate.to_parquet(CATCHMENT_2MIN_FILE, index=False)
catchment_15min.to_parquet(CATCHMENT_15MIN_FILE, index=False)
event_15min_timeseries.to_csv(EVENT_15MIN_CSV, index=False)

if len(failed_files) > 0:
    pd.DataFrame(failed_files).to_csv(FAILED_FILES_CSV, index=False)

print("\nSaved outputs:")
print(CATCHMENT_2MIN_FILE)
print(CATCHMENT_15MIN_FILE)
print(EVENT_15MIN_CSV)

if len(failed_files) > 0:
    print(FAILED_FILES_CSV)


# ------------------------------------------------------------
# 10. Plot 15-min event rainfall time series
# ------------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.plot(
    event_15min_timeseries["mrms_time"],
    event_15min_timeseries["mean_precip_mm_15min"],
    linewidth=1.5,
    label="Mean catchment rainfall",
)

plt.plot(
    event_15min_timeseries["mrms_time"],
    event_15min_timeseries["max_catchment_precip_mm_15min"],
    linewidth=1.0,
    linestyle="--",
    label="Max catchment rainfall",
)

plt.xlabel("Time")
plt.ylabel("15-min precipitation depth (mm)")
plt.title(f"Event {EVENT_TO_PROCESS} — MRMS 15-min precipitation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

event_15min_timeseries.head()

In [ ]:
# ============================================================
# Step 3.9 — Read downloaded 2-min MRMS files,
#            create 15-min catchment rainfall,
#            and plot event rainfall time series
# ============================================================

import gzip
import tempfile
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Choose event to process
# ------------------------------------------------------------
# Use largest_event_id if you already selected/downloaded the largest event.
# Otherwise this falls back to test_event_id.


# if "largest_event_id" in globals():
#     EVENT_TO_PROCESS = largest_event_id
# else:
#     EVENT_TO_PROCESS = test_event_id


print("Processing event:", EVENT_TO_PROCESS)


print("Processing event:", EVENT_TO_PROCESS)

# ------------------------------------------------------------
# 1. Required inputs
# ------------------------------------------------------------
assert "event_mrms_weights" in globals(), "❌ event_mrms_weights missing."
assert "CACHE" in globals(), "❌ CACHE missing."

DOWNLOAD_MANIFEST_FILE = (
    CACHE / "mrms_downloads" / f"event_{EVENT_TO_PROCESS}" / "download_manifest.parquet"
)

assert DOWNLOAD_MANIFEST_FILE.exists(), (
    f"❌ Missing download manifest: {DOWNLOAD_MANIFEST_FILE}"
)

download_manifest = pd.read_parquet(DOWNLOAD_MANIFEST_FILE)

download_manifest["mrms_time"] = pd.to_datetime(
    download_manifest["mrms_time"],
    utc=True,
)

good_files = download_manifest[
    download_manifest["status"].isin(["downloaded", "exists"])
    & (download_manifest["size_bytes"] > 0)
].copy()

good_files = good_files.sort_values("mrms_time").reset_index(drop=True)

print(f"Good MRMS files: {len(good_files):,}")
print(f"Time range: {good_files['mrms_time'].min()} to {good_files['mrms_time'].max()}")

# ------------------------------------------------------------
# 2. Get event-specific MRMS fractional weights
# ------------------------------------------------------------
weights = event_mrms_weights[event_mrms_weights["event_id"] == EVENT_TO_PROCESS].copy()

weights = weights[["divide_id", "cell_id", "row", "col", "weight"]].drop_duplicates(
    subset=["divide_id", "cell_id"],
)

assert len(weights) > 0, "❌ No MRMS weights found for this event."

weights["row"] = weights["row"].astype(int)
weights["col"] = weights["col"].astype(int)

print(f"Catchments for event: {weights['divide_id'].nunique():,}")
print(f"Unique MRMS cells for event: {weights['cell_id'].nunique():,}")
print(f"Weight rows: {len(weights):,}")


# ------------------------------------------------------------
# 3. Helper to read one local GRIB2.GZ file
# ------------------------------------------------------------
def read_mrms_preciprate_array(gz_path):
    """
    Read one local MRMS .grib2.gz file and return the precipitation array.
    Usually MRMS PrecipRate is stored as variable 'unknown'.
    """
    gz_path = Path(gz_path)

    with gzip.open(gz_path, "rb") as gz:
        raw_grib = gz.read()

    with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
        tmp.write(raw_grib)
        tmp.flush()

        ds = xr.load_dataset(
            tmp.name,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""},
            decode_timedelta=False,
        )

    varname = list(ds.data_vars)[0]
    arr = ds[varname].values.astype(float)

    # MRMS missing/no-data values are commonly negative
    arr[arr < 0] = np.nan

    return arr


# ------------------------------------------------------------
# 4. Loop through files and compute catchment 2-min rates
# ------------------------------------------------------------
records = []
t0 = time.time()

for _, r in tqdm(good_files.iterrows(), total=len(good_files)):
    mrms_time = r["mrms_time"]
    local_path = r["local_path"]

    try:
        arr = read_mrms_preciprate_array(local_path)

        vals = arr[
            weights["row"].values,
            weights["col"].values,
        ]

        tmp = weights.copy()
        tmp["precip_rate_mm_hr"] = vals

        # Weighted catchment-average precipitation rate
        tmp["weighted_rate"] = tmp["precip_rate_mm_hr"] * tmp["weight"]

        catchment_rates = (
            tmp.groupby("divide_id", as_index=False)["weighted_rate"]
            .sum()
            .rename(columns={"weighted_rate": "precip_rate_mm_hr"})
        )

        catchment_rates["mrms_time"] = mrms_time
        catchment_rates["event_id"] = EVENT_TO_PROCESS

        records.append(catchment_rates)

    except Exception as e:
        print(f"⚠️ Failed: {local_path} | {type(e).__name__}: {e}")

elapsed_min = (time.time() - t0) / 60
print(f"\nFinished reading GRIB files in {elapsed_min:.2f} minutes")

catchment_2min_rate = pd.concat(records, ignore_index=True)

print(f"2-min catchment rows: {len(catchment_2min_rate):,}")
print(f"Unique catchments: {catchment_2min_rate['divide_id'].nunique():,}")
print(f"Unique times: {catchment_2min_rate['mrms_time'].nunique():,}")

# ------------------------------------------------------------
# 5. Aggregate 2-min rate to 15-min rainfall
# ------------------------------------------------------------
catchment_2min_rate["mrms_time"] = pd.to_datetime(
    catchment_2min_rate["mrms_time"],
    utc=True,
)

catchment_15min = (
    catchment_2min_rate.set_index("mrms_time")
    .groupby(["event_id", "divide_id"])
    .resample("15min")["precip_rate_mm_hr"]
    .mean()
    .reset_index()
)

# Convert mean rate mm/hr to 15-min accumulation depth mm
catchment_15min["precip_mm_15min"] = catchment_15min["precip_rate_mm_hr"] * 0.25

print(f"15-min catchment rows: {len(catchment_15min):,}")
print(f"15-min timestamps: {catchment_15min['mrms_time'].nunique():,}")

# ------------------------------------------------------------
# 6. Create event-average 15-min time series
# ------------------------------------------------------------
# Simple mean across event catchments.
# Later we can replace this with catchment-area-weighted mean if needed.
event_15min_timeseries = catchment_15min.groupby(
    ["event_id", "mrms_time"],
    as_index=False,
).agg(
    mean_precip_rate_mm_hr=("precip_rate_mm_hr", "mean"),
    mean_precip_mm_15min=("precip_mm_15min", "mean"),
    max_catchment_precip_mm_15min=("precip_mm_15min", "max"),
)

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------
PROCESSED_DIR = CACHE / "mrms_processed" / f"event_{EVENT_TO_PROCESS}"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CATCHMENT_2MIN_FILE = PROCESSED_DIR / "catchment_2min_precip_rate.parquet"
CATCHMENT_15MIN_FILE = PROCESSED_DIR / "catchment_15min_precip.parquet"
EVENT_15MIN_CSV = PROCESSED_DIR / "event_15min_timeseries.csv"

catchment_2min_rate.to_parquet(CATCHMENT_2MIN_FILE, index=False)
catchment_15min.to_parquet(CATCHMENT_15MIN_FILE, index=False)
event_15min_timeseries.to_csv(EVENT_15MIN_CSV, index=False)

print("\nSaved outputs:")
print(CATCHMENT_2MIN_FILE)
print(CATCHMENT_15MIN_FILE)
print(EVENT_15MIN_CSV)

# ------------------------------------------------------------
# 8. Plot 15-min event rainfall time series
# ------------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.plot(
    event_15min_timeseries["mrms_time"],
    event_15min_timeseries["mean_precip_mm_15min"],
    linewidth=1.5,
    label="Mean catchment rainfall",
)

plt.plot(
    event_15min_timeseries["mrms_time"],
    event_15min_timeseries["max_catchment_precip_mm_15min"],
    linewidth=1.0,
    linestyle="--",
    label="Max catchment rainfall",
)

plt.xlabel("Time")
plt.ylabel("15-min precipitation depth (mm)")
plt.title(f"Event {EVENT_TO_PROCESS} — MRMS 15-min precipitation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

event_15min_timeseries.head()

In [ ]:
# ============================================================
# Check final_events (5).csv and count required MRMS files
#
# Selection:
#   events with duration_hours < 48
#
# Fixed 10-day MRMS window:
#   window_end   = event end + 2 days
#   window_start = window_end - 10 days
#
# Output:
#   number of event windows
#   number of total event-time rows
#   number of unique MRMS timestamps after overlap removal
#   number already downloaded
#   number still needed
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------
CSV_FILE = Path("final_events (5).csv")
CACHE = Path("mrms_crosswalk_cache")

# Choose event definition
EVENT_ID_COL = "storm_index"  # one storm-gage case per row
# EVENT_ID_COL = "cluster_id"  # one merged storm event

MRMS_TIME_FREQ = "2min"
MAX_DURATION_HOURS = 48

WINDOW_LENGTH_DAYS = 10
WINDOW_END_AFTER_EVENT_END_DAYS = 2

MRMS_PRODUCT = "PrecipRate"
MRMS_FIELD = "PrecipRate_00.00"

GLOBAL_MRMS_DIR = CACHE / "mrms_downloads_global" / MRMS_PRODUCT
OLD_EVENT_DIR = CACHE / "mrms_downloads"

print("CSV file:", CSV_FILE)
print("Event ID column:", EVENT_ID_COL)


# ------------------------------------------------------------
# 1. Load and check CSV
# ------------------------------------------------------------
events = pd.read_csv(CSV_FILE)

print("\nCSV rows:", len(events))
print("CSV columns:")
print(list(events.columns))

needed_cols = [
    EVENT_ID_COL,
    "begin",
    "end",
    "duration_hours",
    "centroid_lat",
    "centroid_lon",
    "usgs_gage_id",
    "gage_lat",
    "gage_lon",
]

missing = [c for c in needed_cols if c not in events.columns]

assert len(missing) == 0, f"❌ Missing columns: {missing}"

events["begin_time"] = pd.to_datetime(events["begin"], errors="coerce", utc=True)
events["end_time"] = pd.to_datetime(events["end"], errors="coerce", utc=True)
events["duration_hours"] = pd.to_numeric(events["duration_hours"], errors="coerce")

events = events.dropna(
    subset=[EVENT_ID_COL, "begin_time", "end_time", "duration_hours"],
).copy()

print("\nValid rows after time/duration check:", len(events))


# ------------------------------------------------------------
# 2. Select events shorter than 2 days
# ------------------------------------------------------------
events_lt2d = events[events["duration_hours"] < MAX_DURATION_HOURS].copy()

# one row per selected event/case
events_lt2d_unique = (
    events_lt2d.sort_values(["end_time", "begin_time"])
    .drop_duplicates(subset=[EVENT_ID_COL])
    .reset_index(drop=True)
)

print("\nEvents/cases with duration < 2 days:")
print(f"Rows selected: {len(events_lt2d):,}")
print(f"Unique {EVENT_ID_COL}: {events_lt2d_unique[EVENT_ID_COL].nunique():,}")


# ------------------------------------------------------------
# 3. Create fixed 10-day window
# ------------------------------------------------------------
events_lt2d_unique["window_end"] = events_lt2d_unique["end_time"] + pd.Timedelta(
    days=WINDOW_END_AFTER_EVENT_END_DAYS,
)

events_lt2d_unique["window_start"] = events_lt2d_unique["window_end"] - pd.Timedelta(
    days=WINDOW_LENGTH_DAYS,
)

events_lt2d_unique["window_start"] = events_lt2d_unique["window_start"].dt.floor(
    MRMS_TIME_FREQ,
)
events_lt2d_unique["window_end"] = events_lt2d_unique["window_end"].dt.floor(
    MRMS_TIME_FREQ,
)

# half-open window: start <= time < end
# 10 days at 2-min = 7200 timestamps per event
all_rows = []

for _, r in events_lt2d_unique.iterrows():
    times = pd.date_range(
        start=r["window_start"],
        end=r["window_end"] - pd.Timedelta(minutes=2),
        freq=MRMS_TIME_FREQ,
    )

    tmp = pd.DataFrame(
        {
            "event_id": r[EVENT_ID_COL],
            "begin_time": r["begin_time"],
            "end_time": r["end_time"],
            "duration_hours": r["duration_hours"],
            "window_start": r["window_start"],
            "window_end": r["window_end"],
            "mrms_time": times,
        },
    )

    all_rows.append(tmp)

event_time_manifest_10d = pd.concat(all_rows, ignore_index=True)

unique_mrms_times_10d = (
    event_time_manifest_10d[["mrms_time"]]
    .drop_duplicates()
    .sort_values("mrms_time")
    .reset_index(drop=True)
)

expected_per_event = WINDOW_LENGTH_DAYS * 24 * 30

print("\nWindow summary:")
print(f"Expected timestamps per event: {expected_per_event:,}")
print(
    f"Total event-time rows before removing overlap: {len(event_time_manifest_10d):,}",
)
print(f"Unique MRMS timestamps after removing overlap: {len(unique_mrms_times_10d):,}")
print(
    f"Duplicate timestamp downloads avoided: {len(event_time_manifest_10d) - len(unique_mrms_times_10d):,}",
)


# ------------------------------------------------------------
# 4. Check files already downloaded
# ------------------------------------------------------------
def mrms_filename(t):
    t = pd.Timestamp(t).tz_convert("UTC")
    stamp = t.strftime("%Y%m%d-%H%M%S")
    return f"{MRMS_FIELD}_{stamp}.grib2.gz"


def expected_global_path(t):
    t = pd.Timestamp(t).tz_convert("UTC")
    yyyy = t.strftime("%Y")
    mm = t.strftime("%m")
    dd = t.strftime("%d")
    return GLOBAL_MRMS_DIR / yyyy / mm / dd / mrms_filename(t)


unique_mrms_times_10d["filename"] = unique_mrms_times_10d["mrms_time"].apply(
    mrms_filename,
)
unique_mrms_times_10d["expected_global_path"] = unique_mrms_times_10d[
    "mrms_time"
].apply(expected_global_path)

unique_mrms_times_10d["exists_global"] = unique_mrms_times_10d[
    "expected_global_path"
].apply(
    lambda p: Path(p).exists() and Path(p).stat().st_size > 0,
)

old_files = set()

if OLD_EVENT_DIR.exists():
    for f in OLD_EVENT_DIR.glob("event_*/PrecipRate/*.grib2.gz"):
        if f.exists() and f.stat().st_size > 0:
            old_files.add(f.name)

unique_mrms_times_10d["exists_old_event_folder"] = unique_mrms_times_10d[
    "filename"
].isin(old_files)

unique_mrms_times_10d["already_available"] = (
    unique_mrms_times_10d["exists_global"]
    | unique_mrms_times_10d["exists_old_event_folder"]
)

new_files_needed = unique_mrms_times_10d[
    ~unique_mrms_times_10d["already_available"]
].copy()


# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------
print("\n============================================================")
print("FINAL MRMS FILE COUNT SUMMARY ✅")
print("============================================================")
print(f"Selected unique events/cases: {events_lt2d_unique[EVENT_ID_COL].nunique():,}")
print(f"Window length: {WINDOW_LENGTH_DAYS} days")
print(f"Window ends: {WINDOW_END_AFTER_EVENT_END_DAYS} days after event end")
print(f"Total timestamps before de-duplication: {len(event_time_manifest_10d):,}")
print(f"Unique MRMS files needed after de-duplication: {len(unique_mrms_times_10d):,}")
print(f"Already in global folder: {unique_mrms_times_10d['exists_global'].sum():,}")
print(
    f"Already in old event folders: {unique_mrms_times_10d['exists_old_event_folder'].sum():,}",
)
print(
    f"Already available anywhere: {unique_mrms_times_10d['already_available'].sum():,}",
)
print(f"New files still needed: {len(new_files_needed):,}")

print("\nRequired MRMS time range:")
print(
    unique_mrms_times_10d["mrms_time"].min(),
    "to",
    unique_mrms_times_10d["mrms_time"].max(),
)


# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------
OUT_DIR = CACHE / "step3_event_relevant" / "fixed_10day_window"
OUT_DIR.mkdir(parents=True, exist_ok=True)

events_lt2d_unique.to_parquet(
    OUT_DIR / "events_lt2d_fixed_10day_windows.parquet",
    index=False,
)

event_time_manifest_10d.to_parquet(
    OUT_DIR / "event_time_manifest_10day.parquet",
    index=False,
)

unique_mrms_times_10d.to_parquet(
    OUT_DIR / "unique_mrms_times_10day.parquet",
    index=False,
)

new_files_needed.to_parquet(
    OUT_DIR / "new_mrms_files_needed_10day.parquet",
    index=False,
)

print("\nSaved files to:")
print(OUT_DIR)

unique_mrms_times_10d.head()